In [ ]:
# Import required packages
import copy
import math
import folium
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.patheffects as PathEffects
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib import colors as mcolours
from matplotlib.animation import FuncAnimation
from pathlib import Path
from pyproj import Transformer
from shapely.geometry import box
from skimage.exposure import rescale_intensity
import matplotlib.cm as cm



def _degree_to_zoom_level(l1, l2, margin=0.0):
    """
    Calculates an integer zoom level based on the difference between two geographic coordinates.

    This function estimates an appropriate map zoom level such that the bounding box defined by 
    the two coordinates fits nicely within the viewport, optionally including a margin.

    Parameters:
    -----------
    l1 : float
        The first coordinate (latitude or longitude in degrees).

    l2 : float
        The second coordinate (latitude or longitude in degrees).

    margin : float, optional (default=0.0)
        A fractional margin to increase the bounding box size, e.g., 0.1 adds 10% padding.

    Returns:
    --------
    zoom_level_int : int
        An integer zoom level (typically between 0 and 18), where higher values mean closer zoom.

    Notes:
    ------
    - If the coordinates are identical (`degree == 0`), returns a default zoom level of 18.
    - Uses the formula based on logarithm of the ratio between full map width (360 degrees) and the bounding box.
    """


    degree = abs(l1 - l2) * (1 + margin)
    zoom_level_int = 0
    if degree != 0:
        zoom_level_float = math.log(360 / degree) / math.log(2)
        zoom_level_int = int(zoom_level_float)
    else:
        zoom_level_int = 18
    return zoom_level_int

def display_map(x, y, crs='EPSG:4326', margin=-0.5, zoom_bias=0, centroid=None):
    """ 
    Generates an interactive map displaying a bounding rectangle or centroid overlay on Google Maps imagery.

    This function takes coordinate bounds in any projected coordinate reference system (CRS), transforms them 
    to latitude and longitude (EPSG:4326), and plots an interactive folium map. It overlays a red bounding 
    rectangle outlining the coordinate extent or optionally a red circle marking a centroid point.

    The map's zoom level is automatically calculated to frame the bounding box as tightly as possible 
    without clipping, with options to adjust zoom level and add padding.

    Last modified: July 2025
    
    Adapted from a function by Otto Wagner: 
    https://github.com/ceos-seo/data_cube_utilities/tree/master/data_cube_utilities

    Parameters
    ----------
    x : tuple of float
        Tuple of (min, max) x coordinates in the specified CRS.
    y : tuple of float
        Tuple of (min, max) y coordinates in the specified CRS.
    crs : str, optional
        Coordinate reference system of the input coordinates (default 'EPSG:4326').
    margin : float, optional
        Degrees of latitude/longitude padding added around the bounding box to increase spacing 
        between the rectangle and map edges (default -0.5).
    zoom_bias : float or int, optional
        Adjustment to zoom level; positive values zoom in, negative zoom out (default 0).
    centroid : tuple of float or None, optional
        Optional centroid coordinate as (latitude, longitude). If provided, a red circle will 
        mark this point instead of drawing the bounding rectangle.

    Returns
    -------
    folium.Map
        A folium interactive map centered on the bounding box or centroid, with overlays and zoom 
        level optimized to the input coordinates.

    Example
    -------
    >>> display_map((500000, 510000), (2000000, 2010000), crs='EPSG:3857', margin=0.1)
    """
    # Convert each corner coordinates to lat-lon
    all_x = (x[0], x[1], x[0], x[1])
    all_y = (y[0], y[0], y[1], y[1])
    transformer = Transformer.from_crs(crs, "EPSG:4326")
    all_longitude, all_latitude = transformer.transform(all_x, all_y)

    # Calculate zoom level based on coordinates
    lat_zoom_level = _degree_to_zoom_level(
        min(all_latitude), max(all_latitude), margin=margin) + zoom_bias
    lon_zoom_level = _degree_to_zoom_level(
        min(all_longitude), max(all_longitude), margin=margin) + zoom_bias
    zoom_level = min(lat_zoom_level, lon_zoom_level)

    # Identify centre point for plotting
    center = [np.mean(all_latitude), np.mean(all_longitude)]

    # Create map
    interactive_map = folium.Map(
        location=center,
        zoom_start=zoom_level,
        tiles="http://mt1.google.com/vt/lyrs=y&z={z}&x={x}&y={y}",
        attr="Google")

    # Create bounding box coordinates to overlay on map
    line_segments = [(all_latitude[0], all_longitude[0]),
                     (all_latitude[1], all_longitude[1]),
                     (all_latitude[3], all_longitude[3]),
                     (all_latitude[2], all_longitude[2]),
                     (all_latitude[0], all_longitude[0])]

    

    # Add the centroid point as an overlay 
    if centroid is not None:
        interactive_map.add_child(
        folium.Circle(location=[centroid[0],centroid[-1]],
                                 color='red',
                                 opacity=1,
                                radius=10000,
                               fill=False
                              ),
        
        
        )



        
    else:
        # Add bounding box as an overlay
        interactive_map.add_child(
        folium.features.PolyLine(locations=line_segments,
                                 color='red',
                                 opacity=0.8))
        

    # Add clickable lat-lon popup box
    interactive_map.add_child(folium.features.LatLngPopup())

    return interactive_map

def rgb(ds,
        bands=['nbart_red', 'nbart_green', 'nbart_blue'],
        index=None,
        index_dim='time',
        robust=True,
        percentile_stretch=None,
        col_wrap=4,
        size=6,
        aspect=None,
        titles=None,
        savefig_path=None,
        savefig_kwargs={},
        **kwargs):
    """
    Plots RGB images from an xarray Dataset using specified bands, with support for single or multiple observations.

    This function serves as a convenient wrapper around xarray’s `.plot.imshow()` for creating true-color or false-color
    composite images from satellite data. It allows selecting specific observations by index or creating faceted plots
    when multiple images are selected.

    Images can optionally be saved to a file by specifying a save path.

    Last modified: July 2025

    Adapted from dc_rgb.py by John Rattz:
    https://github.com/ceos-seo/data_cube_utilities/blob/master/data_cube_utilities/dc_rgb.py

    Parameters
    ----------
    ds : xarray.Dataset
        Input dataset containing imagery bands with spatial dimensions and optionally a time dimension.
    
    bands : list of str, optional
        List of three band names (strings) to use for RGB channels. Defaults to
        ['nbart_red', 'nbart_green', 'nbart_blue'].

    index : int or list of int, optional
        Index or list of indices along the `index_dim` dimension selecting observations to plot.
        If multiple indices are given, a faceted plot will be created. Defaults to None (plot all).

    index_dim : str, optional
        Dimension name to index along when selecting observations. Defaults to 'time'.

    robust : bool, optional
        Whether to scale the image color limits using 2nd and 98th percentiles (robust stretching).
        Defaults to True.

    percentile_stretch : tuple(float, float), optional
        Tuple specifying manual percentile clipping (e.g., (0.02, 0.98)) for color limits. Overrides `robust` if set.
        Defaults to None.

    col_wrap : int, optional
        Number of columns in faceted plot when plotting multiple images. Defaults to 4.

    size : int or float, optional
        Height in inches of each subplot. Defaults to 6.

    aspect : float or None, optional
        Aspect ratio (width/height) of each subplot. If None, computed automatically based on dataset geobox.

    titles : str or list of str, optional
        Custom titles for each subplot when plotting multiple images. Defaults to None (uses default titles).

    savefig_path : str, optional
        File path to save the generated figure. If None, figure is not saved.

    savefig_kwargs : dict, optional
        Additional keyword arguments passed to `matplotlib.pyplot.savefig()` when saving the figure.

    **kwargs
        Additional keyword arguments passed to `xarray.plot.imshow()` (e.g., `ax` to specify matplotlib axes).

    Returns
    -------
    matplotlib.axes.Axes or FacetGrid
        The matplotlib axes object or seaborn FacetGrid object created by xarray plotting.

    Raises
    ------
    Exception
        If input dataset has multiple observations but no `index` or `col` argument is supplied, instructing user
        to provide either.

    Example
    -------
    >>> rgb(ds, index=0)  # Plot the first image in the time dimension
    >>> rgb(ds, index=[0,1], titles=['Jan', 'Feb'])  # Faceted plot of first two images with custom titles
    >>> rgb(ds, savefig_path='output.png')  # Save the RGB plot to a file
    """

    
    # Get names of x and y dims
    y_dim, x_dim = ds.odc.spatial_dims

    # If ax is supplied via kwargs, ignore aspect and size
    if 'ax' in kwargs:

        # Create empty aspect size kwarg that will be passed to imshow
        aspect_size_kwarg = {}
    else:
        # Compute image aspect
        if not aspect:
            aspect = ds.odc.geobox.aspect

        # Populate aspect size kwarg with aspect and size data
        aspect_size_kwarg = {'aspect': aspect, 'size': size}

    # If no value is supplied for `index` (the default), plot using default
    # values and arguments passed via `**kwargs`
    if index is None:

        # Select bands and convert to DataArray
        da = ds[bands].to_array().compute()

        # If percentile_stretch == True, clip plotting to percentile vmin, vmax
        if percentile_stretch:
            vmin, vmax = da.quantile(percentile_stretch).values
            kwargs.update({'vmin': vmin, 'vmax': vmax})

        # If there are more than three dimensions and the index dimension == 1,
        # squeeze this dimension out to remove it
        if ((len(ds.dims) > 2) and ('col' not in kwargs) and
            (len(da[index_dim]) == 1)):

            da = da.squeeze(dim=index_dim)

        # If there are more than three dimensions and the index dimension
        # is longer than 1, raise exception to tell user to use 'col'/`index`
        elif ((len(ds.dims) > 2) and ('col' not in kwargs) and
              (len(da[index_dim]) > 1)):

            raise Exception(
                f'The input dataset `ds` has more than two dimensions: '
                f'{list(ds.dims.keys())}. Please select a single observation '
                'using e.g. `index=0`, or enable faceted plotting by adding '
                'the arguments e.g. `col="time", col_wrap=4` to the function '
                'call')

        img = da.plot.imshow(x=x_dim,
                             y=y_dim,
                             robust=robust,
                             col_wrap=col_wrap,
                             **aspect_size_kwarg,
                             **kwargs)
        if titles is not None:
            for ax, title in zip(img.axs.flat, titles):
                ax.set_title(title)

    # If values provided for `index`, extract corresponding observations and
    # plot as either single image or facet plot
    else:

        # If a float is supplied instead of an integer index, raise exception
        if isinstance(index, float):
            raise Exception(
                f'Please supply `index` as either an integer or a list of '
                'integers')

        # If col argument is supplied as well as `index`, raise exception
        if 'col' in kwargs:
            raise Exception(
                f'Cannot supply both `index` and `col`; please remove one and '
                'try again')

        # Convert index to generic type list so that number of indices supplied
        # can be computed
        index = index if isinstance(index, list) else [index]

        # Select bands and observations and convert to DataArray
        da = ds[bands].isel(**{index_dim: index}).to_array().compute()

        # If percentile_stretch == True, clip plotting to percentile vmin, vmax
        if percentile_stretch:
            vmin, vmax = da.quantile(percentile_stretch).values
            kwargs.update({'vmin': vmin, 'vmax': vmax})

        # If multiple index values are supplied, plot as a faceted plot
        if len(index) > 1:

            img = da.plot.imshow(x=x_dim,
                                 y=y_dim,
                                 robust=robust,
                                 col=index_dim,
                                 col_wrap=col_wrap,
                                 **aspect_size_kwarg,
                                 **kwargs)
            if titles is not None:
                for ax, title in zip(img.axs.flat, titles):
                    ax.set_title(title)

        # If only one index is supplied, squeeze out index_dim and plot as a
        # single panel
        else:

            img = da.squeeze(dim=index_dim).plot.imshow(robust=robust,
                                                        **aspect_size_kwarg,
                                                        **kwargs)
            if titles is not None:
                for ax, title in zip(img.axs.flat, titles):
                    ax.set_title(title)

    # If an export path is provided, save image to file. Individual and
    # faceted plots have a different API (figure vs fig) so we get around this
    # using a try statement:
    if savefig_path:

        print(f'Exporting image to {savefig_path}')

        try:
            img.fig.savefig(savefig_path, **savefig_kwargs)
        except:
            img.figure.savefig(savefig_path, **savefig_kwargs)

def single_band(ds,
        band=None,
        index=None,
        index_dim='time',
        robust=True,
        vmin=None,
        vmax=None,
        label=None,
        col_wrap=4,
        size=6,
        aspect=None,
        titles=None,
        savefig_path=None,
        savefig_kwargs={},
        **kwargs):
    """
    Parameters
    ----------  
    ds : xarray datarray
        A two-dimensional or multi-dimensional array to plot as an RGB 
        image. If the array has more than two dimensions (e.g. multiple 
        observations along a 'time' dimension), either use `index` to 
        select one (`index=0`) or multiple observations 
        (`index=[0, 1]`), or create a custom faceted plot using e.g. 
        `col="time"`.       
    bands : list of strings, optional
        A list of three strings giving the band names to plot. Defaults 
        to '['nbart_red', 'nbart_green', 'nbart_blue']'.
    index : integer or list of integers, optional
        `index` can be used to select one (`index=0`) or multiple 
        observations (`index=[0, 1]`) from the input dataset for 
        plotting. If multiple images are requested these will be plotted
        as a faceted plot.
    index_dim : string, optional
        The dimension along which observations should be plotted if 
        multiple observations are requested using `index`. Defaults to 
        `time`.
    robust : bool, optional
        Produces an enhanced image where the colormap range is computed 
        with 2nd and 98th percentiles instead of the extreme values. 
        Defaults to True.
    percentile_stretch : tuple of floats
        An tuple of two floats (between 0.00 and 1.00) that can be used 
        to clip the colormap range to manually specified percentiles to 
        get more control over the brightness and contrast of the image. 
        The default is None; '(0.02, 0.98)' is equivelent to 
        `robust=True`. If this parameter is used, `robust` will have no 
        effect.
    col_wrap : integer, optional
        The number of columns allowed in faceted plots. Defaults to 4.
    size : integer, optional
        The height (in inches) of each plot. Defaults to 6.
    aspect : integer, optional
        Aspect ratio of each facet in the plot, so that aspect * size 
        gives width of each facet in inches. Defaults to None, which 
        will calculate the aspect based on the x and y dimensions of 
        the input data.
    titles : string or list of strings, optional
        Replace the xarray 'time' dimension on plot titles with a string
        or list of string titles, when a list of index values are
        provided, of your choice. Defaults to None.
    savefig_path : string, optional
        Path to export image file for the RGB plot. Defaults to None, 
        which does not export an image file.
    savefig_kwargs : dict, optional
        A dict of keyword arguments to pass to 
        `matplotlib.pyplot.savefig` when exporting an image file. For 
        all available options, see: 
        https://matplotlib.org/api/_as_gen/matplotlib.pyplot.savefig.html        
    **kwargs : optional
        Additional keyword arguments to pass to `xarray.plot.imshow()`.
        For example, the function can be used to plot into an existing
        matplotlib axes object by passing an `ax` keyword argument.
        For more options, see:
        http://xarray.pydata.org/en/stable/generated/xarray.plot.imshow.html  
        
    Returns
    -------
    An RGB plot of one or multiple observations, and optionally an image
    file written to file.
    
    """
    
    # Get names of x and y dims
    y_dim, x_dim = ds.odc.spatial_dims

    # If ax is supplied via kwargs, ignore aspect and size
    if 'ax' in kwargs:

        # Create empty aspect size kwarg that will be passed to imshow
        aspect_size_kwarg = {}
    else:
        # Compute image aspect
        if not aspect:
            aspect = ds.odc.geobox.aspect

        # Populate aspect size kwarg with aspect and size data
        aspect_size_kwarg = {'aspect': aspect, 'size': size}

    # If no value is supplied for `index` (the default), plot using default
    # values and arguments passed via `**kwargs`
    if index is None:

        # Select bands and convert to DataArray
        da = ds.to_array().compute()

        # If percentile_stretch == True, clip plotting to percentile vmin, vmax
        
        kwargs.update({'vmin': vmin, 'vmax': vmax})

        # If there are more than three dimensions and the index dimension == 1,
        # squeeze this dimension out to remove it
        if ((len(ds.dims) > 2) and ('col' not in kwargs) and
            (len(da[index_dim]) == 1)):

            da = da.squeeze(dim=index_dim)

        # If there are more than three dimensions and the index dimension
        # is longer than 1, raise exception to tell user to use 'col'/`index`
        elif ((len(ds.dims) > 2) and ('col' not in kwargs) and
              (len(da[index_dim]) > 1)):

            raise Exception(
                f'The input dataset `ds` has more than two dimensions: '
                f'{list(ds.dims.keys())}. Please select a single observation '
                'using e.g. `index=0`, or enable faceted plotting by adding '
                'the arguments e.g. `col="time", col_wrap=4` to the function '
                'call')

        img = da.plot.imshow(x=x_dim,
                             y=y_dim,
                             robust=robust,
                             col_wrap=col_wrap,
                             **aspect_size_kwarg,
                             **kwargs)
        if titles is not None:
            for ax, title in zip(img.axs.flat, titles):
                ax.set_title(title,fontsize=22)
        img.cbar.ax.tick_params(labelsize=30)
        img.cbar.set_label(label=label, size=30, weight='bold')

    # If values provided for `index`, extract corresponding observations and
    # plot as either single image or facet plot
    else:

        # If a float is supplied instead of an integer index, raise exception
        if isinstance(index, float):
            raise Exception(
                f'Please supply `index` as either an integer or a list of '
                'integers')

        # If col argument is supplied as well as `index`, raise exception
        if 'col' in kwargs:
            raise Exception(
                f'Cannot supply both `index` and `col`; please remove one and '
                'try again')

        # Convert index to generic type list so that number of indices supplied
        # can be computed
        index = index if isinstance(index, list) else [index]

        # Select bands and observations and convert to DataArray
        da = ds.isel(**{index_dim: index}).compute()

        # If percentile_stretch == True, clip plotting to percentile vmin, vmax

        kwargs.update({'vmin': vmin, 'vmax': vmax})

        # If multiple index values are supplied, plot as a faceted plot
        if len(index) > 1:

            img = da.plot.imshow(x=x_dim,
                                 y=y_dim,
                                 robust=robust,
                                 col=index_dim,
                                 col_wrap=col_wrap,
                                 **aspect_size_kwarg,
                                 **kwargs)
            if titles is not None:
                for ax, title in zip(img.axs.flat, titles):
                    ax.set_title(title,fontsize=22)

            img.cbar.ax.tick_params(labelsize=30)
            img.cbar.set_label(label=label, size=30, weight='bold')

        # If only one index is supplied, squeeze out index_dim and plot as a
        # single panel
        else:

            img = da.squeeze(dim=index_dim).plot.imshow(robust=robust,
                                                        **aspect_size_kwarg,
                                                        **kwargs)
            if titles is not None:
                for ax, title in zip(img.axs.flat, titles):
                    ax.set_title(title,fontsize=22)
    

            img.cbar.ax.tick_params(labelsize=30)
            img.cbar.set_label(label=label, size=30, weight='bold')
    # If an export path is provided, save image to file. Individual and
    # faceted plots have a different API (figure vs fig) so we get around this
    # using a try statement:
    if savefig_path:

        print(f'Exporting image to {savefig_path}')

        try:
            img.fig.savefig(savefig_path, **savefig_kwargs)
        except:
            img.figure.savefig(savefig_path, **savefig_kwargs)
    return img


def urban_growth_plot(ds,urban_area,baseline_year,analysis_year):
    """

    Last modified: July 2025

    Rewritten from the DEA notebook here
    https://knowledge.dea.ga.gov.au/notebooks/Real_world_examples/Urban_change_detection/

    Plots urban growth between two specified years using data from a dataset.

    This function visualizes urban extent for a baseline year as a grey background,
    and highlights areas of new urban growth (change from non-urban to urban) 
    between the baseline year and analysis year in red.

    Parameters:
    -----------
    ds : xarray.Dataset
        Dataset containing urban index data with a variable `ENDISI` indexed by `year`.
    
    urban_area : xarray.DataArray
        Binary or categorical DataArray indicating urban extent by year. 
        Values of 1 indicate urban areas, and 0 (or other) indicate non-urban.
    
    baseline_year : int
        The starting year to visualize urban extent.
    
    analysis_year : int
        The ending year to compare against the baseline year to detect urban growth.

    Notes:
    ------
    - The plot shows baseline urban areas in grey.
    - Urban growth hotspots (areas non-urban at baseline but urban at analysis) are highlighted in red.
    - The plot legend indicates growth hotspots, areas that remained urban, and non-urban regions.
    - Adapted from the Digital Earth Australia urban change detection notebook (July 2025).

    Example:
    --------
    >>> urban_growth_plot(ds, urban_area, 2015, 2020)

    
    """
    # Plot urban extent from first year in grey as a background
    plot = ds.ENDISI.sel(year=baseline_year).plot(cmap='Greys',
                                           size=6,
                                           aspect=ds.y.size / ds.y.size,
                                           add_colorbar=False,
                                          
                                          )
  
    # Plot the meaningful change in urban area
    to_urban = '#b91e1e'
    urban_area_diff = urban_area.sel(year=analysis_year)-urban_area.sel(year=baseline_year)
    xr.where(urban_area_diff == 1, 1,
             np.nan).plot(ax=plot.axes,
                          add_colorbar=False,
                          cmap=ListedColormap([to_urban]))
    
    # Add the legend
    plot.axes.legend([Patch(facecolor=to_urban),
                      Patch(facecolor='darkgrey'),
                      Patch(facecolor='white')],
                     ['Urban growth hotspots', 'Remains urban'])
    plt.title('Urban growth between ' + str(baseline_year) + ' and ' +
              str(analysis_year));
    

Fmask plot

In [ ]:
def linear_stretch(band, lower_percent=2, upper_percent=98):
    """
    Perform linear contrast stretching on a single band.

    Parameters:
        band (np.ndarray): 2D array (single band).
        lower_percent (float): Lower percentile for stretch.
        upper_percent (float): Upper percentile for stretch.

    Returns:
        np.ndarray: Stretched band with values normalized to [0,1].
    """
    lower = np.percentile(band, lower_percent)
    upper = np.percentile(band, upper_percent)

    stretched = (band - lower) / (upper - lower)
    stretched = np.clip(stretched, 0, 1)  # limit values to [0,1]
    return stretched

def stretch_rgb(rgb_image, lower_percent=2, upper_percent=98):
    """
    Apply linear contrast stretching to each band in an RGB image.

    Parameters:
        rgb_image (np.ndarray): 3D array (H, W, 3).
    
    Returns:
        np.ndarray: Contrast-stretched RGB image.
    """
    stretched = np.zeros_like(rgb_image)
    for i in range(3):  # For each band R, G, B
        stretched[..., i] = linear_stretch(rgb_image[..., i], lower_percent, upper_percent)
        
    return stretched

In [ ]:
import numpy as np
import math
import numpy as np
import math

def log_stretching_reflectance_optimized(arr,REF_MIN_THRESHOLD=0.01,REF_MAX_THRESHOLD=0.65):
    """
    Apply log stretching optimized for float reflectance values (-1.0 to 1.0).

    This function maps positive reflectance values into an output range of [1, 255].
    Negative or zero values are excluded from the stretch and assigned an output value of 0.

    Parameters:
    arr (np.ndarray): Input array (2D or 3D) of float values, range -1.0 to 1.0.

    Returns:
    np.ndarray: Stretched array (same shape as input) in uint8 format (0-255 range).
    """
    # Define thresholds for the log stretch based on typical display values (e.g., 0.01 to 0.65)
    # These must be positive reflectance values > 0
    #REF_MIN_THRESHOLD = 0.01 
    #REF_MAX_THRESHOLD = 0.65

    # Define the output range for stretched data
    LOW_VALUE = 1
    HIGH_VALUE = 255

    # Calculate log thresholds
    # Ensure thresholds are > 0 before taking the log
    low_thresh_log = math.log(max(REF_MIN_THRESHOLD, 1e-6))
    high_thresh_log = math.log(max(REF_MAX_THRESHOLD, REF_MIN_THRESHOLD))
    thresh_diff = high_thresh_log - low_thresh_log
    
    # Check if a meaningful stretch can be performed
    if thresh_diff == 0:
        print("gen_browse_img: high and low thresholds are equal.")
        return np.zeros_like(arr, dtype=np.uint8)

    # Convert to float32 and ensure it's a copy we can modify
    arr_float = arr.astype(np.float32)
    
    # Identify data points that are valid for log stretching (non-NaN and strictly positive)
    # This also naturally excludes the original -1.0 values
    valid_mask = ~np.isnan(arr_float) & (arr_float > 0)
    
    if not np.any(valid_mask):
        print("gen_browse_img: no valid positive data.")
        return np.zeros_like(arr, dtype=np.uint8)

    # Initialize the output array as uint8, defaulted to 0 (background/invalid)
    output_arr = np.zeros_like(arr, dtype=np.uint8)

    # Use boolean indexing to work only with valid positive data
    valid_data_subset = arr_float[valid_mask]
    
    # 1. Clip the positive data within the defined reflectance thresholds
    clipped_data = np.clip(valid_data_subset, REF_MIN_THRESHOLD, REF_MAX_THRESHOLD)

    # 2. Apply log transformation
    log_transformed_data = np.log(clipped_data)

    # 3. Rescale the log values to the 1-255 range
    stretched_values = HIGH_VALUE * (log_transformed_data - low_thresh_log) / thresh_diff

    # 4. Clip the results to ensure they stay exactly within [LOW_VALUE, HIGH_VALUE]
    final_clipped_values = np.clip(stretched_values, LOW_VALUE, HIGH_VALUE)

    # 5. Place the final uint8 values back into the output array using the mask
    output_arr[valid_mask] = final_clipped_values.astype(np.uint8)
    
    # All non-positive, NaN, or invalid data points remain 0 as intended.
    return output_arr





In [ ]:


def plot_fmask_raster(ax, raster, i=None, fig=None, show_colorbar=False):
    """
    Plot a categorical Fmask raster with a custom colormap and optional colorbar.
    
    Parameters:
        ax (matplotlib.axes.Axes): The axis to plot on.
        raster (np.ndarray): 2D raster array (values 1–4, NaNs allowed).
        i (int, optional): Index of subplot (used for conditional colorbar).
        fig (matplotlib.figure.Figure, optional): Figure handle, required if show_colorbar=True.
        show_colorbar (bool, optional): Whether to display a colorbar (e.g., for last subplot).
    """
    # Define categorical boundaries and colors
    vmin, vmax = 1, 4
    bounds = [1, 2, 3, 4, 5]
    colors_ = ['darkblue', 'cyan', 'grey', 'white']

    custom_cmap = mcolours.ListedColormap(colors_)
    custom_cmap.set_bad(color='black')  # for NaN/masked values
    norm = mcolours.BoundaryNorm(bounds, custom_cmap.N)

    # Plot raster
    ax.imshow(raster, cmap=custom_cmap, norm=norm, interpolation='nearest')

    cax = None
    
    # # Optionally add colorbar
    # if show_colorbar and fig is not None:
    #     cax = fig.add_axes([
    #         ax.get_position().x1 + 0.005,  # slightly right of plot
    #         ax.get_position().y0,         # aligned bottom
    #         0.01,                        # width
    #         ax.get_position().height      # same height
    #     ])

    #     cbar = plt.colorbar(
    #         plt.cm.ScalarMappable(norm=norm, cmap=custom_cmap),
    #         boundaries=bounds,
    #         cax=cax
    #     )

    #     cbar.ax.set_yticks([1.5, 2.5, 3.5, 4.5])
    #     cbar.ax.set_yticklabels(['Water', 'Cloud Shadow', 'Snow/Ice', 'Cloud'])
    #     cbar.ax.tick_params(labelsize=10)

    ax.set_axis_off()

    return cax


def Cirrus_percentile(arr, p_min=10, p_max=90):
    """
    Stretch a cirrus (B09) band to enhance visibility using percentile clipping.

    Parameters:
        arr (np.ndarray): 2D cirrus band array.
        p_min, p_max (float): Percentile limits for contrast stretch.

    Returns:
        np.ndarray: Contrast-stretched cirrus band (float, scaled 0–1).
    """
    arr = arr.astype(np.float32)
    v_min = np.nanpercentile(arr, p_min)
    v_max = np.nanpercentile(arr, p_max)
    arr = np.clip(arr, v_min, v_max)
    return arr.astype(np.uint8)


def plot_cirrus_band(ax, cirrus_band, cmap='viridis', p_min=10, p_max=90, title="Cirrus (B10)"):
    """
    Plot the cirrus band (B10) as a grayscale image with percentile stretching.

    Parameters:
        ax (matplotlib.axes.Axes): Axis to plot on.
        cirrus_band (np.ndarray): 2D cirrus band array.
        cmap (str): Colormap for grayscale visualization (default: 'Greys').
        p_min, p_max (float): Percentile limits for stretch.
        title (str): Plot title.
    """
    stretched = Cirrus_percentile(cirrus_band, p_min, p_max)
    # Get the colormap and set NaN color to black

    cmap = cm.get_cmap(cmap).copy()

    # 2. Set the color for bad values (e.g., NaNs)
    # You can use color names (like 'red', 'white', 'black') or hex codes
    cmap.set_bad(color='black') 

    ax.imshow(stretched, cmap=cmap)
    ax.set_title(title, fontsize=9)
    ax.axis('off')




In [ ]:
# ------------------------------------------------------------
#  Helper function
# ------------------------------------------------------------
def plot_fmask_diff(raster_list, axes, fig):
    """
    Compare cloud and cloud-shadow masks between Fmask 4.7 and Fmask 5.

    Parameters:
        raster_list (list of np.ndarray): [RGB, Fmask4.7, Fmask5, ...].
        axes (list): List of matplotlib Axes for plotting.
        fig (matplotlib.figure.Figure): Figure object for colorbar positioning.
    """
    for j in [4, 2]:  # 4 = Cloud, 2 = Cloud shadow
        # Select subplot based on j
        ax = axes[5] if j == 4 else axes[4]

        # Deep copy so originals are not modified
        raster_fmask_47 = copy.deepcopy(raster_list[1])
        raster_fmask_5  = copy.deepcopy(raster_list[2])

        # --- Binary mask for selected class (cloud or shadow)
        raster_fmask_47 = np.where(raster_fmask_47 == j, 1, 0)
        raster_fmask_5  = np.where(raster_fmask_5 == j, 1, 0)

        # --- Define valid regions (where either detects)
        mask_union = (raster_fmask_47 + raster_fmask_5)
        mask_union = np.where(mask_union != 0, 1, np.nan)

        # --- Compute signed difference
        raster_diff = (raster_fmask_47 - raster_fmask_5) * mask_union

        # --- Visualization setup
        mask_values = [-1, 0, 1]  # -1 = only Fmask5, 0 = same, +1 = only Fmask4.7
        colors_diff = ['darkorange', 'white', 'dodgerblue']
        cmap_diff = mcolours.ListedColormap(colors_diff)
        cmap_diff.set_bad('black')
        norm_diff = mcolours.BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap_diff.N)

        # --- Plot diff map
        im = ax.imshow(raster_diff, cmap=cmap_diff, norm=norm_diff, interpolation='nearest')
        ax.axis("off")

        # --- Add colorbar for cloud shadow subplot only
        cax = None
        
        # if j == 4:
        #     cax = fig.add_axes([
        #         ax.get_position().x1 + 0.002,
        #         ax.get_position().y0,
        #         0.01,
        #         ax.get_position().height
        #     ])
        #     cbar = plt.colorbar(
        #         plt.cm.ScalarMappable(norm=norm_diff, cmap=cmap_diff),
        #         cax=cax
        #     )
        #     cbar.ax.set_yticks([-1, 0, 1])
        #     cbar.ax.set_yticklabels([
        #         "Only Fmask5",
        #         "No difference",
        #         "Only Fmask4.7"
        #     ])

        # --- Add title
        if j == 4:
            ax.set_title("Cloud mask (Fmask 4.7 - Fmask 5)", fontsize=9)
        else:
            ax.set_title("Cloud shadow (Fmask 4.7 - Fmask 5)", fontsize=9)

    return cax

In [ ]:
def plot_fmask_percentage_by_day(day, df_fmask_percentage, axes, mrgs_id=None):
    """
    Plot Fmask percentage barplot for a specific day and MRGS ID.

    Parameters:
        day (str): Date string in Julian format, e.g., '2019187' or '2019175'
        df_fmask_percentage (pd.DataFrame): DataFrame with columns ['Date', 'mrgs_id', 'Features', 'Percentage', 'fmask_version']
        axes (list): List of matplotlib Axes, seaborn will plot on axes[7]
        mrgs_id (str, optional): MRGS ID to filter by. If None, filters by date only. 
                                 The function handles both 'T08VMM' (with T) and '08VMM' (without T) formats.
    """
    # Convert day string to target datetime format
    # day_convert = datetime.strptime(day, '%Y%m%d').strftime('%Y-%m-%d')
    day_convert = day
    
    # Filter dataframe for selected day
    df_filtered = df_fmask_percentage.loc[df_fmask_percentage['Date'] == day_convert]
    
    # Also filter by MRGS ID if provided
    if mrgs_id is not None:
        # Handle both 'T08VMM' and '08VMM' formats in the CSV
        # The CSV stores MRGS IDs with 'T' prefix, but we might receive it without
        mrgs_id_with_t = 'T' + mrgs_id if not mrgs_id.startswith('T') else mrgs_id
        mrgs_id_without_t = mrgs_id.replace('T', '') if mrgs_id.startswith('T') else mrgs_id
        
        # Filter by MRGS ID (try both formats)
        df_filtered = df_filtered.loc[
            (df_filtered['mrgs_id'] == mrgs_id_with_t) | 
            (df_filtered['mrgs_id'] == mrgs_id_without_t)
        ]

    ax7 = axes[7]

    # Check if we have data to plot
    if df_filtered.empty:
        ax7.text(0.5, 0.5, f'No data available\nfor day {day_convert}' + 
                (f'\nMRGS ID: {mrgs_id}' if mrgs_id else ''),
                ha='center', va='center', transform=ax7.transAxes,
                fontsize=12, color='gray')
        ax7.set_xlabel('Features')
        ax7.set_ylabel('Percentage')
        return

    # Plot barplot on ax7
    sns.barplot(
        data=df_filtered,
        x='Features',
        y='Percentage',
        hue='fmask_version',
        order=['Water', 'Snow/Ice', 'Cloud Shadow', 'Cloud'],
        palette='magma',
        ax=ax7
    )


In [ ]:
def share_axes_group(axes, end_index):
    """
    Share x and y axes among subplots up to end_index.
    Works with modern Matplotlib (>=3.6).
    
    Parameters:
        axes (list or np.ndarray): Collection of subplot axes.
        end_index (int): Last index to include in shared group.
    """
    if not isinstance(axes, (list, np.ndarray)):
        axes = np.ravel(axes)

    # Reference axis
    ref_ax = axes[0]

    # Share x/y axes among the specified range
    for ax in axes[:end_index + 1]:
        if ax is not ref_ax:
            ax.sharex(ref_ax)
            ax.sharey(ref_ax)

    

def customize_axes_layout(axes, cax_fmask, cax_fmask_diff, i, fig):
    """
    Adjust colorbar positions, hide ticks and certain axes, and share axes for interactive comparison.

    Parameters:
        axes (list): List of matplotlib Axes.
        cax_fmask (Axes): Colorbar axes for fmask.
        cax_fmask_diff (Axes): Colorbar axes for difference.
        i (int): Current subplot index to control conditional adjustments.
    """
    fig.subplots_adjust(left=0.005, 
                    right=0.90, 
                    top=0.99, 
                    bottom=0.01, 
                    wspace=0.01, hspace=0.01)

    if i in [2, 5]:
        # Adjust colorbar positions
        ax2 = axes[2]
        cax_fmask.set_position([
            ax2.get_position().x1 + 0.0025,
            ax2.get_position().y0,
            0.015,
            ax2.get_position().height
        ])

        ax5 = axes[5]
        # Note: original code used ax4 height for cax_fmask_diff, assuming a typo? Using ax5 here.
        cax_fmask_diff.set_position([
            ax5.get_position().x1 + 0.0025,
            ax5.get_position().y0,
            0.015,
            ax5.get_position().height
        ])

    # Remove x and y ticks from all axes except axes[7]
    for idx, ax in enumerate(axes):
        if idx != 7:
            ax.set_xticks([])
            ax.set_yticks([])

    # Turn off specific axes visibility
    for k in [6, 8]:
        if k < len(axes):
            axes[k].set_visible(False)

    # Share x and y axes among axes 0 to 5 for interactive comparison
    share_axes_group(axes, 5)


In [ ]:
def plot_fmask_comparison_stats(
    df_fmask_percentage,
    fmask4_col='Fmask4',
    fmask5_col='Fmask5',
    figsize=(8.97, 8.69),
    palette='magma',
    violin_gap=0.1,
    scatter_alpha=0.6,
    line_width=1,
    annotation_fontsize=12,
    top_legend_loc='upper left',
    bottom_legend_loc='lower right',
    save_path=None,
    dpi=300
):
    """
    Create a two-panel comparison plot for Fmask 4.7 and 5.0.
    
    Parameters:
    -----------
    df_fmask_percentage : pandas DataFrame
        DataFrame containing Fmask percentage data with columns:
        - 'Features': Feature class names
        - 'Percentage': Cloud coverage percentage
        - 'fmask_version': Fmask version ('Fmask4' or 'Fmask5')
        - 'Date': Date of observation
        - 'mrgs_id': MGRS tile ID
    fmask4_col : str, default='Fmask4'
        Column name for Fmask 4.7 version in the pivot table
    fmask5_col : str, default='Fmask5'
        Column name for Fmask 5.0 version in the pivot table
    figsize : tuple, default=(8.97, 8.69)
        Figure size (width, height) in inches
    palette : str, default='magma'
        Color palette for the violin plot
    violin_gap : float, default=0.1
        Gap between violins in the split violin plot
    scatter_alpha : float, default=0.6
        Transparency of scatter points in regression plot
    line_width : float, default=1
        Width of regression line
    annotation_fontsize : int, default=12
        Font size for R² annotations
    top_legend_loc : str, default='upper left'
        Location of legend in top panel
    bottom_legend_loc : str, default='lower right'
        Location of legend in bottom panel
    save_path : str, optional
        Path to save the figure. If None, figure is not saved.
    dpi : int, default=300
        Resolution for saved figure
    
    Returns:
    --------
    fig : matplotlib.figure.Figure
        The figure object
    axes : numpy.ndarray
        Array of axes objects
    """
    
    # Create figure with two subplots
    fig, axes = plt.subplots(figsize=figsize, nrows=2)
    ax_top = axes[0]
    ax_bottom = axes[1]
    
    # Top panel: Violin plot comparing distributions
    sns.violinplot(
        data=df_fmask_percentage,
        x="Features",
        y="Percentage",
        hue="fmask_version",
        split=True,
        inner="quart",
        fill=False,
        gap=violin_gap,
        palette=palette,
        ax=ax_top
    )
    ax_top.legend(loc=top_legend_loc)
    
    # Bottom panel: Regression plots for correlation analysis
    # Group and aggregate data
    df_grouped = df_fmask_percentage.groupby(['Date', 'mrgs_id', 'fmask_version', 'Features'])
    df_mean = df_grouped['Percentage'].mean().reset_index()
    
    # Pivot table for regression analysis
    df_pivot = df_mean.pivot_table(
        index=['Date', 'mrgs_id', 'Features'],
        columns='fmask_version',
        values='Percentage'
    ).reset_index()
    
    # Get unique feature classes
    unique_classes = np.unique(df_pivot['Features'])
    n_classes = len(unique_classes)
    
    # Calculate y-coordinate positions for annotations dynamically
    y_coords = np.linspace(0.95, 0.65, n_classes)
    
    # Plot regression for each feature class
    for idx, class_name in enumerate(unique_classes):
        df_class = df_pivot.loc[df_pivot['Features'] == class_name]
        
        # Remove NaN values for regression
        df_clean = df_class.dropna(subset=[fmask4_col, fmask5_col])
        
        if len(df_clean) > 0:
            # Create regression plot
            sns.regplot(
                data=df_clean,
                x=fmask4_col,
                y=fmask5_col,
                ax=ax_bottom,
                scatter_kws={'alpha': scatter_alpha},
                line_kws={'linewidth': line_width},
                label=class_name
            )
            
            # Calculate regression statistics
            x = df_clean[fmask4_col]
            y = df_clean[fmask5_col]
            slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
            r_squared = r_value**2
            
            # Annotate with R-squared and p-value
            ax_bottom.annotate(
                f'{class_name}: R² = {r_squared:.2f} (p={p_value:.3f})',
                xy=(0.05, y_coords[idx]),
                xycoords='axes fraction',
                fontsize=annotation_fontsize,
                color='black'
            )
    
    ax_bottom.legend(loc=bottom_legend_loc)
    plt.tight_layout()
    
    # Save figure if path is provided
    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    return fig, axes

In [ ]:
def plot_cloud_coverage_timeseries(

    cloud_coverage_df,

    figsize=(12, 8),

    linewidth=1,

    markersize=4,

    show_stats=True,

    show_plot=True

):

    """

    Plot cloud coverage time series comparison between Fmask 4.7 and 5.0.

    Parameters

    ----------

    cloud_coverage_df : pandas.DataFrame

        DataFrame with columns: 'time', 'fmask4_coverage', 'fmask5_coverage', 'difference'.

        Output from calculate_cloud_coverage_timeseries().

    figsize : tuple, optional

        Figure size (width, height). Default is (12, 8).

    linewidth : float, optional

        Line width for time series plots. Default is 1.

    markersize : float, optional

        Marker size for time series plots. Default is 4.

    show_stats : bool, optional

        If True, print summary statistics. Default is True.

    show_plot : bool, optional

        If True, display the plot using plt.show(). Default is True.

    Returns

    -------

    fig : matplotlib.figure.Figure

        The matplotlib figure object.

    axes : tuple of matplotlib.axes.Axes

        Tuple of (ax1, ax2) for the two subplots.

    """

    import matplotlib.pyplot as plt

    import matplotlib.dates as mdates

    if cloud_coverage_df is None or len(cloud_coverage_df) == 0:

        print("Error: cloud_coverage_df is None or empty")

        return None, None

    # Create figure with two subplots

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    # Plot 1: Time series comparison

    ax1.plot(cloud_coverage_df['time'], cloud_coverage_df['fmask4_coverage'],

             'o-', label='Fmask 4.7', color='#1f77b4', linewidth=linewidth, markersize=markersize)

    ax1.plot(cloud_coverage_df['time'], cloud_coverage_df['fmask5_coverage'],

             's-', label='Fmask 5.0', color='#ff7f0e', linewidth=linewidth, markersize=markersize)

    ax1.set_ylabel('Cloud Coverage (%)', fontsize=12)

    ax1.set_title('Cloud Coverage Time Series Comparison', fontsize=14, fontweight='bold')

    ax1.legend(loc='best', fontsize=11)

    ax1.grid(True, alpha=0.3)

    ax1.set_ylim(bottom=0)

    # Add mean lines

    mean_4 = cloud_coverage_df['fmask4_coverage'].mean()

    mean_5 = cloud_coverage_df['fmask5_coverage'].mean()

    ax1.axhline(y=mean_4, color='#1f77b4', linestyle='--', alpha=0.5, linewidth=1.5)

    ax1.axhline(y=mean_5, color='#ff7f0e', linestyle='--', alpha=0.5, linewidth=1.5)

    ax1.text(ax1.get_xlim()[1]*0.98, mean_4, f'Mean: {mean_4:.2f}%',

             ha='right', va='bottom', color='#1f77b4', fontsize=9, alpha=0.7)

    ax1.text(ax1.get_xlim()[1]*0.98, mean_5, f'Mean: {mean_5:.2f}%',

             ha='right', va='top', color='#ff7f0e', fontsize=9, alpha=0.7)

    # Plot 2: Difference time series

    colors = ['red' if x > 0 else 'blue' if x < 0 else 'gray' for x in cloud_coverage_df['difference']]

    ax2.bar(cloud_coverage_df['time'], cloud_coverage_df['difference'],

            color=colors, alpha=0.6, width=0.8)

    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)

    ax2.set_ylabel('Difference (Fmask 5.0 - 4.7) (%)', fontsize=12)

    ax2.set_xlabel('Time', fontsize=12)

    ax2.set_title('Cloud Coverage Difference Over Time', fontsize=14, fontweight='bold')

    ax2.grid(True, alpha=0.3, axis='y')

    # Add mean difference line

    mean_diff = cloud_coverage_df['difference'].mean()

    ax2.axhline(y=mean_diff, color='purple', linestyle='--', alpha=0.7, linewidth=1.5)

    ax2.text(ax2.get_xlim()[1]*0.98, mean_diff, f'Mean diff: {mean_diff:.2f}%',

             ha='right', va='bottom' if mean_diff > 0 else 'top',

             color='purple', fontsize=9, fontweight='bold')

    # Format x-axis dates

    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

    ax2.xaxis.set_major_locator(mdates.AutoDateLocator())

    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

    plt.tight_layout()

    # Show plot if requested

    if show_plot:

        plt.show()

    # Print summary statistics if requested

    if show_stats:

        print(f"\nSummary Statistics:")

        print(f"  Mean Fmask 4.7 coverage: {mean_4:.2f}%")

        print(f"  Mean Fmask 5.0 coverage: {mean_5:.2f}%")

        print(f"  Mean difference: {mean_diff:.2f}%")

        print(f"  Total time steps: {len(cloud_coverage_df)}")

    return fig, (ax1, ax2)



def visualize_and_export_cloud_frequency(

    fmask4_freq,

    fmask5_freq,

    fmask4_red=None,

    fmask4_green=None,

    fmask4_blue=None,

    output_dir='./outputs',

    mrgs_id=None,

    mask_type="cloud",  # "cloud" or "cloud_shadow"

    save_format=['geotiff', 'png'],

    figsize=(15, 12),

    cmap='viridis',

    vmin=0,

    vmax=1,

    export=False,

    show_plot=True,

    auto_zoom=True,

    extent=None  # Optional: tuple (x_min, x_max, y_min, y_max) to set plot extent

):

    """

    Visualize and optionally export cloud frequency maps for Fmask 4.7 and 5.0.

    Parameters

    ----------

    fmask4_freq : xarray.DataArray

        Cloud frequency map for Fmask 4.7 with dimensions (y, x).

        Values range from 0.0 (never cloud) to 1.0 (always cloud).

    fmask5_freq : xarray.DataArray

        Cloud frequency map for Fmask 5.0 with dimensions (y, x).

        Values range from 0.0 (never cloud) to 1.0 (always cloud).

    fmask4_red : xarray.DataArray, optional

        Red band DataArray for Fmask 4.7 with dimensions (y, x).

        Expected to be a 2D array (median-computed or single time step).

        If provided along with green and blue, an RGB composite will be plotted.

        Default is None.

    fmask4_green : xarray.DataArray, optional

        Green band DataArray for Fmask 4.7 with dimensions (y, x).

        Expected to be a 2D array. Default is None.

    fmask4_blue : xarray.DataArray, optional

        Blue band DataArray for Fmask 4.7 with dimensions (y, x).

        Expected to be a 2D array. Default is None.

    output_dir : str, optional

        Directory to save exported files. Default is './outputs'.

    mrgs_id : str, optional

        MGRS tile identifier for filename. If None, uses 'unknown'.

    mask_type : str, optional

        Type of mask being visualized. "cloud" or "cloud_shadow". Default is "cloud".

    save_format : list, optional

        List of formats to export: 'geotiff', 'png', 'netcdf', 'jpg'.

        Default is ['geotiff', 'png']. Only used if export=True.

    figsize : tuple, optional

        Figure size for visualization (width, height). Default is (15, 12).

        Layout is 2x2: top row shows cloud frequencies, bottom row shows RGB and difference.

    cmap : str, optional

        Colormap name for visualization. Default is 'viridis'.

    vmin, vmax : float, optional

        Color scale limits. Default is 0 and 1.

    export : bool, optional

        If True, export files to disk. If False, only visualize.

        Default is False.

    show_plot : bool, optional

        If True, display the plot using plt.show(). Default is True.

    auto_zoom : bool, optional

        If True, automatically applies shared zoom to all three plot panels,

        ensuring they have the same spatial extent for direct comparison.

        Calculates the union of data bounds from all datasets and applies

        the same xlim/ylim to all axes. Default is True.

    extent : tuple, optional

        Optional tuple (x_min, x_max, y_min, y_max) to set a custom plot extent.

        If provided, this extent will override the auto_zoom calculation and be applied

        to all plot panels. If None, auto_zoom will calculate extent from data bounds.

        Default is None.

    Returns

    -------

    tuple or dict

        If export=False: Returns (fig, axes) tuple for further customization.

        If export=True: Returns dict with paths to saved files.

    """

    import os

    import numpy as np

    import matplotlib.pyplot as plt

    import matplotlib.colors as mcolors

    from pathlib import Path

    import xarray as xr

    # Set MGRS ID for filenames

    tile_id = mrgs_id if mrgs_id else 'unknown'

    # Calculate difference map

    difference = fmask5_freq - fmask4_freq

    # Determine if RGB data is available

    has_rgb = all([fmask4_red is not None, fmask4_green is not None, fmask4_blue is not None])

    # Create 2x2 subplot layout

    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True, sharey=True)

    axes = axes.flatten()

    # Add figure title with MRGS ID

    if mrgs_id:

        mask_name = "Cloud" if mask_type == "cloud" else "Cloud Shadow"

        fig.suptitle(f'{mask_name} Frequency Comparison - MRGS ID: {mrgs_id}',

                    fontsize=16, fontweight='bold', y=0.98)

    # Set equal aspect ratio for all axes

    for ax in axes:

        ax.set_aspect('equal', adjustable='box')

    # Plot Fmask 4.7 frequency (top left)

    im1 = fmask4_freq.plot(

        ax=axes[0],

        cmap=cmap,

        vmin=vmin,

        vmax=vmax,

        add_colorbar=True,

        cbar_kwargs={'label': f'{mask_name} Frequency'}

    )

    axes[0].set_title(f'Fmask 4.7 {mask_name} Frequency')

    axes[0].set_xlabel('Longitude')

    axes[0].set_ylabel('Latitude')

    # Plot Fmask 5.0 frequency (top right)

    im2 = fmask5_freq.plot(

        ax=axes[1],

        cmap=cmap,

        vmin=vmin,

        vmax=vmax,

        add_colorbar=True,

        cbar_kwargs={'label': f'{mask_name} Frequency'}

    )

    axes[1].set_title(f'Fmask 5.0 {mask_name} Frequency')

    axes[1].set_xlabel('Longitude')

    axes[1].set_ylabel('Latitude')

    # Plot RGB composite or difference map in bottom row

    bottom_left_idx = 2

    bottom_right_idx = 3

    if has_rgb:

        try:

            # Use RGB bands directly (expected to be 2D arrays without time dimension)

            red_median = fmask4_red

            green_median = fmask4_green

            blue_median = fmask4_blue

            # Prepare bands for log stretching (simplified version)

            def prepare_band_for_log_stretch(band):

                """

                Prepare band data for log_stretching_reflectance_optimized."""
                band_data = band.values.copy().astype(np.float32)
                band_data = np.where(band_data == -9999, np.nan, band_data)

                valid_mask = ~np.isnan(band_data)

                if not np.any(valid_mask):

                    return band_data
                valid_data = band_data[valid_mask]

                p_min = np.nanpercentile(valid_data, 0.5)

                p_max = np.nanpercentile(valid_data, 99.5)

                if p_max <= p_min:

                    return np.zeros_like(band_data)

                normalized = (band_data - p_min) / (p_max - p_min)

                reflectance_range = 0.001 + normalized * (0.8 - 0.001)

                reflectance_range[~valid_mask] = np.nan

                return reflectance_range

            def normalize_band_to_01(band_data):

                """

                Normalize band to 0-1 range using its own min/max."""
                band_min = np.nanmin(band_data)
                band_max = np.nanmax(band_data)

                if band_max > band_min:

                    normalized = (band_data - band_min) / (band_max - band_min)

                    normalized = np.where(np.isnan(band_data), np.nan, normalized)

                    return normalized

                else:

                    return np.zeros_like(band_data)

            # Prepare and process bands

            red_prep = prepare_band_for_log_stretch(red_median)

            green_prep = prepare_band_for_log_stretch(green_median)

            blue_prep = prepare_band_for_log_stretch(blue_median)

            red_norm = normalize_band_to_01(red_prep)

            green_norm = normalize_band_to_01(green_prep)

            blue_norm = normalize_band_to_01(blue_prep)

            REF_MIN = 0.01

            REF_MAX = 0.65

            red_scaled = red_norm * (REF_MAX - REF_MIN) + REF_MIN

            green_scaled = green_norm * (REF_MAX - REF_MIN) + REF_MIN

            blue_scaled = blue_norm * (REF_MAX - REF_MIN) + REF_MIN

            # Apply log stretching

            red_stretched = log_stretching_reflectance_optimized(red_scaled, REF_MIN_THRESHOLD=REF_MIN, REF_MAX_THRESHOLD=REF_MAX).astype(np.float32) / 255.0

            green_stretched = log_stretching_reflectance_optimized(green_scaled, REF_MIN_THRESHOLD=REF_MIN, REF_MAX_THRESHOLD=REF_MAX).astype(np.float32) / 255.0

            blue_stretched = log_stretching_reflectance_optimized(blue_scaled, REF_MIN_THRESHOLD=REF_MIN, REF_MAX_THRESHOLD=REF_MAX).astype(np.float32) / 255.0

            # Stack into RGB array

            rgb_array = np.stack([red_stretched, green_stretched, blue_stretched], axis=-1).astype(np.float32)

            rgb_array = np.clip(rgb_array, 0, 1)

            # Plot RGB composite (bottom left)

            axes[bottom_left_idx].imshow(

                rgb_array,

                extent=[red_median.x.min().values, red_median.x.max().values,

                       red_median.y.min().values, red_median.y.max().values],

                origin='upper',

                aspect='equal'

            )

            axes[bottom_left_idx].set_title('Fmask 4.7 RGB Composite (Median)')

            axes[bottom_left_idx].set_xlabel('Longitude')

            axes[bottom_left_idx].set_ylabel('Latitude')

        except Exception as e:

            print(f"Warning: Error creating RGB composite: {e}. Skipping RGB composite.")

            bottom_left_idx = 2

    # Plot difference map (bottom right, or bottom left if no RGB)

    diff_max = max(abs(difference.min().values), abs(difference.max().values))

    diff_ax_idx = bottom_right_idx if has_rgb else bottom_left_idx

    im3 = difference.plot(

        ax=axes[diff_ax_idx],

        cmap='RdBu_r',

        vmin=-diff_max,

        vmax=diff_max,

        add_colorbar=True,

        cbar_kwargs={'label': f'Difference ({mask_name})'}

    )

    axes[diff_ax_idx].set_title('Difference Map')

    axes[diff_ax_idx].set_xlabel('Longitude')

    axes[diff_ax_idx].set_ylabel('Latitude')

    # Hide unused subplot if RGB is not available

    if not has_rgb:

        axes[bottom_right_idx].set_visible(False)

    # Apply shared zoom if requested

    if auto_zoom:

        # If extent is provided, use it directly

        if extent is not None:

            x_min, x_max, y_min, y_max = extent

            for ax in axes:

                if ax.get_visible():

                    ax.set_xlim(x_min, x_max)

                    ax.set_ylim(y_min, y_max)

        else:

            # Calculate extent from data bounds

            def get_data_bounds(data_array):

                """

                Get bounding box of valid (non-NaN) data."""

                valid_mask = ~np.isnan(data_array.values)

                if not valid_mask.any():

                        return None

                if 'x' in data_array.dims and 'y' in data_array.dims:

                        x_coords = data_array.x.values

                y_coords = data_array.y.values

                valid_y, valid_x = np.where(valid_mask)

                if len(valid_y) == 0 or len(valid_x) == 0:

                        return None

                x_min = x_coords[valid_x.min()]

                x_max = x_coords[valid_x.max()]

                y_min = y_coords[valid_y.min()]

                y_max = y_coords[valid_y.max()]

                return (x_min, x_max, y_min, y_max)


            bounds_list = []

            for data in [fmask4_freq, fmask5_freq, difference]:

                bounds = get_data_bounds(data)

                if bounds:

                    bounds_list.append(bounds)

            if has_rgb:

                try:

                    rgb_bounds = get_data_bounds(fmask4_red)

                    if rgb_bounds:

                        bounds_list.append(rgb_bounds)

                except Exception as e:

                    print(f"Warning: Could not get RGB data bounds for auto_zoom: {e}")

            if bounds_list:

                x_mins, x_maxs, y_mins, y_maxs = zip(*bounds_list)

                x_min = min(x_mins)

                x_max = max(x_maxs)

                y_min = min(y_mins)

                y_max = max(y_maxs)

                x_pad = (x_max - x_min) * 0.02

                y_pad = (y_max - y_min) * 0.02

                for ax in axes:

                    if ax.get_visible():

                        ax.set_xlim(x_min - x_pad, x_max + x_pad)

                        ax.set_ylim(y_min - y_pad, y_max + y_pad)

    # Adjust subplot spacing

    top_margin = 0.92 if mrgs_id else 0.95

    plt.subplots_adjust(left=0.05, right=0.95, top=top_margin, bottom=0.05,

                       wspace=0.1, hspace=0.1)

    plt.tight_layout(rect=[0.05, 0.05, 0.95, top_margin])

    # Show plot if requested

    if show_plot:

        plt.show()

    # Export files only if export=True

    saved_files = {}

    if export:

        output_path = Path(output_dir)

        output_path.mkdir(parents=True, exist_ok=True)

        # Save figure

        if 'png' in save_format or 'jpg' in save_format:

            for fmt in ['png', 'jpg']:

                if fmt in save_format:

                    mask_suffix = "cloud" if mask_type == "cloud" else "cloud_shadow"

                    fig_path = output_path / f'{mask_suffix}_frequency_{tile_id}.{fmt}'

                    plt.savefig(fig_path, dpi=300, bbox_inches='tight')

                    saved_files['figure'] = str(fig_path)

                    print(f"Saved figure: {fig_path}")

        # Export as GeoTIFF

        if 'geotiff' in save_format:

            try:

                import rioxarray

                mask_suffix = "cloud" if mask_type == "cloud" else "cloud_shadow"

                fmask4_path = output_path / f'fmask4_{mask_suffix}_frequency_{tile_id}.tif'

                fmask4_freq.rio.to_raster(fmask4_path)

                saved_files['fmask4_freq'] = str(fmask4_path)

                print(f"Saved Fmask 4.7 frequency map: {fmask4_path}")

                fmask5_path = output_path / f'fmask5_{mask_suffix}_frequency_{tile_id}.tif'

                fmask5_freq.rio.to_raster(fmask5_path)

                saved_files['fmask5_freq'] = str(fmask5_path)

                print(f"Saved Fmask 5.0 frequency map: {fmask5_path}")

                diff_path = output_path / f'{mask_suffix}_frequency_difference_{tile_id}.tif'

                difference.rio.to_raster(diff_path)

                saved_files['difference'] = str(diff_path)

                print(f"Saved difference map: {diff_path}")

            except Exception as e:

                print(f"Warning: Could not export GeoTIFF files: {e}")

        # Export as NetCDF

        if 'netcdf' in save_format:

            try:

                ds = xr.Dataset({

                    'fmask4_frequency': fmask4_freq,

                    'fmask5_frequency': fmask5_freq,

                    'difference': difference

                })

                mask_suffix = "cloud" if mask_type == "cloud" else "cloud_shadow"

                nc_path = output_path / f'{mask_suffix}_frequency_{tile_id}.nc'

                ds.to_netcdf(nc_path)

                saved_files['netcdf'] = str(nc_path)

                print(f"Saved NetCDF file: {nc_path}")

            except Exception as e:

                print(f"Warning: Could not export NetCDF file: {e}")

        saved_files['fig'] = fig

        saved_files['axes'] = axes

        plt.close()

        return saved_files

    else:

        return fig, axes

def calculate_cloud_frequency_per_pixel(fmask_data, mask_value=4):

    """

    Calculate frequency per pixel for a specific mask value from time-series Fmask DataArray.

    Can be used for cloud (mask_value=4) or cloud shadow (mask_value=2).

    Parameters

    ----------

    fmask_data : xarray.DataArray

        Time-series Fmask DataArray with dimensions (time, y, x).

        Values: 1=water, 2=cloud_shadow, 3=snow_ice, 4=cloud, NaN=clear.

    mask_value : int, optional

        Fmask value to calculate frequency for. Default is 4 (cloud).

        Use 2 for cloud shadow, 4 for cloud.

    Returns

    -------

    xarray.DataArray

        Frequency map with dimensions (y, x).

        Values range from 0.0 (never detected) to 1.0 (always detected).

    """

    import numpy as np

    if fmask_data is None:

        return None

    # Count mask occurrences (value == mask_value) along time dimension

    mask_count = (fmask_data == mask_value).sum(dim='time')

    # Count valid (non-NaN) observations along time dimension

    valid_count = (~np.isnan(fmask_data)).sum(dim='time')

    # Calculate frequency: mask_count / valid_count

    mask_frequency = mask_count / valid_count

    # Handle division by zero (set to NaN where no valid observations)

    mask_frequency = mask_frequency.where(valid_count > 0)

    return mask_frequency

In [ ]:
def batch_time_series_comparison(
    mrgs_id_list,
    bucket,
    fmask4_files,
    fmask5_files,
    output_dir='./outputs',
    min_valid_ratio=0.6,
    vmin=0.0,
    vmax=1.0,
    cmap='viridis',
    figsize=(15, 12),
    export=False,
    show_plot=True,
    auto_zoom=True,
    mask_value=4,  # 4 for cloud, 2 for cloud shadow
    extent=None  # Optional: tuple (x_min, x_max, y_min, y_max) to set plot extent
):
    """
    Batch process time series comparison (Section 5 workflow) for multiple MRGS IDs.
    
    For each MRGS ID, this function:
    1. Loads Fmask 4.7 and 5.0 time-series data
    2. Filters RGB bands to keep only time steps with sufficient valid pixels
    3. Calculates frequency per pixel for both Fmask versions (cloud or cloud shadow)
    3.5. Calculates and plots cloud coverage time series (line plots showing coverage over time)
    4. Creates a 2x2 panel plot showing:
       - Top left: Fmask 4.7 Frequency
       - Top right: Fmask 5.0 Frequency
       - Bottom left: RGB Composite (if available)
       - Bottom right: Difference Map
    
    Parameters
    ----------
    mrgs_id_list : list of str
        List of MGRS tile IDs to process (e.g., ['10UEV', '31UDQ', '11SLT']).
    bucket : str
        S3 bucket name containing the Fmask data.
    fmask4_files : str
        S3 link/pattern to Fmask 4.7 files. Can be:
        - Full S3 URI with MRGS ID: 's3://bucket/path/31UDQ/'
        - S3 URI with placeholder: 's3://bucket/path/{mrgs_id}/'
        - S3 URI ending with '/': 's3://bucket/path/' (MRGS ID will be appended)
        - List of file keys: ['key1', 'key2', ...]
    fmask5_files : str
        S3 link/pattern to Fmask 5.0 files. Same format options as fmask4_files.
        Example: 's3://hls-debug-output/Fmask5_outputs/TS/S30/{mrgs_id}/'
    output_dir : str, optional
        Directory to save exported files. Default is './outputs'.
    min_valid_ratio : float, optional
        Minimum ratio of valid pixels required for RGB time steps (0.0-1.0).
        Default is 0.6 (60%).
    vmin, vmax : float, optional
        Color scale limits for cloud frequency maps. Default is 0.0 and 1.0.
    cmap : str, optional
        Colormap name for cloud frequency visualization. Default is 'viridis'.
    figsize : tuple, optional
        Figure size (width, height) in inches. Default is (15, 12).
    export : bool, optional
        If True, export plots and data files for each MRGS ID. Default is False.
    show_plot : bool, optional
        If True, display each plot using plt.show(). Default is True.
    auto_zoom : bool, optional
        If True, applies shared zoom to all panels for direct comparison.
        Default is True.
    mask_value : int, optional
        Fmask value to analyze. Default is 4 (cloud). Use 2 for cloud shadow.
    extent : tuple, optional
        Optional tuple (x_min, x_max, y_min, y_max) to set a custom plot extent.
        If provided, this extent will override the auto_zoom calculation and be applied
        to all plot panels. If None, auto_zoom will calculate extent from data bounds.
        Default is None.
    
    Returns
    -------
    dict
        Dictionary with MRGS IDs as keys and results as values:
        {
            'mrgs_id_1': {
                'status': 'success',
                'fmask4_frequency': xarray.DataArray,
                'fmask5_frequency': xarray.DataArray,
                'fig': matplotlib.figure.Figure,  # 2x2 cloud frequency plot
                'axes': numpy.ndarray,  # Axes for cloud frequency plot
                'cloud_coverage_df': pandas.DataFrame,  # Time series data
                'timeseries_fig': matplotlib.figure.Figure,  # Time series plot
                'timeseries_axes': tuple,  # Axes for time series plot
                'exported_files': dict (if export=True)
            },
            ...
        }
    
    Example
    -------
    >>> from module.data_access import load_fmask_pair
    >>> from module.plotting import batch_time_series_comparison
    >>> 
    >>> mrgs_ids = ['10UEV', '31UDQ', '11SLT', '42QWM']
    >>> results = batch_time_series_comparison(
    ...     mrgs_id_list=mrgs_ids,
    ...     bucket='hls-debug-output',
    ...     fmask4_files='s3://hls-debug-output/Fmask4_outputs/TS/S30/{mrgs_id}/',
    ...     fmask5_files='s3://hls-debug-output/Fmask5_outputs/TS/S30/{mrgs_id}/',
    ...     export=True,
    ...     show_plot=True
    ... )
    """
    import numpy as np
    import xarray as xr
    from pathlib import Path
    import re
    import matplotlib.pyplot as plt
    
    # Helper function to construct S3 path with MRGS ID
    def _construct_s3_path_with_mrgs(base_path, mrgs_id):
        """
        Construct S3 path by inserting MRGS ID into the path.
        
        Handles multiple patterns:
        1. Path contains '{mrgs_id}' or '{MRGS_ID}' placeholder: replaces it with actual MRGS ID
        2. Path ends with '/': appends MRGS ID to the path
        3. Path already contains the MRGS ID: uses as-is
        4. Path is a list: returns as-is (no modification)
        
        Examples:
        - 's3://bucket/path/{mrgs_id}/' -> 's3://bucket/path/31UDQ/'
        - 's3://bucket/path/' -> 's3://bucket/path/31UDQ/'
        - 's3://bucket/path/31UDQ/' -> 's3://bucket/path/31UDQ/' (unchanged)
        - 's3://hls-debug-output/Fmask5_outputs/TS/S30/{mrgs_id}/' -> 's3://hls-debug-output/Fmask5_outputs/TS/S30/31UDQ/'
        """
        if isinstance(base_path, list):
            # If it's already a list, return as-is
            return base_path
        
        base_path = str(base_path).strip()
        
        # Check if path contains {mrgs_id} placeholder (case-insensitive)
        if '{mrgs_id}' in base_path.lower():
            # Replace placeholder with actual MRGS ID (case-insensitive replacement)
            # Replace {mrgs_id} or {MRGS_ID} with actual MRGS ID
            path = re.sub(r'\{mrgs_id\}', mrgs_id, base_path, flags=re.IGNORECASE)
        elif base_path.endswith('/'):
            # If path ends with '/', append MRGS ID
            path = base_path + mrgs_id + '/'
        else:
            # Check if MRGS ID is already in the path (as a directory component)
            # Look for MRGS ID as a standalone path component (between slashes or at end)
            if re.search(r'/' + re.escape(mrgs_id) + r'(/|$)', base_path):
                # MRGS ID already in path as a directory, use as-is
                path = base_path
            else:
                # Append MRGS ID to path
                path = base_path.rstrip('/') + '/' + mrgs_id + '/'
        return path
    
    # Note: This function assumes the following are available in the namespace:
    # - load_fmask_pair (from module.data_access)
    # - calculate_cloud_frequency_per_pixel (from this module)
    # - visualize_and_export_cloud_frequency (from this module)
    # - filter_valid_time_steps (from this module)
    # - calculate_cloud_coverage_timeseries (from module/fmask)
    # - plot_cloud_coverage_timeseries (from this module)
    # These should be loaded via %run in the notebook before calling this function
    
    results = {}
    
    mask_name = "Cloud" if mask_value == 4 else "Cloud Shadow" if mask_value == 2 else f"Mask {mask_value}"
    
    print(f"{'='*80}")
    print(f"Batch {mask_name} Time Series Comparison")
    print(f"Processing {len(mrgs_id_list)} MRGS IDs: {mrgs_id_list}")
    print(f"{'='*80}\n")
    
    for idx, mrgs_id in enumerate(mrgs_id_list, 1):
        print(f"\n{'='*80}")
        print(f"Processing MRGS ID {idx}/{len(mrgs_id_list)}: {mrgs_id}")
        print(f"{'='*80}")
        
        try:
            # Step 1: Load Fmask 4.7 and 5.0 time-series data
            print(f"\n[Step 1] Loading Fmask data for {mrgs_id}...")
            
            # Construct S3 paths with MRGS ID
            fmask4_path = _construct_s3_path_with_mrgs(fmask4_files, mrgs_id)
            fmask5_path = _construct_s3_path_with_mrgs(fmask5_files, mrgs_id)
            
            print(f"  Fmask 4.7 path: {fmask4_path if isinstance(fmask4_path, str) else 'list of files'}")
            print(f"  Fmask 5.0 path: {fmask5_path if isinstance(fmask5_path, str) else 'list of files'}")
            
            fmask_pair = load_fmask_pair(
                mrgs_id,
                bucket=bucket,
                fmask4_files=fmask4_path,
                fmask5_files=fmask5_path,
            )
            
            fmask4_data = fmask_pair.get("fmask4")
            fmask5_data = fmask_pair.get("fmask5")
            
            if fmask4_data is None or fmask5_data is None:
                print(f"  ⚠ Warning: Missing data for {mrgs_id}. Skipping...")
                results[mrgs_id] = {
                    'status': 'skipped',
                    'reason': 'missing_data'
                }
                continue
            
            print(f"  ✓ Loaded Fmask 4.7: {fmask4_data.shape if fmask4_data is not None else 'None'}")
            print(f"  ✓ Loaded Fmask 5.0: {fmask5_data.shape if fmask5_data is not None else 'None'}")
            
            # Step 2: Filter RGB bands
            print(f"\n[Step 2] Filtering RGB bands for {mrgs_id}...")
            fmask4_red_raw = fmask_pair.get("fmask4_red")
            fmask4_green_raw = fmask_pair.get("fmask4_green")
            fmask4_blue_raw = fmask_pair.get("fmask4_blue")
            
            # Replace -9999 with NaN for all RGB bands
            if fmask4_red_raw is not None:
                fmask4_red_raw = fmask4_red_raw.where(fmask4_red_raw != -9999, np.nan)
            if fmask4_green_raw is not None:
                fmask4_green_raw = fmask4_green_raw.where(fmask4_green_raw != -9999, np.nan)
            if fmask4_blue_raw is not None:
                fmask4_blue_raw = fmask4_blue_raw.where(fmask4_blue_raw != -9999, np.nan)
            
            # Filter RGB bands and calculate medians
            fmask4_red_median = None
            fmask4_green_median = None
            fmask4_blue_median = None
            
            has_rgb = all([fmask4_red_raw is not None, 
                          fmask4_green_raw is not None, 
                          fmask4_blue_raw is not None])
            
            if has_rgb:
                # Filter each band
                if fmask4_red_raw is not None:
                    fmask4_red = filter_valid_time_steps(fmask4_red_raw, min_valid_ratio=min_valid_ratio)
                    if fmask4_red is not None:
                        fmask4_red_median = fmask4_red.median(dim='time')
                
                if fmask4_green_raw is not None:
                    fmask4_green = filter_valid_time_steps(fmask4_green_raw, min_valid_ratio=min_valid_ratio)
                    if fmask4_green is not None:
                        fmask4_green_median = fmask4_green.median(dim='time')
                
                if fmask4_blue_raw is not None:
                    fmask4_blue = filter_valid_time_steps(fmask4_blue_raw, min_valid_ratio=min_valid_ratio)
                    if fmask4_blue is not None:
                        fmask4_blue_median = fmask4_blue.median(dim='time')
                
                # Check if all medians are available
                has_rgb = all([fmask4_red_median is not None,
                              fmask4_green_median is not None,
                              fmask4_blue_median is not None])
                
                if has_rgb:
                    print(f"  ✓ RGB bands filtered and medians calculated")
                else:
                    print(f"  ⚠ Warning: Some RGB bands missing after filtering")
            
            # Step 3: Calculate frequency per pixel
            print(f"\n[Step 3] Calculating {mask_name.lower()} frequency per pixel for {mrgs_id}...")
            fmask4_frequency = calculate_cloud_frequency_per_pixel(fmask4_data, mask_value=mask_value)
            fmask5_frequency = calculate_cloud_frequency_per_pixel(fmask5_data, mask_value=mask_value)
            
            print(f"  ✓ Fmask 4.7 {mask_name.lower()} frequency: {fmask4_frequency.shape}, "
                  f"range: {fmask4_frequency.min().values:.3f} to {fmask4_frequency.max().values:.3f}")
            print(f"  ✓ Fmask 5.0 {mask_name.lower()} frequency: {fmask5_frequency.shape}, "
                  f"range: {fmask5_frequency.min().values:.3f} to {fmask5_frequency.max().values:.3f}")
            
            # Step 3.5: Calculate and plot coverage time series
            print(f"\n[Step 3.5] Calculating and plotting {mask_name.lower()} coverage time series for {mrgs_id}...")
            cloud_coverage_df = None
            timeseries_fig = None
            timeseries_axes = None
            
            try:
                # Note: calculate_cloud_coverage_timeseries should be available from module/fmask
                # Note: plot_cloud_coverage_timeseries should be available from this module
                
                cloud_coverage_df = calculate_cloud_coverage_timeseries(
                    fmask4_data,
                    fmask5_data,
                    verbose=False,  # Suppress verbose output in batch mode
                    mask_value=mask_value
                )
                
                if cloud_coverage_df is not None and len(cloud_coverage_df) > 0:
                    # Plot coverage time series (don't show yet, we'll add title first)
                    timeseries_fig, timeseries_axes = plot_cloud_coverage_timeseries(
                        cloud_coverage_df,
                        figsize=(12, 8),
                        linewidth=1,
                        markersize=4,
                        show_stats=False,  # Suppress stats in batch mode
                        show_plot=False  # Don't show yet, we'll add title first
                    )
                    
                    # Add MRGS ID to the figure title
                    if timeseries_fig is not None:
                        timeseries_fig.suptitle(f'{mask_name} Coverage Time Series - MRGS ID: {mrgs_id}', 
                                               fontsize=14, fontweight='bold', y=0.98)
                        plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for title
                    
                    # Show plot if requested
                    if show_plot:
                        plt.show()
                    
                    print(f"  ✓ {mask_name} coverage time series plotted ({len(cloud_coverage_df)} time steps)")
                else:
                    print(f"  ⚠ Warning: Could not calculate {mask_name.lower()} coverage time series")
            except Exception as e:
                print(f"  ⚠ Warning: Error creating {mask_name.lower()} coverage time series plot: {e}")
                import traceback
                traceback.print_exc()
            
            # Step 4: Visualize and optionally export
            print(f"\n[Step 4] Creating visualization for {mrgs_id}...")
            
            mask_type_str = mask_name.lower().replace(' ', '_')  # "cloud" or "cloud_shadow"
            if has_rgb:
                result = visualize_and_export_cloud_frequency(
                    fmask4_freq=fmask4_frequency,
                    fmask5_freq=fmask5_frequency,
                    fmask4_red=fmask4_red_median,
                    fmask4_green=fmask4_green_median,
                    fmask4_blue=fmask4_blue_median,
                    output_dir=output_dir,
                    mrgs_id=mrgs_id,
                    vmin=vmin,
                    vmax=vmax,
                    cmap=cmap,
                    figsize=figsize,
                    export=export,
                    show_plot=show_plot,
                    auto_zoom=auto_zoom,
                    extent=extent,
                    mask_type=mask_type_str
                )
            else:
                result = visualize_and_export_cloud_frequency(
                    fmask4_freq=fmask4_frequency,
                    fmask5_freq=fmask5_frequency,
                    output_dir=output_dir,
                    mrgs_id=mrgs_id,
                    vmin=vmin,
                    vmax=vmax,
                    cmap=cmap,
                    figsize=figsize,
                    export=export,
                    show_plot=show_plot,
                    auto_zoom=auto_zoom,
                    extent=extent,
                    mask_type=mask_type_str
                )
            
            # Handle return value (tuple if export=False, dict if export=True)
            if export and isinstance(result, dict):
                # When export=True, function returns dict with exported files
                fig = result.get('fig')
                axes = result.get('axes')
                exported_files = result
            else:
                # When export=False, function returns (fig, axes) tuple
                fig, axes = result
                exported_files = None
            
            # Store results
            result_dict = {
                'status': 'success',
                'fmask4_frequency': fmask4_frequency,
                'fmask5_frequency': fmask5_frequency,
                'fig': fig,  # 2x2 cloud frequency plot
                'axes': axes,
                'cloud_coverage_df': cloud_coverage_df,  # Time series DataFrame
                'timeseries_fig': timeseries_fig,  # Time series plot figure
                'timeseries_axes': timeseries_axes  # Time series plot axes
            }
            
            if exported_files is not None:
                result_dict['exported_files'] = exported_files
            
            results[mrgs_id] = result_dict
            print(f"  ✓ Completed processing {mrgs_id}")
            
        except Exception as e:
            print(f"  ✗ Error processing {mrgs_id}: {e}")
            import traceback
            traceback.print_exc()
            results[mrgs_id] = {
                'status': 'error',
                'error': str(e)
            }
    
    print(f"\n{'='*80}")
    print(f"Batch processing complete!")
    print(f"  Successfully processed: {sum(1 for r in results.values() if r.get('status') == 'success')}")
    print(f"  Skipped: {sum(1 for r in results.values() if r.get('status') == 'skipped')}")
    print(f"  Errors: {sum(1 for r in results.values() if r.get('status') == 'error')}")
    print(f"{'='*80}\n")
    
    return results


In [ ]:
def plot_fmask_statistics_summary(
    df_fmask_percentage=None,
    csv_path=None,
    figsize=(24, 12),
    save_path='figures/fmask_statistics_summary.png',
    dpi=300,
    show_plot=True,
    style='whitegrid'
):
    """
    Generate comprehensive summary statistics plots from df_fmask_percentage DataFrame.
    
    Creates a 2x3 grid of plots showing:
    1. Distribution of Cloud Difference (histogram)
    2. Distribution of Cloud Shadow Difference (histogram)
    3. Feature Percentages: Fmask4.7 vs Fmask5.0 (violin plot)
    4. Mean Cloud Difference by MRGS ID (bar chart)
    5. Summary Statistics Table
    
    Parameters
    ----------
    df_fmask_percentage : pandas.DataFrame, optional
        DataFrame containing Fmask percentage comparison data. Must have columns:
        - 'Date': Date string in Julian format (YYYYDDD)
        - 'mrgs_id': MRGS tile ID
        - 'Cloud_fmask4', 'Cloud_fmask5': Cloud percentages for each version
        - 'CloudShadow_fmask4', 'CloudShadow_fmask5': Cloud shadow percentages
        - 'Water_fmask4', 'Water_fmask5': Water percentages
        - 'SnowIce_fmask4', 'SnowIce_fmask5': Snow/Ice percentages
        - 'Cloud_diff', 'CloudShadow_diff': Difference columns
        If None, will attempt to load from csv_path.
    csv_path : str, optional
        Path to CSV file containing df_fmask_percentage data.
        Only used if df_fmask_percentage is None. Default is 'df_fmask_percentage.csv'.
    figsize : tuple, optional
        Figure size (width, height) in inches. Default is (18, 12).
    save_path : str, optional
        Path to save the figure. If None, figure is not saved.
        Default is 'figures/fmask_statistics_summary.png'.
    dpi : int, optional
        Resolution for saved figure. Default is 300.
    show_plot : bool, optional
        If True, display the plot using plt.show(). Default is True.
    style : str, optional
        Seaborn style for plots. Default is 'whitegrid'.
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        The matplotlib figure object.
    axes : list
        List of matplotlib axes objects for each subplot.
    
    Example
    -------
    >>> from module.plotting import plot_fmask_statistics_summary
    >>> import pandas as pd
    >>> 
    >>> # Load data
    >>> df = pd.read_csv('df_fmask_percentage.csv')
    >>> 
    >>> # Generate plots
    >>> fig, axes = plot_fmask_statistics_summary(
    ...     df_fmask_percentage=df,
    ...     save_path='figures/summary.png',
    ...     show_plot=True
    ... )
    """
    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from datetime import datetime
    from scipy import stats as scipy_stats
    
    # Load data if not provided
    if df_fmask_percentage is None:
        if csv_path is None:
            csv_path = 'df_fmask_percentage.csv'
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV file not found: {csv_path}")
        df_fmask_percentage = pd.read_csv(csv_path)
    
    # Validate required columns
    required_cols = ['Date', 'mrgs_id', 'Cloud_fmask4', 'Cloud_fmask5', 
                     'CloudShadow_fmask4', 'CloudShadow_fmask5',
                     'Cloud_diff', 'CloudShadow_diff']
    missing_cols = [col for col in required_cols if col not in df_fmask_percentage.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Set style
    sns.set_style(style)
    
    # Print summary info
    print(f'Total granules: {len(df_fmask_percentage)}')
    print(f'Unique MRGS IDs: {df_fmask_percentage["mrgs_id"].nunique()}')
    print(f'Date range: {df_fmask_percentage["Date"].min()} to {df_fmask_percentage["Date"].max()}')
    
    # Create figure with 2x3 grid layout (5 plots: 2 histograms, 1 violin, 1 bar, 1 table)
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)
    
    # 1. Distribution of Cloud Difference
    ax1 = fig.add_subplot(gs[0, 0])
    cloud_diff = df_fmask_percentage['Cloud_diff'].dropna()
    ax1.hist(cloud_diff, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(cloud_diff.mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {cloud_diff.mean():.2f}%')
    ax1.axvline(cloud_diff.median(), color='green', linestyle='--', linewidth=2, 
               label=f'Median: {cloud_diff.median():.2f}%')
    ax1.set_xlabel('Cloud Difference (Fmask5 - Fmask4) %', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.set_title('Distribution of Cloud Coverage Difference', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Distribution of Cloud Shadow Difference
    ax2 = fig.add_subplot(gs[0, 1])
    shadow_diff = df_fmask_percentage['CloudShadow_diff'].dropna()
    ax2.hist(shadow_diff, bins=50, edgecolor='black', alpha=0.7, color='orange')
    ax2.axvline(shadow_diff.mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {shadow_diff.mean():.2f}%')
    ax2.axvline(shadow_diff.median(), color='green', linestyle='--', linewidth=2, 
               label=f'Median: {shadow_diff.median():.2f}%')
    ax2.set_xlabel('Cloud Shadow Difference (Fmask5 - Fmask4) %', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title('Distribution of Cloud Shadow Difference', fontsize=12, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Violin plot: Feature percentages comparison
    ax3 = fig.add_subplot(gs[0, 2])
    feature_data = []
    for feature in ['Water', 'CloudShadow', 'SnowIce', 'Cloud']:
        for version in ['fmask4', 'fmask5']:
            col = f'{feature}_{version}'
            if col in df_fmask_percentage.columns:
                values = df_fmask_percentage[col].dropna()
                for val in values:
                    feature_data.append({
                        'Feature': feature, 
                        'Version': version.replace('fmask', 'Fmask '), 
                        'Percentage': val
                    })
    df_features = pd.DataFrame(feature_data)
    if not df_features.empty:
        sns.violinplot(data=df_features, x='Feature', y='Percentage', hue='Version', ax=ax3)
        ax3.set_title('Feature Percentages: Fmask4.7 vs Fmask5.0', fontsize=12, fontweight='bold')
        ax3.set_ylabel('Percentage (%)', fontsize=11)
        ax3.tick_params(axis='x', rotation=45)
        ax3.legend(title='Version')
    
    # 4. Mean differences by MRGS ID (top 10)
    ax4 = fig.add_subplot(gs[1, 0])
    top_mrgs = df_fmask_percentage['mrgs_id'].value_counts().head(10).index
    df_top = df_fmask_percentage[df_fmask_percentage['mrgs_id'].isin(top_mrgs)]
    mrgs_diff = df_top.groupby('mrgs_id')['Cloud_diff'].mean().sort_values()
    ax4.barh(range(len(mrgs_diff)), mrgs_diff.values, color='coral')
    ax4.axvline(0, color='black', linestyle='-', linewidth=1)
    ax4.set_yticks(range(len(mrgs_diff)))
    ax4.set_yticklabels(mrgs_diff.index)
    ax4.set_xlabel('Mean Cloud Difference (%)', fontsize=11)
    ax4.set_title('Mean Cloud Difference by MRGS ID (Top 10)', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='x')
    
    # 5. Summary statistics table
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.axis('off')
    stats_data = {
        'Metric': ['Mean Cloud Diff', 'Std Cloud Diff', 'Mean Shadow Diff', 'Std Shadow Diff',
                   'Mean Cloud Fmask4', 'Mean Cloud Fmask5', 'Mean Shadow Fmask4', 'Mean Shadow Fmask5'],
        'Value': [
            f"{df_fmask_percentage['Cloud_diff'].mean():.2f}%",
            f"{df_fmask_percentage['Cloud_diff'].std():.2f}%",
            f"{df_fmask_percentage['CloudShadow_diff'].mean():.2f}%",
            f"{df_fmask_percentage['CloudShadow_diff'].std():.2f}%",
            f"{df_fmask_percentage['Cloud_fmask4'].mean():.2f}%",
            f"{df_fmask_percentage['Cloud_fmask5'].mean():.2f}%",
            f"{df_fmask_percentage['CloudShadow_fmask4'].mean():.2f}%",
            f"{df_fmask_percentage['CloudShadow_fmask5'].mean():.2f}%"
        ]
    }
    stats_df = pd.DataFrame(stats_data)
    table = ax4.table(cellText=stats_df.values, colLabels=stats_df.columns, 
                      cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(14)
    table.scale(1, 2)
    ax4.set_title('Summary Statistics', fontsize=16, fontweight='bold', pad=20)
    
    plt.suptitle('Fmask 4.7 vs 5.0 Statistical Summary', fontsize=20, fontweight='bold', y=0.995)
    
    # Save figure if path is provided
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f'\n✓ Saved summary statistics plot to {save_path}')
    
    # Show plot if requested
    if show_plot:
        plt.show()
    
    # Collect all axes
    axes = [ax1, ax2, ax3, ax4]
    
    return fig, axes


In [ ]:
# Global Köppen–Geiger major-class raster (to be set by the user)
# Expected to be an xarray.DataArray with dimensions ('lat', 'lon') and
# integer values 1–5 representing major Köppen–Geiger classes from Beck et al. (2023)
# [https://www.nature.com/articles/s41597-023-02549-6].
# 1 = Equatorial (A), 2 = Arid (B), 3 = Warm temperate (C), 4 = Snow (D), 5 = Polar (E)
koppen_da = None


def load_koppen_geiger_raster(raster_path=None, resolution='1km'):
    """
    Load Köppen–Geiger climate classification raster from Beck et al. (2023).
    
    Parameters
    ----------
    raster_path : str, optional
        Path to the Köppen–Geiger raster file. If None, uses default path:
        '/Users/vo25-mbp/Documents/koppen_geiger_tif/1991_2020/koppen_geiger_0p00833333.tif'
    resolution : str, optional
        Resolution to use if raster_path is None. Options: '1km' (default), '0.1deg', '0.5deg', '1.0deg'.
        Default is '1km' which uses the highest resolution file.
    
    Returns
    -------
    xarray.DataArray
        DataArray with dimensions ('y', 'x') and coordinates ('lat', 'lon').
        Values are integer codes: 1=Equatorial (A), 2=Arid (B), 3=Warm temperate (C),
        4=Snow (D), 5=Polar (E).
    
    Notes
    -----
    The raster is loaded using rioxarray and converted to have lat/lon coordinates
    for easy spatial querying. The global variable `koppen_da` is automatically set.
    """
    global koppen_da
    
    import rioxarray as rxr
    import xarray as xr
    
    if raster_path is None:
        # Default path based on user's download location
        base_path = '/Users/vo25-mbp/Documents/koppen_geiger_tif/1991_2020'
        
        # Map resolution string to filename
        resolution_map = {
            '1km': 'koppen_geiger_0p00833333.tif',      # ~1 km resolution
            '0.1deg': 'koppen_geiger_0p1.tif',          # 0.1 degree
            '0.5deg': 'koppen_geiger_0p5.tif',          # 0.5 degree
            '1.0deg': 'koppen_geiger_1p0.tif'           # 1.0 degree
        }
        
        filename = resolution_map.get(resolution, resolution_map['1km'])
        raster_path = f'{base_path}/{filename}'
    
    print(f"Loading Köppen–Geiger raster from: {raster_path}")
    
    # Load raster using rioxarray
    da = rxr.open_rasterio(raster_path)
    
    # Remove band dimension if present (assuming single band)
    if 'band' in da.dims:
        da = da.squeeze('band', drop=True)
    
    # Ensure CRS is WGS84 (EPSG:4326) for lat/lon coordinates
    if hasattr(da, 'rio') and da.rio.crs is not None:
        if str(da.rio.crs) != 'EPSG:4326':
            # Reproject to WGS84 if needed
            da = da.rio.reproject('EPSG:4326')
    
    # rioxarray uses 'x' and 'y' as dimension names, with coordinates also named 'x' and 'y'
    # For WGS84, x = longitude and y = latitude
    # Create aliases for easier querying: assign lat/lon as additional coordinates
    if 'x' in da.coords and 'y' in da.coords:
        # For rioxarray, x is longitude and y is latitude in WGS84
        da = da.assign_coords(lon=('x', da.x.values), lat=('y', da.y.values))
    
    # Set global variable
    koppen_da = da
    
    print(f"✓ Loaded Köppen–Geiger raster: {da.shape}, CRS: {da.rio.crs if hasattr(da, 'rio') else 'unknown'}")
    print(f"  Coordinate ranges: lat [{da.lat.min().item():.2f}, {da.lat.max().item():.2f}], "
          f"lon [{da.lon.min().item():.2f}, {da.lon.max().item():.2f}]")
    
    return da


def vectorize_koppen_geiger_to_shapefile(
    raster_path='/Users/vo25-mbp/Documents/koppen_geiger_tif/1991_2020/koppen_geiger_0p00833333.tif',
    output_path='koppen_geiger_major_zones.shp',
    simplify_tolerance=None,
    target_resolution_deg=None
):
    """
    Vectorize Köppen–Geiger raster into three climate-zone polygons.

    Classes are defined from legend.txt codes:
    - Arid (B): codes 4, 5, 6, 7
    - Cold (D+E): codes 17-29
    - Temperate (A+C): codes 1-3, 8-16

    Parameters
    ----------
    raster_path : str
        Path to the Köppen–Geiger raster file.
    output_path : str
        Output shapefile path (.shp).
    simplify_tolerance : float, optional
        If provided, simplify geometries with this tolerance (in CRS units).
    target_resolution_deg : float, optional
        If provided (and raster is in EPSG:4326), resample to this degree resolution
        using nearest-neighbor before vectorization. Strongly recommended for the
        0.008333° (~1 km) global raster.

    Returns
    -------
    geopandas.GeoDataFrame
        Vectorized climate zone polygons with zone_code and zone_name.
    """
    import os
    import numpy as np
    import geopandas as gpd
    import rasterio
    from rasterio import features

    zone_names = {1: 'Arid', 2: 'Cold', 3: 'Temperate'}

    with rasterio.open(raster_path) as src:
        crs = src.crs
        nodata = src.nodata

        # Optional resampling to reduce polygon count / runtime
        if target_resolution_deg is not None and str(crs) == 'EPSG:4326':
            from rasterio.enums import Resampling
            xres, yres = src.res
            scale = float(target_resolution_deg) / float(abs(xres))
            new_width = max(1, int(src.width / scale))
            new_height = max(1, int(src.height / scale))
            data = src.read(
                1,
                out_shape=(new_height, new_width),
                resampling=Resampling.nearest,
            )
            transform = src.transform * src.transform.scale(
                src.width / new_width,
                src.height / new_height,
            )
        else:
            data = src.read(1)
            transform = src.transform

    # Build 3-class raster (0 = nodata/unclassified)
    zone = np.zeros(data.shape, dtype=np.uint8)
    if nodata is not None:
        data = np.where(data == nodata, 0, data)

    zone[np.isin(data, [4, 5, 6, 7])] = 1  # Arid
    zone[np.isin(data, list(range(17, 30)))] = 2  # Cold
    zone[np.isin(data, [1, 2, 3] + list(range(8, 17)))] = 3  # Temperate

    mask = zone > 0

    # Polygonize and dissolve to one geometry per zone
    from shapely.geometry import shape as shapely_shape
    from shapely.ops import unary_union

    geoms_by_zone = {1: [], 2: [], 3: []}
    for geom, value in features.shapes(zone, mask=mask, transform=transform):
        z = int(value)
        if z > 0:
            geoms_by_zone[z].append(shapely_shape(geom))

    rows = []
    for z, geoms in geoms_by_zone.items():
        if not geoms:
            continue
        rows.append(
            {
                'zone_code': z,
                'zone_name': zone_names[z],
                'geometry': unary_union(geoms),
            }
        )

    gdf = gpd.GeoDataFrame(rows, crs=crs)

    if simplify_tolerance is not None:
        gdf['geometry'] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)

    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    gdf.to_file(output_path, driver='ESRI Shapefile')

    return gdf


def classify_climate_zone(lat, lon):
    """Classify climate zone using Köppen–Geiger maps (Beck et al., 2023).

    This function samples a global Köppen–Geiger *major class* raster (1 km)
    such as the dataset of Beck et al. (2023) \[`Sci Data`, 10, 724\]
    \([link](https://www.nature.com/articles/s41597-023-02549-6)\).

    It expects a global variable ``koppen_da`` to be set to an
    ``xarray.DataArray`` with dimensions (``lat``, ``lon``) and integer
    major-class codes:

    - 1: Equatorial (A)
    - 2: Arid (B)
    - 3: Warm temperate (C)
    - 4: Snow (D)
    - 5: Polar (E)

    For the purposes of this study, these are grouped into three broad
    climate categories:

    - "Arid"      → major class B (code 2)
    - "Cold"      → major classes D/E (codes 4–5)
    - "Temperate" → major classes A/C (codes 1 and 3)

    Parameters
    ----------
    lat : float
        Latitude in degrees.
    lon : float
        Longitude in degrees.

    Returns
    -------
    str
        Climate zone label: 'Arid', 'Cold', or 'Temperate'.

    Notes
    -----
    - Before calling this function, load the Köppen–Geiger major-class map, e.g.::

        import xarray as xr
        from hls_validation.module import plotting
        plotting.koppen_da = xr.open_dataarray("/path/to/koppen_major_1991-2020.tif")

      or set ``koppen_da`` to the appropriate DataArray in the current namespace.
    """
    global koppen_da
    import numpy as np

    if koppen_da is None:
        # Try to auto-load from default path
        try:
            load_koppen_geiger_raster()
        except Exception as e:
            raise RuntimeError(
                f"Global `koppen_da` is not set and auto-loading failed: {e}\n"
                "Please load a Köppen–Geiger major-class raster (e.g., Beck et al. 2023, 1991–2020 period) "
                "using `load_koppen_geiger_raster()` or assign it to `plotting.koppen_da`."
            ) from e

    def _get_code_at_coord(lat_coord, lon_coord):
        """Helper function to get Köppen code at a specific coordinate."""
        val = None
        try:
            # Try lat/lon coordinates (if we assigned them)
            val = koppen_da.sel(lat=lat_coord, lon=lon_coord, method="nearest")
        except (KeyError, ValueError):
            try:
                # Try x/y coordinates (rioxarray standard: x=lon, y=lat)
                val = koppen_da.sel(x=lon_coord, y=lat_coord, method="nearest")
            except (KeyError, ValueError):
                # Fallback: use isel with closest coordinate values
                if 'x' in koppen_da.coords and 'y' in koppen_da.coords:
                    x_idx = np.abs(koppen_da.x.values - lon_coord).argmin()
                    y_idx = np.abs(koppen_da.y.values - lat_coord).argmin()
                    val = koppen_da.isel(x=x_idx, y=y_idx)
        
        # Extract scalar value
        if val is not None:
            if hasattr(val, 'item'):
                val = val.item()
            elif hasattr(val, 'values'):
                val = val.values
            if isinstance(val, (list, np.ndarray)):
                val = val[0] if len(val) > 0 else None
        
        try:
            code = int(val) if val is not None else None
            return code
        except (TypeError, ValueError):
            return None
    
    def _is_valid_code(code):
        """Check if code is a valid Köppen-Geiger code for classification."""
        if code is None:
            return False
        # Valid codes: 1-30 (based on the classification scheme)
        # Exclude 0 (ocean/unclassified) and NaN
        return 1 <= code <= 30
    
    # First, try direct lookup at the exact coordinate
    code = _get_code_at_coord(lat, lon)
    
    # If we got a valid code, classify it
    if _is_valid_code(code):
        # Map Köppen-Geiger codes to three broad categories based on legend.txt
        # Arid (B): codes 4, 5, 6, 7 (BWh, BWk, BSh, BSk)
        if code in (4, 5, 6, 7):
            return "Arid"
        # Cold (D+E): codes 17-30 (Cold Dsa-Dfd, Polar ET-EF), including code 29
        elif code >= 17 and code <= 29:
            return "Cold"
        # Temperate (A+C): codes 1-3 (Tropical Af, Am, Aw) + codes 8-16 (Temperate Csa-Cfc)
        elif (code >= 1 and code <= 3) or (code >= 8 and code <= 16):
            return "Temperate"
    
    # If direct lookup failed or returned invalid code, search in expanding bounding box
    # This handles coastal tiles and other edge cases
    # Start with small step size (approximately 0.01 degrees ~ 1km)
    step_size = 0.01
    max_search_radius = 0.5  # Maximum search radius in degrees (~50km)
    max_steps = int(max_search_radius / step_size)
    
    # Determine raster resolution for adaptive step size
    try:
        if 'x' in koppen_da.coords and 'y' in koppen_da.coords:
            lon_res = float(np.abs(np.diff(koppen_da.x.values[:2])).mean())
            lat_res = float(np.abs(np.diff(koppen_da.y.values[:2])).mean())
            step_size = max(abs(lon_res), abs(lat_res)) * 2  # Use 2x resolution as step
        elif 'lon' in koppen_da.coords and 'lat' in koppen_da.coords:
            lon_res = float(np.abs(np.diff(koppen_da.lon.values[:2])).mean())
            lat_res = float(np.abs(np.diff(koppen_da.lat.values[:2])).mean())
            step_size = max(abs(lon_res), abs(lat_res)) * 2
    except:
        pass  # Use default step_size if resolution detection fails
    
    # Search in expanding rings around the point
    for step in range(1, max_steps + 1):
        search_radius = step * step_size
        
        # Generate search points in a ring pattern
        # Use 8 directions (N, NE, E, SE, S, SW, W, NW) plus intermediate points
        n_points = min(8 * step, 32)  # Limit number of points to check
        angles = np.linspace(0, 2 * np.pi, n_points, endpoint=False)
        
        for angle in angles:
            # Calculate offset coordinates
            # cos(angle) for longitude (east-west), sin(angle) for latitude (north-south)
            lon_offset = lon + search_radius * np.cos(angle)
            lat_offset = lat + search_radius * np.sin(angle)
            
            # Check if coordinates are within raster bounds
            try:
                if 'x' in koppen_da.coords and 'y' in koppen_da.coords:
                    if (lon_offset < koppen_da.x.min() or lon_offset > koppen_da.x.max() or
                        lat_offset < koppen_da.y.min() or lat_offset > koppen_da.y.max()):
                        continue
                elif 'lon' in koppen_da.coords and 'lat' in koppen_da.coords:
                    if (lon_offset < koppen_da.lon.min() or lon_offset > koppen_da.lon.max() or
                        lat_offset < koppen_da.lat.min() or lat_offset > koppen_da.lat.max()):
                        continue
            except:
                pass  # Continue if bounds check fails
            
            # Try to get code at this offset coordinate
            code = _get_code_at_coord(lat_offset, lon_offset)
            
            # If we found a valid code, classify and return
            if _is_valid_code(code):
                # Map Köppen-Geiger codes to three broad categories
                if code in (4, 5, 6, 7):
                    return "Arid"
                elif code >= 17 and code <= 29:
                    return "Cold"
                elif (code >= 1 and code <= 3) or (code >= 8 and code <= 16):
                    return "Temperate"
    
    # If no valid code found after expanding search, use fallback
    # Default to Temperate as a conservative fallback
    return "Polar"


def _simplify_koppen_raster_for_plotting(koppen_da, target_resolution=0.5):
    """
    Simplify Köppen-Geiger raster for faster plotting by resampling to coarser resolution.
    
    Parameters
    ----------
    koppen_da : xarray.DataArray
        Original high-resolution Köppen-Geiger raster
    target_resolution : float, optional
        Target resolution in degrees (default 0.5 degrees)
    
    Returns
    -------
    xarray.DataArray
        Simplified raster with target resolution
    """
    import xarray as xr
    import numpy as np
    
    # Determine current resolution
    if 'x' in koppen_da.coords and 'y' in koppen_da.coords:
        lon_res = float(np.abs(np.diff(koppen_da.x.values[:2])).mean())
        lat_res = float(np.abs(np.diff(koppen_da.y.values[:2])).mean())
        current_res = max(abs(lon_res), abs(lat_res))
    elif 'lon' in koppen_da.coords and 'lat' in koppen_da.coords:
        lon_res = float(np.abs(np.diff(koppen_da.lon.values[:2])).mean())
        lat_res = float(np.abs(np.diff(koppen_da.lat.values[:2])).mean())
        current_res = max(abs(lon_res), abs(lat_res))
    else:
        # Fallback: assume high resolution and resample anyway
        current_res = 0.008  # ~1km
    
    # Calculate resampling factor
    factor = max(1, int(np.ceil(target_resolution / current_res)))
    
    if factor <= 1:
        # Already coarse enough
        return koppen_da
    
    print(f"Simplifying Köppen raster: {current_res:.4f}° -> {target_resolution}° (factor: {factor}x)")
    
    # Resample using coarsen (much faster than interpolate)
    try:
        # Use coarsen to downsample
        dims_to_coarsen = {}
        if 'x' in koppen_da.dims:
            dims_to_coarsen['x'] = factor
        if 'y' in koppen_da.dims:
            dims_to_coarsen['y'] = factor
        if 'lon' in koppen_da.dims:
            dims_to_coarsen['lon'] = factor
        if 'lat' in koppen_da.dims:
            dims_to_coarsen['lat'] = factor
        
        # For categorical data, we need mode but xarray doesn't have it
        # For fast display, use first() which takes the first value in each window
        # This is fast and adequate since we're just creating a background visualization
        # and will remap codes to 3 categories anyway
        simplified = koppen_da.coarsen(dims_to_coarsen, boundary='trim').first()
        
        # Ensure integer dtype for categorical codes
        if simplified.dtype != koppen_da.dtype:
            simplified = simplified.astype(koppen_da.dtype)
        
        return simplified
    except Exception as e:
        print(f"Warning: Could not coarsen raster, using original: {e}")
        return koppen_da


def plot_spatial_mean_differences(
    df_fmask_percentage,
    figsize=(16, 12),
    save_path=None,
    dpi=300,
    show_plot=True,
    climate_zone_stats_results=None
):
    """
    Create spatial maps showing average cloud and cloud shadow differences per MRGS ID,
    with Köppen-Geiger climate classification and violin plots showing distribution across climate zones.
    
    Calculates mean differences per MRGS ID (same aggregation as bar chart) and displays
    them on geographic maps. Creates a 2x1 grid:
    - Row 1: Cloud difference map with violin plot inset in lower left corner
    - Row 2: Cloud shadow difference map with violin plot inset in lower left corner
    
    Parameters
    ----------
    df_fmask_percentage : pandas.DataFrame
        DataFrame containing Fmask percentage comparison data. Must have columns:
        - 'mrgs_id': MRGS tile ID
        - 'Cloud_diff': Cloud difference (Fmask5 - Fmask4)
        - 'CloudShadow_diff': Cloud shadow difference (Fmask5 - Fmask4)
    figsize : tuple, optional
        Figure size (width, height) in inches. Default is (16, 12) for 2x1 layout.
    save_path : str, optional
        Path to save the figure. If None, figure is not saved.
    dpi : int, optional
        Resolution for saved figure. Default is 300.
    show_plot : bool, optional
        If True, display the plot using plt.show(). Default is True.
    climate_zone_stats_results : dict, optional
        Pre-calculated climate zone comparison analysis results from perform_climate_zone_comparison_analysis().
        If provided, violin plot statistics (median, N, p-value) will be taken from this
        instead of recalculating. Should contain 'medians' key with 'cloud_medians', 'shadow_medians',
        'sample_sizes', and 'median_significance_tests' sub-keys. If None, will check globals() for
        'climate_zone_stats_results'.
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        The matplotlib figure object.
    axes : numpy.ndarray
        Array of matplotlib axes objects (2x1 grid).
    """
    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import geopandas as gpd
    import matplotlib.cm as cm
    from matplotlib.colors import Normalize, TwoSlopeNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    import seaborn as sns
    import xarray as xr
    
    try:
        import mgrs
    except ImportError as e:
        raise ImportError(
            "The 'mgrs' package is required for MGRS tile mapping. "
            "Install it in this environment with: %pip install mgrs"
        ) from e
    
# Calculate average cloud/shadow differences per MRGS ID (for map visualization)
    # Note: Map shows one point per MRGS tile, violin plots use per-MRGS aggregated statistics
    # Calculate average differences per MRGS ID, using climate_zone from df_fmask_percentage if available
    # Since all rows for the same MRGS ID have the same climate_zone, we can aggregate both
    # the differences and the climate_zone column directly
    
    # Check if climate_zone column exists in input DataFrame
    agg_dict = {
        'Cloud_diff': 'mean',
        'CloudShadow_diff': 'mean'
    }
    
    if 'climate_zone' in df_fmask_percentage.columns:
        # Include climate_zone in aggregation (use 'first' since all rows for same MRGS ID have same zone)
        agg_dict['climate_zone'] = 'first'
        print("Using climate_zone column from df_fmask_percentage for aggregation")
    else:
        print("Warning: 'climate_zone' column not found in df_fmask_percentage. Will classify manually.")
    
    df_mean = df_fmask_percentage.groupby('mrgs_id').agg(agg_dict).reset_index()
    
    # Prepare per-MRGS aggregated data for violin plots
    # Use df_mean (already aggregated per MRGS ID) for violin plots
    
    print(f"Calculated average differences per MRGS ID for {len(df_mean)} unique tiles (for map)")
    print(f"Using per-MRGS aggregated data for violin plots: {len(df_mean)} MRGS IDs (will be filtered)")
    
    # Convert MRGS IDs to lat/lon (for map points - using aggregated data)
    m = mgrs.MGRS()
    lats = []
    lons = []
    cloud_diffs = []
    shadow_diffs = []
    original_mrgs_ids = []
    
    for idx, row in df_mean.iterrows():
        mrgs_id = str(row['mrgs_id']).strip()
        mgrs_str = mrgs_id[1:] if mrgs_id.startswith('T') else mrgs_id
        
        try:
            lat, lon = m.toLatLon(f"{mgrs_str} 50000 50000")
            lats.append(lat)
            lons.append(lon)
            cloud_diffs.append(row['Cloud_diff'])
            shadow_diffs.append(row['CloudShadow_diff'])
            original_mrgs_ids.append(mrgs_id)
        except Exception as e:
            continue
    
    # Prepare climate zones for violin plots
    # If climate_zone was included in aggregation, use it directly; otherwise classify manually
    if 'climate_zone' in df_mean.columns:
        # Use climate_zone from aggregation
        print("Using climate_zone from aggregated data for violin plots")
        df_mean_clean = df_mean.copy()
    else:
        # Fallback: classify climate zones manually (for backward compatibility)
        print("Classifying climate zones for per-MRGS violin plots...")
        climate_zones_list_mrgs = []
        
        for idx, row in df_mean.iterrows():
            mrgs_id = str(row['mrgs_id']).strip()
            mgrs_str = mrgs_id[1:] if mrgs_id.startswith('T') else mrgs_id
            
            try:
                lat, lon = m.toLatLon(f"{mgrs_str} 50000 50000")
                climate_zone = classify_climate_zone(lat, lon)
                climate_zones_list_mrgs.append(climate_zone)
            except Exception as e:
                climate_zones_list_mrgs.append(None)
                continue
        
        df_mean_clean = df_mean.copy()
        df_mean_clean['climate_zone'] = climate_zones_list_mrgs
    
    # Filter out None/NaN values
    df_mean_clean = df_mean_clean.dropna(subset=['climate_zone', 'Cloud_diff', 'CloudShadow_diff'])
    
    # Filter to only include defined climate zones (Arid, Cold, Temperate)
    # This ensures consistency with analysis cells and excludes any undefined zones
    defined_zones = ['Arid', 'Cold', 'Temperate']
    df_mean_clean = df_mean_clean[df_mean_clean['climate_zone'].isin(defined_zones)]
    
    # DEBUG: expose df_mean_clean so external notebooks can compare MRGS IDs and climate zones
    global df_mean_clean_spatial
    df_mean_clean_spatial = df_mean_clean.copy()
    
    # Convert to numpy arrays for violin plots (per-MRGS aggregated)
    climate_zones_mrgs = np.array(df_mean_clean['climate_zone'].values)  # Per-MRGS for violin plots
    cloud_diffs_mrgs = np.array(df_mean_clean['Cloud_diff'].values)
    shadow_diffs_mrgs = np.array(df_mean_clean['CloudShadow_diff'].values)
    
    print(f"Per-MRGS data prepared: {len(df_mean_clean)} MRGS IDs with defined climate zones (Arid/Cold/Temperate)")
    global koppen_da
    if koppen_da is None:
        print("\nLoading Köppen–Geiger climate classification raster...")
        load_koppen_geiger_raster()  # Uses default path
    
    # Classify climate zones for each MRGS tile (for map visualization)
    climate_zones = []
    for lat, lon in zip(lats, lons):
        climate_zones.append(classify_climate_zone(lat, lon))
    
    climate_zones = np.array(climate_zones)
    print(f"Climate zone distribution:")
    unique_zones, counts = np.unique(climate_zones, return_counts=True)
    for zone, count in zip(unique_zones, counts):
        print(f"  {zone}: {count} tiles ({100*count/len(climate_zones):.1f}%)")
    
    # Load Natural Earth 110m country boundaries (local copy)
    from pathlib import Path

    local_candidates = [
        Path('ne_110m_admin_0_countries.zip'),
        Path('hls_validation/ne_110m_admin_0_countries.zip'),
        Path('../ne_110m_admin_0_countries.zip'),
    ]
    try:
        local_candidates.append(Path(__file__).resolve().parent.parent / 'ne_110m_admin_0_countries.zip')
    except NameError:
        pass

    local_path = next((p for p in local_candidates if p.exists()), None)
    if local_path is None:
        raise FileNotFoundError(
            "Local Natural Earth file not found. Expected one of: "
            f"{', '.join(str(p) for p in local_candidates)}"
        )

    world = gpd.read_file(str(local_path))

    # Load Köppen major-zone polygons (3 classes) from local shapefile (fast overlay)
    koppen_zones_gdf = None
    try:
        from pathlib import Path
        koppen_candidates = [
            Path('hls_validation/koppen_geiger_major_zones/koppen_geiger_major_zones.shp'),
            Path('hls_validation/koppen_geiger_major_zones.shp'),
            Path('koppen_geiger_major_zones.shp'),
            Path('koppen_geiger_major_zones/koppen_geiger_major_zones.shp'),
        ]
        koppen_path = next((p for p in koppen_candidates if p.exists()), None)
        if koppen_path is not None:
            koppen_zones_gdf = gpd.read_file(str(koppen_path))
            # Ensure lon/lat
            if koppen_zones_gdf.crs is None:
                koppen_zones_gdf = koppen_zones_gdf.set_crs('EPSG:4326')
            elif str(koppen_zones_gdf.crs) != 'EPSG:4326':
                koppen_zones_gdf = koppen_zones_gdf.to_crs('EPSG:4326')
            print(f"Using Köppen zone shapefile overlay: {koppen_path}")
        else:
            print("Warning: Köppen zone shapefile not found; will fall back to raster overlay if available")
    except Exception as e:
        print(f"Warning: Could not load Köppen zone shapefile: {e}")
        koppen_zones_gdf = None
    
    # Simplify Köppen raster once for both plots (if available)
    koppen_simplified = None
    if koppen_da is not None:
        try:
            print("Simplifying Köppen raster for faster plotting...")
            koppen_simplified = _simplify_koppen_raster_for_plotting(koppen_da, target_resolution=0.5)
        except Exception as e:
            print(f"Warning: Could not simplify Köppen raster: {e}")
            koppen_simplified = None
    
    # Create figure with 2x1 grid (only maps, no separate histogram panels)
    fig, axes = plt.subplots(2, 1, figsize=(figsize[0], figsize[1]))
    ax_map_cloud = axes[0]  # Cloud map
    ax_map_shadow = axes[1]  # Shadow map
    
    # Convert to numpy arrays for easier handling
    lats = np.array(lats)
    lons = np.array(lons)
    cloud_diffs = np.array(cloud_diffs)
    shadow_diffs = np.array(shadow_diffs)
    original_mrgs_ids = np.array(original_mrgs_ids)
    
    # Determine color scale limits (symmetric around zero for diverging colormap)
    cloud_diff_max = np.nanmax(np.abs(cloud_diffs))
    shadow_diff_max = np.nanmax(np.abs(shadow_diffs))
    
    # Calculate point sizes based on magnitude of differences
    # Scale absolute differences to a reasonable size range (e.g., 20-200)
    cloud_diff_abs = np.abs(cloud_diffs)
    shadow_diff_abs = np.abs(shadow_diffs)
    
    # Identify hotspots: top 10% by absolute difference or above threshold
    # For cloud: top 10% or above 10% absolute difference
    # For shadow: top 10% or above 3% absolute difference
    # Calculate thresholds (90th percentile or fixed threshold, whichever is higher)
    cloud_threshold = max(np.percentile(cloud_diff_abs, 90), 10.0)
    shadow_threshold = max(np.percentile(shadow_diff_abs, 90), 3.0)
    
    # Identify hotspot indices
    cloud_hotspot_mask = cloud_diff_abs >= cloud_threshold
    shadow_hotspot_mask = shadow_diff_abs >= shadow_threshold
    
    print(f"Identified {np.sum(cloud_hotspot_mask)} cloud hotspots (threshold: {cloud_threshold:.2f}%)")
    print(f"Identified {np.sum(shadow_hotspot_mask)} shadow hotspots (threshold: {shadow_threshold:.2f}%)")
    
    # Normalize to 0-1 range, then scale to size range
    cloud_sizes = 20 + (cloud_diff_abs / cloud_diff_max) * 180 if cloud_diff_max > 0 else np.full_like(cloud_diff_abs, 50)
    shadow_sizes = 20 + (shadow_diff_abs / shadow_diff_max) * 180 if shadow_diff_max > 0 else np.full_like(shadow_diff_abs, 50)
    
    # Use diverging colormap (RdBu_r: red for positive, blue for negative)
    cmap = cm.get_cmap('RdBu_r')
    
    # Climate zone colors for histograms
    climate_colors = {'Arid': '#d62728', 'Cold': '#2ca02c', 'Temperate': '#1f77b4'}
    
    # Plot 1: Mean Cloud Difference Map
    # Overlay Köppen major-zone polygons (3 classes) using shapefile (fast)
    if koppen_zones_gdf is not None:
        try:
            # Darker = more prominent background; keep alpha low
            zone_style = {
                'Temperate': '0.8',
                'Arid': '0.6',
                'Cold': '0.4',
            }
            # Plot order so darker classes don't hide others unexpectedly
            for zone_name in ['Temperate', 'Arid', 'Cold']:
                if 'zone_name' in koppen_zones_gdf.columns:
                    subset = koppen_zones_gdf[koppen_zones_gdf['zone_name'] == zone_name]
                elif 'zone_code' in koppen_zones_gdf.columns:
                    code = {'Arid': 1, 'Cold': 2, 'Temperate': 3}[zone_name]
                    subset = koppen_zones_gdf[koppen_zones_gdf['zone_code'] == code]
                else:
                    subset = koppen_zones_gdf
                if len(subset) > 0:
                    subset.plot(
                        ax=ax_map_cloud,
                        color=zone_style.get(zone_name, '0.7'),
                        alpha=0.25,
                        edgecolor='none',
                        zorder=0,
                    )
            print("✓ Köppen zone shapefile overlay completed")
        except Exception as e:
            print(f"Warning: Could not overlay Köppen zone shapefile: {e}")
    
    world.boundary.plot(ax=ax_map_cloud, linewidth=0.5, color='0.5', alpha=0.6, zorder=1)

    # Normalize color scale symmetrically around zero
    norm1 = TwoSlopeNorm(vmin=-cloud_diff_max, vcenter=0, vmax=cloud_diff_max)
    scatter1 = ax_map_cloud.scatter(lons, lats, c=cloud_diffs, s=cloud_sizes, cmap=cmap, 
                           norm=norm1, alpha=0.8, edgecolors='black', linewidths=0.5, zorder=2)
    
    # Add colorbar on the right side (60% of current size, closer to plot)
    cbar1 = plt.colorbar(scatter1, ax=ax_map_cloud, orientation='vertical', pad=0.01, aspect=20, shrink=0.6)
    cbar1.set_label('Average Cloud Difference\n(Fmask5 - Fmask4) %', fontsize=16, fontweight='bold')
    cbar1.ax.tick_params(labelsize=14)
    
    ax_map_cloud.set_xlabel('', fontsize=14)  # Remove label but keep tick numbers
    ax_map_cloud.set_ylabel('', fontsize=14)  # Remove label but keep tick numbers
    ax_map_cloud.set_title('Spatial Variation of Average Cloud Difference', fontsize=16, fontweight='bold')
    # Add label a) in upper left corner (same size as title)
    ax_map_cloud.text(0.02, 0.98, 'a)', transform=ax_map_cloud.transAxes, 
                     fontsize=24, fontweight='bold', va='top', ha='left')
    ax_map_cloud.grid(False)
    ax_map_cloud.tick_params(labelsize=14)  # Increased tick label size
    
    # Add inset axes in lower left corner for violin plots by climate zone (70% of original size)
    ax_inset_cloud = inset_axes(ax_map_cloud, width="35%", height="30%", loc='lower left', 
                                borderpad=3)
    
    # Prepare data for violin plot (cloud differences by climate zone - PER-MRGS AGGREGATED)
    violin_data_cloud = []
    violin_labels_cloud = []
    for zone in ['Arid', 'Cold', 'Temperate']:
        zone_mask = climate_zones_mrgs == zone
        if np.sum(zone_mask) > 0:
            violin_data_cloud.append(cloud_diffs_mrgs[zone_mask])  # Use per-MRGS aggregated data
            violin_labels_cloud.append(zone)
    
    # Create horizontal violin plot using matplotlib for half-violin styling (similar to PNAS paper)
    if violin_data_cloud:
        # Use matplotlib violinplot in horizontal orientation (vert=False)
        positions = range(len(violin_data_cloud))
        parts = ax_inset_cloud.violinplot(violin_data_cloud, positions=positions,
                                         widths=0.7, showmeans=False, showmedians=False, 
                                         showextrema=False, vert=False)
        
        # Customize violin plots to show only bottom half (since rotated horizontally)
        for i, (pc, zone, pos) in enumerate(zip(parts['bodies'], violin_labels_cloud, positions)):
            # Get vertices from the violin path
            vertices = pc.get_paths()[0].vertices
            # For horizontal violins, clip to show only bottom half (y <= position center)
            center_y = pos
            vertices[:, 1] = np.clip(vertices[:, 1], -np.inf, center_y)
            # Update the path
            pc.get_paths()[0].vertices = vertices
            # Set color
            pc.set_facecolor(climate_colors[zone])
            pc.set_alpha(0.7)
            pc.set_edgecolor('black')
            pc.set_linewidth(1.5)
        
        # Calculate and plot half median lines manually (only bottom half)
        from scipy import stats as scipy_stats
        
        # Set xlim first so we can calculate the middle position
        ax_inset_cloud.set_xlim(-20, 20)  # Narrow x-axis range
        
        for i, (zone, data, pos) in enumerate(zip(violin_labels_cloud, violin_data_cloud, positions)):
            # Use pre-calculated statistics from climate_zone_stats_results
            # Get climate_zone_stats_results from parameter or globals
            stats_results = climate_zone_stats_results
            if stats_results is None:
                stats_results = globals().get('climate_zone_stats_results', None)
            
            # Extract median, N, and p-value from pre-calculated results
            if (stats_results is not None and 
                'medians' in stats_results and
                'median_significance_tests' in stats_results['medians']):
                
                medians_by_zone = stats_results['medians']
                significance_tests = stats_results['medians']['median_significance_tests']
                
                # Get median and N from cloud_medians
                if (zone in medians_by_zone.get('cloud_medians', {}) and
                    zone in medians_by_zone.get('sample_sizes', {})):
                    median_val = medians_by_zone['cloud_medians'][zone]
                    n_size = medians_by_zone['sample_sizes'][zone].get('cloud', len(data))
                else:
                    # Fallback: calculate from data
                    median_val = np.median(data)
                    n_size = len(data)
                
                # Get p-value from median_significance_tests
                if (zone in significance_tests and
                    'cloud' in significance_tests[zone]):
                    p_value = significance_tests[zone]['cloud'].get('p_value')
                    if p_value is None:
                        p_value = 1.0  # Default if test couldn't be performed
                else:
                    p_value = 1.0  # Default if not found
                
                # Format p-value string (can't use conditional in f-string format specifier)
                p_str = f"{p_value:.4f}" if p_value != 1.0 else "N/A"
                print(f'[INFO] Using pre-calculated stats from climate_zone_stats_results for {zone} cloud: median={median_val:.2f}%, N={n_size}, p={p_str}')
            else:
                # Fallback: calculate from data (should not happen if climate_zone_stats_results is provided)
                median_val = np.median(data)
                n_size = len(data)
                try:
                    w_stat, p_value = scipy_stats.wilcoxon(data, alternative='two-sided')
                except Exception:
                    p_value = 1.0
                print(f'[WARNING] Using calculated stats (climate_zone_stats_results not available) for {zone} cloud: median={median_val:.2f}%, N={n_size}, p={p_value:.4f}')
            
            # Draw solid vertical median line at the median value
            # For horizontal violin plot, median should be a vertical line at x=median_val
            # Span vertically from bottom of violin (pos - width/2) to center (pos)
            violin_width = 0.7  # Match the width used in violinplot
            y_bottom = pos - violin_width / 2
            y_top = pos
            ax_inset_cloud.plot([median_val, median_val], [y_bottom, y_top], 
                               color='black', linewidth=2.5, linestyle='-', zorder=10)
            
            # Annotate with zone name, median value, sample size, and significance symbol
            # Position label much further to the right (around 102% of the way across the plot)
            x_min, x_max = ax_inset_cloud.get_xlim()
            x_pos = x_min + 1.02 * (x_max - x_min)  # 102% from left edge
            # Convert p-value to significance symbols
            if p_value < 0.001:
                sig_symbol = '***'
            elif p_value < 0.01:
                sig_symbol = '**'
            elif p_value < 0.05:
                sig_symbol = '*'
            else:
                sig_symbol = 'ns'
            # Keep text in one line with significance symbol
            label_text = f'{zone} ({median_val:.2f}, N={n_size}, {sig_symbol})'
            ax_inset_cloud.text(x_pos, pos, label_text, 
                               va='center', ha='left', fontsize=18, fontweight='bold')
        
        ax_inset_cloud.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5, zorder=0)
        ax_inset_cloud.set_xlabel('', fontsize=14)
        ax_inset_cloud.set_ylabel('', fontsize=9)
        ax_inset_cloud.set_yticks(positions)
        ax_inset_cloud.set_yticklabels([''] * len(positions))  # Remove y-axis labels
        ax_inset_cloud.tick_params(labelsize=12, axis='x')
        ax_inset_cloud.tick_params(labelsize=12, axis='y', length=0)  # Hide y-axis ticks
        ax_inset_cloud.grid(False)
        ax_inset_cloud.set_facecolor('none')  # Transparent background
        ax_inset_cloud.patch.set_alpha(0)  # Fully transparent
        # Remove bounding box (all spines except bottom)
        ax_inset_cloud.spines['top'].set_visible(False)
        ax_inset_cloud.spines['right'].set_visible(False)
        ax_inset_cloud.spines['left'].set_visible(False)
        ax_inset_cloud.spines['bottom'].set_linewidth(1.2)
    
    # Plot 3: Mean Cloud Shadow Difference Map
    # Overlay Köppen major-zone polygons (3 classes) using shapefile (fast)
    if koppen_zones_gdf is not None:
        try:
            zone_style = {
                'Temperate': '0.8',
                'Arid': '0.6',
                'Cold': '0.4',
            }
            for zone_name in ['Temperate', 'Arid', 'Cold']:
                if 'zone_name' in koppen_zones_gdf.columns:
                    subset = koppen_zones_gdf[koppen_zones_gdf['zone_name'] == zone_name]
                elif 'zone_code' in koppen_zones_gdf.columns:
                    code = {'Arid': 1, 'Cold': 2, 'Temperate': 3}[zone_name]
                    subset = koppen_zones_gdf[koppen_zones_gdf['zone_code'] == code]
                else:
                    subset = koppen_zones_gdf
                if len(subset) > 0:
                    subset.plot(
                        ax=ax_map_shadow,
                        color=zone_style.get(zone_name, '0.7'),
                        alpha=0.25,
                        edgecolor='none',
                        zorder=0,
                    )
            print("✓ Köppen zone shapefile overlay completed")
        except Exception as e:
            print(f"Warning: Could not overlay Köppen zone shapefile: {e}")
    
    world.boundary.plot(ax=ax_map_shadow, linewidth=0.5, color='0.5', alpha=0.6, zorder=1)
    
    # Normalize color scale symmetrically around zero
    norm2 = TwoSlopeNorm(vmin=-shadow_diff_max, vcenter=0, vmax=shadow_diff_max)
    scatter2 = ax_map_shadow.scatter(lons, lats, c=shadow_diffs, s=shadow_sizes, cmap=cmap, 
                           norm=norm2, alpha=0.8, edgecolors='black', linewidths=0.5, zorder=2)
    
    # Add colorbar on the right side (60% of current size, closer to plot)
    cbar2 = plt.colorbar(scatter2, ax=ax_map_shadow, orientation='vertical', pad=0.01, aspect=20, shrink=0.6)
    cbar2.set_label('Average Cloud Shadow Difference\n(Fmask5 - Fmask4) %', fontsize=16, fontweight='bold')
    cbar2.ax.tick_params(labelsize=14)
    
    ax_map_shadow.set_xlabel('', fontsize=14)  # Remove label but keep tick numbers
    ax_map_shadow.set_ylabel('', fontsize=14)  # Remove label but keep tick numbers
    ax_map_shadow.set_title('Spatial Variation of Average Cloud Shadow Difference', fontsize=16, fontweight='bold')
    # Add label b) in upper left corner (same size as title)
    ax_map_shadow.text(0.02, 0.98, 'b)', transform=ax_map_shadow.transAxes, 
                      fontsize=24, fontweight='bold', va='top', ha='left')
    ax_map_shadow.grid(False)
    ax_map_shadow.tick_params(labelsize=14)  # Increased tick label size
    
    # Add inset axes in lower left corner for violin plots by climate zone (70% of original size)
    ax_inset_shadow = inset_axes(ax_map_shadow, width="35%", height="30%", loc='lower left',
                                 borderpad=3)
    
    # Prepare data for violin plot (shadow differences by climate zone - PER-MRGS AGGREGATED)
    violin_data_shadow = []
    violin_labels_shadow = []
    for zone in ['Arid', 'Cold', 'Temperate']:
        zone_mask = climate_zones_mrgs == zone
        if np.sum(zone_mask) > 0:
            violin_data_shadow.append(shadow_diffs_mrgs[zone_mask])  # Use per-MRGS aggregated data
            violin_labels_shadow.append(zone)
    
    # Create horizontal violin plot using matplotlib for half-violin styling (similar to PNAS paper)
    if violin_data_shadow:
        # Use matplotlib violinplot in horizontal orientation (vert=False)
        positions = range(len(violin_data_shadow))
        parts = ax_inset_shadow.violinplot(violin_data_shadow, positions=positions,
                                          widths=0.7, showmeans=False, showmedians=False, 
                                          showextrema=False, vert=False)
        
        # Customize violin plots to show only bottom half (since rotated horizontally)
        for i, (pc, zone, pos) in enumerate(zip(parts['bodies'], violin_labels_shadow, positions)):
            # Get vertices from the violin path
            vertices = pc.get_paths()[0].vertices
            # For horizontal violins, clip to show only bottom half (y <= position center)
            center_y = pos
            vertices[:, 1] = np.clip(vertices[:, 1], -np.inf, center_y)
            # Update the path
            pc.get_paths()[0].vertices = vertices
            # Set color
            pc.set_facecolor(climate_colors[zone])
            pc.set_alpha(0.7)
            pc.set_edgecolor('black')
            pc.set_linewidth(1.5)
        
        # Calculate and plot half median lines manually (only bottom half)
        from scipy import stats as scipy_stats
        
        # Set xlim first so we can calculate the middle position
        ax_inset_shadow.set_xlim(-4, 4)  # Narrow x-axis range
        
        for i, (zone, data, pos) in enumerate(zip(violin_labels_shadow, violin_data_shadow, positions)):
            print(i,zone,data,pos)
            # Use pre-calculated statistics from climate_zone_stats_results
            # Get climate_zone_stats_results from parameter or globals
            stats_results = climate_zone_stats_results
            if stats_results is None:
                stats_results = globals().get('climate_zone_stats_results', None)
            
            # Extract median, N, and p-value from pre-calculated results
            if (stats_results is not None and 
                'medians' in stats_results and
                'median_significance_tests' in stats_results['medians']):
                
                medians_by_zone = stats_results['medians']
                significance_tests = stats_results['medians']['median_significance_tests']
                
                # Get median and N from shadow_medians
                if (zone in medians_by_zone.get('shadow_medians', {}) and
                    zone in medians_by_zone.get('sample_sizes', {})):
                    median_val = medians_by_zone['shadow_medians'][zone]
                    n_size = medians_by_zone['sample_sizes'][zone].get('shadow', len(data))
                else:
                    # Fallback: calculate from data
                    median_val = np.median(data)
                    n_size = len(data)
                
                # Get p-value from median_significance_tests
                if (zone in significance_tests and
                    'shadow' in significance_tests[zone]):
                    p_value = significance_tests[zone]['shadow'].get('p_value')
                    if p_value is None:
                        p_value = 1.0  # Default if test couldn't be performed
                else:
                    p_value = 1.0  # Default if not found
                
                # Format p-value string (can't use conditional in f-string format specifier)
                p_str = f"{p_value:.4f}" if p_value != 1.0 else "N/A"
                print(f'[INFO] Using pre-calculated stats from climate_zone_stats_results for {zone} shadow: median={median_val:.2f}%, N={n_size}, p={p_str}')
            else:
                # Fallback: calculate from data (should not happen if climate_zone_stats_results is provided)
                median_val = np.median(data)
                n_size = len(data)
                try:
                    w_stat, p_value = scipy_stats.wilcoxon(data, alternative='two-sided')
                except Exception:
                    p_value = 1.0
                print(f'[WARNING] Using calculated stats (climate_zone_stats_results not available) for {zone} shadow: median={median_val:.2f}%, N={n_size}, p={p_value:.4f}')
            
            print('Tranghe')
            # Draw solid vertical median line at the median value
            # For horizontal violin plot, median should be a vertical line at x=median_val
            # Span vertically from bottom of violin (pos - width/2) to center (pos)
            violin_width = 0.7  # Match the width used in violinplot
            y_bottom = pos - violin_width / 2
            y_top = pos
            ax_inset_shadow.plot([median_val, median_val], [y_bottom, y_top], 
                                color='black', linewidth=2.5, linestyle='-', zorder=10)
            
            # Annotate with zone name, median value, sample size, and significance symbol
            # Position label much further to the right (around 102% of the way across the plot)
            x_min, x_max = ax_inset_shadow.get_xlim()
            x_pos = x_min + 1.02 * (x_max - x_min)  # 102% from left edge
            # Convert p-value to significance symbols
            if p_value < 0.001:
                sig_symbol = '***'
            elif p_value < 0.01:
                sig_symbol = '**'
            elif p_value < 0.05:
                sig_symbol = '*'
            else:
                sig_symbol = 'ns'
            # Keep text in one line with significance symbol
            label_text = f'{zone} ({median_val:.2f}, N={n_size}, {sig_symbol})'
            ax_inset_shadow.text(x_pos, pos, label_text, 
                                va='center', ha='left', fontsize=18, fontweight='bold')
        
        ax_inset_shadow.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5, zorder=0)
        ax_inset_shadow.set_xlabel('', fontsize=14)
        ax_inset_shadow.set_ylabel('', fontsize=9)
        ax_inset_shadow.set_yticks(positions)
        ax_inset_shadow.set_yticklabels([''] * len(positions))  # Remove y-axis labels
        ax_inset_shadow.tick_params(labelsize=12, axis='x')
        ax_inset_shadow.tick_params(labelsize=12, axis='y', length=0)  # Hide y-axis ticks
        ax_inset_shadow.grid(False)
        ax_inset_shadow.set_facecolor('none')  # Transparent background
        ax_inset_shadow.patch.set_alpha(0)  # Fully transparent
        # Remove bounding box (all spines except bottom)
        ax_inset_shadow.spines['top'].set_visible(False)
        ax_inset_shadow.spines['right'].set_visible(False)
        ax_inset_shadow.spines['left'].set_visible(False)
        ax_inset_shadow.spines['bottom'].set_linewidth(1.2)
    
    # Title removed per user request
    # plt.suptitle('Spatial Distribution of Average Differences per MRGS Tile\n(Köppen-Geiger Climate Classification)', fontsize=16, fontweight='bold', y=0.995)
    # Adjust layout to ensure colorbars are included - leave space on right side
    plt.tight_layout(rect=[0, 0, 0.92, 0.97])
    
    # Save figure if path is provided
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
        # Use pad_inches to ensure colorbars are included in the saved figure
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', pad_inches=0.1)
        print(f'\n✓ Saved spatial difference maps with climate classification to {save_path}')
    
    # Show plot if requested
    if show_plot:
        plt.show()
    
    return fig, axes



In [ ]:
def plot_seasonal_timeseries_by_climate_zone(
    df_fmask_percentage,
    difference_type='cloud',
    figsize=(14, 8),
    save_path=None,
    dpi=300,
    show_plot=True
):
    """
    Option B: Time series plots showing trends over months/seasons, colored by climate zone.
    
    Creates time series plots showing how differences vary over time, with separate lines
    for each climate zone (Arid, Cold, Temperate).
    
    Parameters
    ----------
    df_fmask_percentage : pandas.DataFrame
        DataFrame with columns: Date, mrgs_id, Cloud_diff, CloudShadow_diff
    difference_type : str, optional
        'cloud' or 'cloud_shadow'. Default is 'cloud'.
    figsize : tuple, optional
        Figure size. Default is (14, 8).
    save_path : str, optional
        Path to save figure.
    dpi : int, optional
        Resolution. Default is 300.
    show_plot : bool, optional
        Whether to display plot. Default is True.
    
    Returns
    -------
    fig : matplotlib.figure.Figure
    axes : matplotlib.axes.Axes
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    from datetime import datetime
    import mgrs
    from scipy import stats as scipy_stats
    
    # Select difference column
    diff_col = 'Cloud_diff' if difference_type == 'cloud' else 'CloudShadow_diff'
    
    # Calculate mean differences per MRGS ID per time period
    df_mean = df_fmask_percentage.groupby(['mrgs_id', 'Date']).agg({
        diff_col: 'mean'
    }).reset_index()
    
    # Convert Date to datetime
    df_mean['datetime'] = pd.to_datetime(df_mean['Date'].astype(str), format='%Y%j', errors='coerce')
    df_mean['month'] = df_mean['datetime'].dt.month
    df_mean['season'] = df_mean['datetime'].dt.month.map({
        12: 'DJF', 1: 'DJF', 2: 'DJF',
        3: 'MAM', 4: 'MAM', 5: 'MAM',
        6: 'JJA', 7: 'JJA', 8: 'JJA',
        9: 'SON', 10: 'SON', 11: 'SON'
    })
    
    # Classify climate zones for each MRGS ID
    m = mgrs.MGRS()
    climate_zones_dict = {}
    
    for mrgs_id in df_mean['mrgs_id'].unique():
        mrgs_str = str(mrgs_id).strip()
        mgrs_str = mrgs_str[1:] if mrgs_str.startswith('T') else mrgs_str
        try:
            lat, lon = m.toLatLon(f"{mgrs_str} 50000 50000")
            climate_zone = classify_climate_zone(lat, lon)
            climate_zones_dict[mrgs_id] = climate_zone
        except:
            climate_zones_dict[mrgs_id] = None
    
    df_mean['climate_zone'] = df_mean['mrgs_id'].map(climate_zones_dict)
    df_mean = df_mean.dropna(subset=['climate_zone', 'datetime', diff_col])
    
    # Create figure
    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True)
    ax1 = axes[0]  # Monthly time series
    ax2 = axes[1]  # Seasonal box plots
    
    # Climate zone colors
    climate_colors = {'Arid': '#d62728', 'Cold': '#2ca02c', 'Temperate': '#1f77b4'}
    
    # Plot 1: Monthly time series by climate zone
    for zone in ['Arid', 'Cold', 'Temperate']:
        df_zone = df_mean[df_mean['climate_zone'] == zone].copy()
        if len(df_zone) > 0:
            # Group by month and calculate mean
            monthly_mean = df_zone.groupby('month')[diff_col].mean()
            monthly_std = df_zone.groupby('month')[diff_col].std()
            months = monthly_mean.index
            
            ax1.plot(months, monthly_mean.values, 'o-', label=zone, 
                    color=climate_colors[zone], linewidth=2, markersize=6)
            ax1.fill_between(months, 
                           monthly_mean.values - monthly_std.values,
                           monthly_mean.values + monthly_std.values,
                           alpha=0.2, color=climate_colors[zone])
    
    ax1.set_xlabel('Month', fontsize=14)
    ax1.set_ylabel(f'Mean {difference_type.replace("_", " ").title()} Difference (%)', fontsize=14)
    ax1.set_title(f'Monthly Variation of {difference_type.replace("_", " ").title()} Differences by Climate Zone', 
                 fontsize=16, fontweight='bold')
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(range(1, 13))
    ax1.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    
    # Plot 2: Seasonal box plots by climate zone
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    positions = np.arange(len(season_order))
    width = 0.25
    
    for i, zone in enumerate(['Arid', 'Cold', 'Temperate']):
        df_zone = df_mean[df_mean['climate_zone'] == zone].copy()
        if len(df_zone) > 0:
            seasonal_data = [df_zone[df_zone['season'] == s][diff_col].values 
                           for s in season_order]
            bp = ax2.boxplot(seasonal_data, positions=positions + i*width, 
                            widths=width, patch_artist=True,
                            labels=season_order if i == 0 else ['']*4)
            for patch in bp['boxes']:
                patch.set_facecolor(climate_colors[zone])
                patch.set_alpha(0.7)
            for element in ['whiskers', 'fliers', 'means', 'medians', 'caps']:
                plt.setp(bp[element], color=climate_colors[zone])
    
    ax2.set_xlabel('Season', fontsize=14)
    ax2.set_ylabel(f'{difference_type.replace("_", " ").title()} Difference (%)', fontsize=14)
    ax2.set_title(f'Seasonal Distribution of {difference_type.replace("_", " ").title()} Differences by Climate Zone',
                 fontsize=16, fontweight='bold')
    ax2.set_xticks(positions + width)
    ax2.set_xticklabels(season_order)
    ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Create custom legend for box plots
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=climate_colors[zone], alpha=0.7, label=zone) 
                     for zone in ['Arid', 'Cold', 'Temperate']]
    ax2.legend(handles=legend_elements, fontsize=12)
    
    plt.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f'Saved to: {save_path}')
    
    if show_plot:
        plt.show()
    
    return fig, axes



In [ ]:
def plot_seasonal_distributions_by_climate_zone(
    df_fmask_percentage,
    difference_type='cloud',
    figsize=(16, 10),
    save_path=None,
    dpi=300,
    show_plot=True
):
    """
    Option C: Box plots/violin plots showing seasonal distributions stratified by climate zone.
    
    Creates comprehensive visualization showing seasonal distributions within each climate zone,
    with statistical tests comparing seasons within each zone.
    
    Parameters
    ----------
    df_fmask_percentage : pandas.DataFrame
        DataFrame with columns: Date, mrgs_id, Cloud_diff, CloudShadow_diff
    difference_type : str, optional
        'cloud' or 'cloud_shadow'. Default is 'cloud'.
    figsize : tuple, optional
        Figure size. Default is (16, 10).
    save_path : str, optional
        Path to save figure.
    dpi : int, optional
        Resolution. Default is 300.
    show_plot : bool, optional
        Whether to display plot. Default is True.
    
    Returns
    -------
    fig : matplotlib.figure.Figure
    axes : matplotlib.axes.Axes
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import mgrs
    from scipy import stats as scipy_stats
    
    # Select difference column
    diff_col = 'Cloud_diff' if difference_type == 'cloud' else 'CloudShadow_diff'
    
    # Prepare data
    df_work = df_fmask_percentage[['Date', 'mrgs_id', diff_col]].copy()
    
    # Convert Date to season
    df_work['datetime'] = pd.to_datetime(df_work['Date'].astype(str), format='%Y%j', errors='coerce')
    df_work['month'] = df_work['datetime'].dt.month
    df_work['season'] = df_work['month'].map({
        12: 'DJF', 1: 'DJF', 2: 'DJF',
        3: 'MAM', 4: 'MAM', 5: 'MAM',
        6: 'JJA', 7: 'JJA', 8: 'JJA',
        9: 'SON', 10: 'SON', 11: 'SON'
    })
    
    # Classify climate zones
    m = mgrs.MGRS()
    climate_zones_dict = {}
    
    for mrgs_id in df_work['mrgs_id'].unique():
        mrgs_str = str(mrgs_id).strip()
        mgrs_str = mrgs_str[1:] if mrgs_str.startswith('T') else mrgs_str
        try:
            lat, lon = m.toLatLon(f"{mgrs_str} 50000 50000")
            climate_zone = classify_climate_zone(lat, lon)
            climate_zones_dict[mrgs_id] = climate_zone
        except:
            climate_zones_dict[mrgs_id] = None
    
    df_work['climate_zone'] = df_work['mrgs_id'].map(climate_zones_dict)
    df_work = df_work.dropna(subset=['climate_zone', 'season', diff_col])
    
    # Create figure with subplots for each climate zone
    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    climate_colors = {'Arid': '#d62728', 'Cold': '#2ca02c', 'Temperate': '#1f77b4'}
    
    for idx, zone in enumerate(['Arid', 'Cold', 'Temperate']):
        ax = axes[idx]
        df_zone = df_work[df_work['climate_zone'] == zone].copy()
        
        if len(df_zone) > 0:
            # Create violin plot
            data_for_plot = []
            for season in season_order:
                season_data = df_zone[df_zone['season'] == season][diff_col].values
                if len(season_data) > 0:
                    data_for_plot.append(season_data)
                else:
                    data_for_plot.append([])
            
            parts = ax.violinplot(data_for_plot, positions=range(len(season_order)),
                                 widths=0.7, showmeans=False, showmedians=True,
                                 showextrema=True)
            
            # Customize violins
            for pc in parts['bodies']:
                pc.set_facecolor(climate_colors[zone])
                pc.set_alpha(0.7)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.5)
            
            # Calculate and annotate statistics
            stats_text = []
            for i, season in enumerate(season_order):
                season_data = df_zone[df_zone['season'] == season][diff_col].values
                if len(season_data) > 0:
                    median_val = np.median(season_data)
                    n_size = len(season_data)
                    stats_text.append(f'{season}: median={median_val:.2f}%, N={n_size}')
            
            # Perform Kruskal-Wallis test for seasonal differences within this zone
            seasonal_groups = [df_zone[df_zone['season'] == s][diff_col].values 
                             for s in season_order if len(df_zone[df_zone['season'] == s]) > 0]
            if len(seasonal_groups) > 2:
                try:
                    h_stat, p_kw = scipy_stats.kruskal(*seasonal_groups)
                    if p_kw < 0.001:
                        p_str = '<0.001'
                    else:
                        p_str = f'{p_kw:.3f}'
                    stats_text.append(f'Kruskal-Wallis: p={p_str}')
                except:
                    pass
            
            ax.text(0.02, 0.98, '\n'.join(stats_text), transform=ax.transAxes,
                   fontsize=9, va='top', ha='left',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_xticks(range(len(season_order)))
        ax.set_xticklabels(season_order)
        ax.set_xlabel('Season', fontsize=12)
        ax.set_title(f'{zone} Climate Zone', fontsize=14, fontweight='bold')
        ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax.grid(True, alpha=0.3, axis='y')
    
    axes[0].set_ylabel(f'{difference_type.replace("_", " ").title()} Difference (%)', fontsize=14)
    fig.suptitle(f'Seasonal Distributions of {difference_type.replace("_", " ").title()} Differences by Climate Zone',
                fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f'Saved to: {save_path}')
    
    if show_plot:
        plt.show()
    
    return fig, axes



In [ ]:
def plot_seasonal_maps_with_climate_zones(
    df_fmask_percentage,
    difference_type='cloud',
    figsize=(20, 16),
    save_path=None,
    dpi=300,
    show_plot=True,
    seasonal_stats_results=None
):
    """
    Create seasonal maps with climate zone overlays and violin plot insets.
    
    Parameters
    ----------
    df_fmask_percentage : pandas.DataFrame
        DataFrame with columns: Date, mrgs_id, Cloud_diff, CloudShadow_diff
    difference_type : str
        'cloud' or 'cloud_shadow'
    figsize : tuple
        Figure size (width, height)
    save_path : str, optional
        Path to save figure
    dpi : int
        Resolution for saved figure
    show_plot : bool
        Whether to display the plot
    seasonal_stats_results : dict, optional
        Pre-calculated seasonal statistics
        
    Returns
    -------
    fig, axes : matplotlib figure and axes
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import geopandas as gpd
    import mgrs
    import matplotlib.cm as cm
    from matplotlib.colors import TwoSlopeNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    from scipy import stats as scipy_stats
    
    # Select difference column
    diff_col = 'Cloud_diff' if difference_type == 'cloud' else 'CloudShadow_diff'
    
    # Prepare data
    df_work = df_fmask_percentage[['Date', 'mrgs_id', diff_col]].copy()
    
    # Convert Date to season
    df_work['datetime'] = pd.to_datetime(df_work['Date'].astype(str), format='%Y%j', errors='coerce')
    df_work['month'] = df_work['datetime'].dt.month
    df_work['season'] = df_work['month'].map({
        12: 'DJF', 1: 'DJF', 2: 'DJF',
        3: 'MAM', 4: 'MAM', 5: 'MAM',
        6: 'JJA', 7: 'JJA', 8: 'JJA',
        9: 'SON', 10: 'SON', 11: 'SON'
    })
    df_work = df_work.dropna(subset=['season', diff_col])
    
    # Calculate mean differences per MRGS ID per season
    df_seasonal_mean = df_work.groupby(['mrgs_id', 'season']).agg({
        diff_col: 'mean'
    }).reset_index()
    
    # Load global Köppen raster if needed
    global koppen_da
    if koppen_da is None:
        try:
            load_koppen_geiger_raster(resolution='0.5deg')
        except:
            pass
    
    # Load world boundaries
    from pathlib import Path
    local_candidates = [
        Path('ne_110m_admin_0_countries.zip'),
        Path('hls_validation/ne_110m_admin_0_countries.zip'),
        Path('../ne_110m_admin_0_countries.zip'),
    ]
    try:
        local_candidates.append(Path(__file__).resolve().parent.parent / 'ne_110m_admin_0_countries.zip')
    except NameError:
        pass
    local_path = next((p for p in local_candidates if p.exists()), None)
    if local_path is None:
        raise FileNotFoundError(f"Natural Earth file not found in: {', '.join(str(p) for p in local_candidates)}")
    world = gpd.read_file(str(local_path))

    # Load Köppen zone polygons
    koppen_zones_gdf = None
    try:
        koppen_candidates = [
            Path('hls_validation/koppen_geiger_major_zones/koppen_geiger_major_zones.shp'),
            Path('hls_validation/koppen_geiger_major_zones.shp'),
            Path('koppen_geiger_major_zones.shp'),
            Path('koppen_geiger_major_zones/koppen_geiger_major_zones.shp'),
        ]
        koppen_path = next((p for p in koppen_candidates if p.exists()), None)
        if koppen_path:
            koppen_zones_gdf = gpd.read_file(str(koppen_path))
            if koppen_zones_gdf.crs is None:
                koppen_zones_gdf = koppen_zones_gdf.set_crs('EPSG:4326')
            elif str(koppen_zones_gdf.crs) != 'EPSG:4326':
                koppen_zones_gdf = koppen_zones_gdf.to_crs('EPSG:4326')
    except Exception as e:
        print(f"Warning: Could not load Köppen shapefile: {e}")
    
    # Create figure with adjusted space for right colorbar
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(4, 1, hspace=0.3, right=0.88)  # Leave space for colorbar
    axes = [fig.add_subplot(gs[i]) for i in range(4)]
    
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    m = mgrs.MGRS()
    climate_colors = {'Arid': '#d62728', 'Cold': '#2ca02c', 'Temperate': '#1f77b4'}
    
    # Find global color scale limits across all seasons
    all_diffs = []
    for season in season_order:
        df_season = df_seasonal_mean[df_seasonal_mean['season'] == season]
        if len(df_season) > 0:
            all_diffs.extend(df_season[diff_col].values)
    
    if len(all_diffs) > 0:
        diff_max = np.nanmax(np.abs(all_diffs))
    else:
        diff_max = 1.0
    
    norm = TwoSlopeNorm(vmin=-diff_max, vcenter=0, vmax=diff_max)
    cmap = cm.get_cmap('RdBu_r')
    
    # Process each season
    for season_idx, season in enumerate(season_order):
        df_season = df_seasonal_mean[df_seasonal_mean['season'] == season].copy()
        
        # Convert MRGS IDs to lat/lon and classify climate zones
        lats, lons, diffs, climate_zones_list = [], [], [], []
        
        for idx, row in df_season.iterrows():
            mrgs_id = str(row['mrgs_id']).strip()
            mgrs_str = mrgs_id[1:] if mrgs_id.startswith('T') else mrgs_id
            try:
                lat, lon = m.toLatLon(f"{mgrs_str} 50000 50000")
                climate_zone = classify_climate_zone(lat, lon)
                lats.append(lat)
                lons.append(lon)
                diffs.append(row[diff_col])
                climate_zones_list.append(climate_zone)
            except:
                continue
        
        if len(lats) == 0:
            continue
        
        lats = np.array(lats)
        lons = np.array(lons)
        diffs = np.array(diffs)
        climate_zones = np.array(climate_zones_list)
        
        ax = axes[season_idx]
        
        # Overlay Köppen zones
        if koppen_zones_gdf is not None:
            try:
                zone_style = {'Temperate': '0.8', 'Arid': '0.6', 'Cold': '0.4'}
                for zone_name in ['Temperate', 'Arid', 'Cold']:
                    if 'zone_name' in koppen_zones_gdf.columns:
                        subset = koppen_zones_gdf[koppen_zones_gdf['zone_name'] == zone_name]
                    elif 'zone_code' in koppen_zones_gdf.columns:
                        code = {'Arid': 1, 'Cold': 2, 'Temperate': 3}[zone_name]
                        subset = koppen_zones_gdf[koppen_zones_gdf['zone_code'] == code]
                    else:
                        subset = koppen_zones_gdf
                    if len(subset) > 0:
                        subset.plot(ax=ax, color=zone_style.get(zone_name, '0.7'),
                                  alpha=0.25, edgecolor='none', zorder=0)
            except Exception as e:
                print(f"Warning: Could not overlay Köppen zones: {e}")
        
        world.boundary.plot(ax=ax, linewidth=0.5, color='0.5', alpha=0.6, zorder=1)
        
        # Plot scatter
        sizes = 20 + (np.abs(diffs) / diff_max) * 180 if diff_max > 0 else np.full_like(diffs, 50)
        scatter = ax.scatter(lons, lats, c=diffs, s=sizes, cmap=cmap, norm=norm,
                           alpha=0.8, edgecolors='black', linewidths=0.5, zorder=2)
        
        ax.set_title(f'{season} ({difference_type.replace("_", " ").title()} Differences)', 
                    fontsize=14, fontweight='bold', pad=10)
        ax.grid(False)
        
        # Add inset violin plot showing distribution by climate zone
        ax_inset = inset_axes(ax, width="35%", height="30%", loc='lower left', borderpad=3)
        
        violin_data, violin_labels = [], []
        for zone in ['Arid', 'Cold', 'Temperate']:
            zone_mask = climate_zones == zone
            if np.sum(zone_mask) > 0:
                violin_data.append(diffs[zone_mask])
                violin_labels.append(zone)
        
        if violin_data:
            positions = range(len(violin_data))
            parts = ax_inset.violinplot(violin_data, positions=positions, widths=0.7,
                                       showmeans=False, showmedians=False, showextrema=False, vert=False)
            
            for i, (pc, zone, pos) in enumerate(zip(parts['bodies'], violin_labels, positions)):
                vertices = pc.get_paths()[0].vertices
                center_y = pos
                vertices[:, 1] = np.clip(vertices[:, 1], -np.inf, center_y)
                pc.set_facecolor(climate_colors.get(zone, '0.5'))
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                pc.set_alpha(0.8)
            
            # Add median lines and labels
            for i, (zone_data, zone, pos) in enumerate(zip(violin_data, violin_labels, positions)):
                median_val = np.median(zone_data)
                n_size = len(zone_data)
                
                # Median line
                violin_width = 0.7
                y_bottom = pos - violin_width / 2
                y_top = pos
                ax_inset.plot([median_val, median_val], [y_bottom, y_top],
                            color='black', linewidth=2.5, linestyle='-', zorder=10)
                
                # Statistical significance test (one-sample t-test against 0)
                _, p_value = scipy_stats.ttest_1samp(zone_data, 0)
                
                # Convert p-value to significance symbols
                if p_value < 0.001:
                    sig_symbol = '***'
                elif p_value < 0.01:
                    sig_symbol = '**'
                elif p_value < 0.05:
                    sig_symbol = '*'
                else:
                    sig_symbol = 'ns'
                
                # Label with significance
                x_min, x_max = ax_inset.get_xlim()
                x_pos = x_min + 1.02 * (x_max - x_min)
                label_text = f'{zone} ({median_val:.2f}, N={n_size}, {sig_symbol})'
                ax_inset.text(x_pos, pos, label_text, va='center', ha='left',
                            fontsize=11, fontweight='bold')
            
            ax_inset.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5, zorder=0)
            ax_inset.set_xlabel('', fontsize=10)
            ax_inset.set_yticks(positions)
            ax_inset.set_yticklabels([''] * len(positions))
            ax_inset.tick_params(labelsize=10, axis='x')
            ax_inset.tick_params(labelsize=10, axis='y', length=0)
            ax_inset.grid(False)
            ax_inset.set_facecolor('none')
            ax_inset.patch.set_alpha(0)
            ax_inset.spines['top'].set_visible(False)
            ax_inset.spines['right'].set_visible(False)
            ax_inset.spines['left'].set_visible(False)
            ax_inset.spines['bottom'].set_linewidth(1.2)
    
    # Add single colorbar on the right side
    cbar_ax = fig.add_axes([0.90, 0.30, 0.02, 0.28])  # [left, bottom, width, height]
    cbar = fig.colorbar(scatter, cax=cbar_ax, orientation='vertical')
    cbar.set_label(f'{difference_type.replace("_", " ").title()} Difference (%)\n(Fmask5 - Fmask4)',
                  fontsize=14, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)
    
    # Save and show
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f'Saved to: {save_path}')
    
    if show_plot:
        plt.show()
    
    return fig, axes


In [ ]:
def batch_fmask_comparison_plots(
    files_s3_fmask5,
    files_s3_fmask4,
    df_fmask_percentage,
    bucket_name,
    read_true_color_from_files,
    to_julian_date=None,
    output_dir='figures',
    display_mode='inline_export',  # 'inline_export' or 'interactive_no_export'
    skip_existing=True,
    mrgs_id_target=None,
    day_target=None,
    fig=None  # Optional: matplotlib figure object. If None, creates a new figure.
):
    """
    Generate batch comparison plots for Fmask 4.7 vs 5.0 across multiple MRGS IDs and days.
    
    Creates a 3x3 grid of plots for each day/MRGS ID combination showing:
    - RGB reference image
    - Fmask 4.7 classification
    - Fmask 5.0 classification
    - Cirrus band
    - Cloud shadow difference map
    - Cloud difference map
    - Feature percentage bar chart
    
    Parameters
    ----------
    files_s3_fmask5 : list
        List of Fmask 5.0 file paths in S3.
    files_s3_fmask4 : list
        List of Fmask 4.7 file paths in S3.
    df_fmask_percentage : pandas.DataFrame
        DataFrame containing Fmask percentage comparison data.
    bucket_name : str
        S3 bucket name.
    read_true_color_from_files : function
        Function to read RGB data from files.
    to_julian_date : function, optional
        Function to convert dates to Julian format. Default is None.
    output_dir : str, optional
        Directory to save output figures. Only used when display_mode='inline_export'.
        Default is 'figures'.
    display_mode : str, optional
        Display mode: 
        - 'inline_export': Display figures inline in notebook AND save to files (default)
        - 'interactive_no_export': Show interactive plots (popup windows) and DO NOT save files
        Default is 'inline_export'.
    skip_existing : bool, optional
        Skip processing if output file already exists. Only used when display_mode='inline_export'.
        Default is True.
    mrgs_id_target : str, optional
        Specific MRGS ID to process. If None, processes all MRGS IDs. Default is None.
    day_target : str, optional
        Specific day (in Julian format YYYYDDD) to process. If None, processes all days.
        Only used if mrgs_id_target is also specified. Default is None.
    fig : matplotlib.figure.Figure, optional
        Matplotlib figure object to use for plotting. If None, creates a new figure
        with size (8.5, 5.22) and 2x3 subplot layout. If provided, uses existing
        figure and expects it to have at least 6 axes. When processing multiple days,
        the same figure will be reused (plots will be overwritten). For single-day
        processing, provide both `mrgs_id_target` and `day_target` along with `fig`.
        Default is None.
    
    Returns
    -------
    dict
        Dictionary with processing statistics:
        - 'total_processed': Number of figures generated
        - 'total_skipped': Number of figures skipped (already exist)
        - 'total_errors': Number of errors encountered
        - 'processed_files': List of generated file paths (empty if display_mode='interactive_no_export')
        - 'skipped_files': List of skipped file paths
        - 'errors': List of error messages
        - 'fig': matplotlib.figure.Figure or None - The figure object used/created (only returned when processing a single day via mrgs_id_target and day_target)
    
    Note
    ----
    This function requires the following helper functions to be available:
    - extract_unique_mrgs_ids
    - filter_fmask_files
    - get_fmask_info
    - select_fmask_files
    - read_raster_or_rgb
    - plot_fmask_raster
    - plot_cirrus_band
    - plot_fmask_diff
    - plot_fmask_percentage_by_day
    - customize_axes_layout
    - log_stretching_reflectance_optimized
    
    These should be loaded via %run in the notebook before calling this function.
    """
    import os
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt

    # Performance optimizations
    matplotlib.use('Agg', force=True)  # Non-interactive backend for speed
    plt.ioff()  # Turn off interactive mode
    from datetime import datetime
    
    # Validate display_mode and determine behavior
    if display_mode not in ['inline_export', 'interactive_no_export']:
        raise ValueError(f"display_mode must be 'inline_export' or 'interactive_no_export', got '{display_mode}'")
    
    export_figures = (display_mode == 'inline_export')
    display_inline = (display_mode == 'inline_export')
    
    # Set matplotlib backend based on display_mode
    if display_mode == 'inline_export':
        # For inline display, keep the current backend (should be 'inline' set by %matplotlib inline)
        # Don't switch backends - let the notebook's %matplotlib inline handle it
        current_backend = matplotlib.get_backend()
        if 'inline' not in current_backend.lower() and 'agg' not in current_backend.lower():
            print(f"⚠ Warning: Backend is {current_backend}. For inline display, ensure %matplotlib inline is set.")
        else:
            print(f"✓ Using backend: {current_backend} (inline display + export)")
    else:  # interactive_no_export
        # For interactive display, switch to an interactive backend
        try:
            plt.switch_backend('qt5')
            print('✓ Using interactive backend: qt5 (interactive display, no export)')
        except Exception:
            try:
                plt.switch_backend('TkAgg')
                print('✓ Using interactive backend: TkAgg (interactive display, no export)')
            except Exception:
                print('⚠ Warning: Could not set interactive backend, using default')
                print('  Interactive plots may not work correctly')
    
    # Statistics tracking
    stats = {
        'total_processed': 0,
        'total_skipped': 0,
        'total_errors': 0,
        'processed_files': [],
        'skipped_files': [],
        'errors': []
    }
    
    # Create output directory (only if exporting)
    if export_figures:
        os.makedirs(output_dir, exist_ok=True)
        print(f"Output directory: {output_dir}")
    
    # Filter to specific MRGS ID and/or day if specified
    if mrgs_id_target is not None:
        # Process only the specified MRGS ID
        unique_mrgs_ids = [mrgs_id_target]
        print(f'Processing specific MRGS ID: {mrgs_id_target}')
    else:
        # Process all MRGS IDs
        unique_mrgs_ids = extract_unique_mrgs_ids(files_s3_fmask4)
        print(f'Found {len(unique_mrgs_ids)} unique MRGS IDs: {unique_mrgs_ids}')
    
    # Loop over each unique MRGS ID
    
    # Track which MRGS IDs are processed
    processed_mrgs_ids = []
    skipped_mrgs_ids = []
    error_mrgs_ids = []
    
    # Track the figure object (for single-day processing or when fig is provided)
    last_fig = None
    
    for mrgs_id_target in unique_mrgs_ids:
        print(f'\nProcessing MRGS ID: {mrgs_id_target}')
        
        try:
            # Filter files for this specific MRGS ID
            files_fmask5_id, files_fmask4_id, day_uniques_id, day_uniques_julian_id = filter_fmask_files(
                files_s3_fmask5, files_s3_fmask4, mrgs_id_target=mrgs_id_target, to_julian_date=to_julian_date
            )
            
            print(f'  Found {len(day_uniques_id)} days for MRGS ID {mrgs_id_target}')
            
            # Track MRGS IDs with no days
            if len(day_uniques_id) == 0:
                print(f'  ⚠ Warning: No days found for MRGS ID {mrgs_id_target} - skipping')
                stats['total_errors'] += 1
                stats['errors'].append(f'MRGS {mrgs_id_target}: No days found in files')
                skipped_mrgs_ids.append(mrgs_id_target)
                continue
            
            # Track MRGS IDs with no days
            if len(day_uniques_id) == 0:
                print(f'  ⚠ Warning: No days found for MRGS ID {mrgs_id_target} - skipping')
                stats['total_errors'] += 1
                stats['errors'].append(f'MRGS {mrgs_id_target}: No days found')
                continue
            
            # Filter to specific day if specified
            if day_target is not None:
                if day_target not in day_uniques_id:
                    print(f'  ✗ Error: Day {day_target} not found for MRGS ID {mrgs_id_target}')
                    print(f'  Available days ({len(day_uniques_id)} total):')
                    # Show first 20 days, or all if less than 20
                    days_to_show = day_uniques_id[:20] if len(day_uniques_id) > 20 else day_uniques_id
                    for d in days_to_show:
                        print(f'    - {d}')
                    if len(day_uniques_id) > 20:
                        print(f'    ... and {len(day_uniques_id) - 20} more days')
                    # Try to find similar days
                    similar_days = [d for d in day_uniques_id if str(day_target)[:4] in str(d) or str(day_target)[-3:] in str(d)]
                    if similar_days:
                        print(f'  Similar days found: {similar_days[:5]}')
                    stats['total_errors'] += 1
                    stats['errors'].append(f'MRGS {mrgs_id_target}, Day {day_target}: Day not found in available days')
                    continue
                # Find the index of day_target in the original day_uniques_id BEFORE filtering
                day_idx = day_uniques_id.index(day_target)
                # Get the corresponding julian_day BEFORE filtering day_uniques_id
                corresponding_julian = day_uniques_julian_id[day_idx] if day_idx < len(day_uniques_julian_id) else None
                # Filter both lists to maintain alignment
                day_uniques_id = [day_target]
                day_uniques_julian_id = [corresponding_julian] if corresponding_julian is not None else []
                print(f'  Processing specific day: {day_target}')
                if corresponding_julian:
                    print(f'  Corresponding julian_day: {corresponding_julian}')
            
            # Loop over each unique day for this MRGS ID
            for day in day_uniques_id:
                try:
                    # Check if output file already exists (only if exporting)
                    if skip_existing and export_figures:
                        sat_id_check = None
                        time_str_check = '000000'
                        
                        sample_files_check = files_fmask4_id + files_fmask5_id
                        if sample_files_check:
                            matching_files_check = [f for f in sample_files_check if day in f and mrgs_id_target in f]
                            if matching_files_check:
                                sample_file_check = matching_files_check[0]
                                parts_check = sample_file_check.split('.')
                                if len(parts_check) >= 2:
                                    sat_id_check = parts_check[1]
                                if len(parts_check) >= 4:
                                    date_part_check = parts_check[3]
                                    if 'T' in date_part_check and len(date_part_check) > 8:
                                        time_str_check = date_part_check.split('T')[1] if 'T' in date_part_check else '000000'
                        
                        if sat_id_check is None:
                            sat_id_check = 'S30'
                        
                        mrgs_id_formatted_check = 'T' + mrgs_id_target if not mrgs_id_target.startswith('T') else mrgs_id_target
                        
                        # We now export only PNG files (no TIFF)
                        expected_filename = f'{output_dir}/HLS.{sat_id_check}.{mrgs_id_formatted_check}.{day}T{time_str_check}.v2.0_evaluation.png'
                        
                        if os.path.exists(expected_filename):
                            existing_file = expected_filename
                            print(f'  Skipping {day} - output already exists: {existing_file}')
                            stats['total_skipped'] += 1
                            stats['skipped_files'].append(existing_file)
                            continue
                    
                    timings = {}
                    overall_start = time.time()
                    
                    # Create or use provided figure
                    if fig is None:
                        # Create new figure with default layout
                        fig, axes = plt.subplots(figsize=(12, 5.22), ncols=3, nrows=2)
                        axes = axes.flatten() if hasattr(axes, 'flatten') else [ax for row in axes for ax in row]
                    else:
                        # Use provided figure and get its axes
                        all_axes = list(fig.get_axes())  # Convert to list to ensure we can check length
                        
                        # Check if this is a newly created figure with subplots
                        # If we have exactly 6 axes, use them directly (likely a 2x3 subplot)
                        if len(all_axes) == 6:
                            # Likely a newly created 2x3 subplot - use axes directly
                            axes = all_axes
                        elif len(all_axes) == 0:
                            # No axes found - create them using the figure's subplot method
                            # This handles cases where figure was created but axes weren't attached
                            # Clear any existing content first
                            fig.clear()
                            try:
                                axes = fig.subplots(2, 3)
                                axes = axes.flatten() if hasattr(axes, 'flatten') else [ax for row in axes for ax in row]
                                print(f"  ⚠ Created 6 subplot axes for provided figure (figure had 0 axes)")
                            except Exception as e:
                                raise ValueError(f"Could not get or create axes from provided figure. "
                                               f"Figure has {len(all_axes)} axes. "
                                               f"Please create figure with: fig, axes = plt.subplots(figsize=(12, 5.22), ncols=3, nrows=2). "
                                               f"Error: {e}")
                        else:
                            # Filter to only plot axes (exclude colorbars)
                            # This handles cases where the figure has been used before and has colorbars
                            plot_axes = []
                            for ax in all_axes:
                                try:
                                    # Check if it's a subplot axis (has subplotspec attribute)
                                    if hasattr(ax, 'get_subplotspec') and ax.get_subplotspec() is not None:
                                        plot_axes.append(ax)
                                    else:
                                        # Fallback: check dimensions (but be more lenient for new axes)
                                        xlim = ax.get_xlim()
                                        ylim = ax.get_ylim()
                                        width = abs(xlim[1] - xlim[0])
                                        height = abs(ylim[1] - ylim[0])
                                        # Accept axes with reasonable dimensions OR newly created axes (width/height around 1.0)
                                        if width > 10 and height > 10:
                                            plot_axes.append(ax)
                                        elif 0.9 <= width <= 1.1 and 0.9 <= height <= 1.1:
                                            # Newly created axes often have limits around (0, 1)
                                            plot_axes.append(ax)
                                except:
                                    # If we can't determine, include it (better to have too many than too few)
                                    plot_axes.append(ax)
                            
                            axes = plot_axes
                            
                            # Ensure we have at least 6 axes for the 6 panels
                            if len(axes) < 6:
                                raise ValueError(f"Provided figure must have at least 6 axes, found {len(axes)}. "
                                               f"Total axes in figure: {len(all_axes)}. "
                                               f"Please create figure with: fig, axes = plt.subplots(figsize=(12, 5.22), ncols=3, nrows=2)")
                            # Use first 6 axes
                            axes = axes[:6]
                    
                    # Track the figure for return value
                    last_fig = fig
                    
                    # Determine satellite ID early for cirrus band labeling
                    sat_id = None
                    sample_files_early = files_fmask4_id + files_fmask5_id
                    if sample_files_early:
                        matching_files_early = [f for f in sample_files_early if day in f and mrgs_id_target in f]
                        if matching_files_early:
                            sample_file_early = matching_files_early[0]
                            parts_early = sample_file_early.split('.')
                            if len(parts_early) >= 2:
                                sat_id = parts_early[1]
                    if sat_id is None:
                        sat_id = 'S30'  # Default to Sentinel-2
                    
                    # Helper function to load raster data
                    def load_raster_data(i):
                        fmask_version = get_fmask_info(i=i)['fmask_version']
                        file_path = select_fmask_files(
                            fmask_version=fmask_version,
                            day=day,
                            files_fmask4_test=files_fmask4_id,
                            files_fmask5_test=files_fmask5_id,
                            day_uniques=day_uniques_id,
                            day_uniques_julian=day_uniques_julian_id
                        )
                        fmask4_ref = select_fmask_files(
                            fmask_version='fmask4.7',
                            day=day,
                            files_fmask4_test=files_fmask4_id,
                            files_fmask5_test=files_fmask5_id,
                            day_uniques=day_uniques_id,
                            day_uniques_julian=day_uniques_julian_id
                        )
                        
                        # Check if file_path is valid before loading
                        if file_path is None:
                            raise ValueError(f'No file found for {fmask_version} on day {day} for MRGS ID {mrgs_id_target}')
                        if fmask4_ref is None:
                            raise ValueError(f'No Fmask4 reference file found for day {day} for MRGS ID {mrgs_id_target}')
                        
                        raster = read_raster_or_rgb(file_path, bucket_name, read_true_color_from_files, fmask4_ref)
                        return i, fmask_version, raster
                    
                    # Load rasters sequentially
                    t0 = time.time()
                    print(f'  Loading 4 rasters sequentially...')
                    raster_list = []
                    results = []
                    for i in range(4):
                        result = load_raster_data(i)
                        results.append(result)
                        raster_list.append(result[2])
                    timings['raster_loading'] = time.time() - t0
                    
                    # Plot rasters sequentially (matplotlib is not thread-safe)
                    for i, fmask_version, raster in results:
                        iter_start = time.time()
                        fmask_info = get_fmask_info(i=i)
                        title = fmask_info['title']
                        fmask_version = fmask_info['fmask_version']
                        ax = axes[i]
                        
                        if i == 0:
                            t0 = time.time()
                            stretched_rgb = log_stretching_reflectance_optimized(raster)
                            timings['rgb_stretching'] = time.time() - t0
                            t0 = time.time()
                            ax.imshow(stretched_rgb)
                            ax.set_title("a) RGB stretched reference", fontsize=12)
                            timings['rgb_imshow'] = time.time() - t0
                        elif i == 1:
                            t0 = time.time()
                            cax_fmask = plot_fmask_raster(ax, raster, i=i, fig=fig, show_colorbar=False)
                            ax.set_title("b) Fmask 4.7", fontsize=12)
                            timings[f'plot_fmask_{i}'] = time.time() - t0
                        elif i == 2:
                            t0 = time.time()
                            cax_fmask = plot_fmask_raster(ax, raster, i=i, fig=fig, show_colorbar=False)
                            ax.set_title("c) Fmask 5.0.1", fontsize=12)
                            timings[f'plot_fmask_{i}'] = time.time() - t0
                        elif i == 3:
                            t0 = time.time()
                            # Determine cirrus band based on satellite type
                            # Landsat B09 (band 9), Sentinel-2 B10 (band 10)
                            cirrus_band_title = "Cirrus (B09)" if sat_id == 'L30' else "Cirrus (B10)"
                            plot_cirrus_band(ax, raster, title=f"d) {cirrus_band_title} TOA")
                            timings['plot_cirrus'] = time.time() - t0
                        else:
                            t0 = time.time()
                            cax_fmask = plot_fmask_raster(ax, raster, i=i, fig=fig, show_colorbar=False)
                            timings[f'plot_fmask_{i}'] = time.time() - t0
                        
                        timings[f'plotting_iteration_{i}'] = time.time() - iter_start
                    
                    # Plot difference maps
                    t0 = time.time()
                    cax_fmask_diff = plot_fmask_diff(raster_list, axes, fig)
                    timings['plot_fmask_diff'] = time.time() - t0
                    
                    overall_time = time.time() - overall_start
                    
                    # Print timing summary
                    print(f'\\n  Timing summary for {day}:')
                    print(f'  Total time: {overall_time:.2f}s')
                    if 'raster_loading' in timings:
                        print(f'  ✓ Raster loading: {timings["raster_loading"]:.2f}s')
                    print(f'  Breakdown:')
                    for key, value in sorted(timings.items(), key=lambda x: x[1], reverse=True):
                        percentage = (value / overall_time) * 100
                        print(f'    {key}: {value:.2f}s ({percentage:.1f}%)')
                    
                    # Update difference map titles to include labels (e and f)
                    if len(axes) > 4:
                        if axes[4].get_title():
                            axes[4].set_title("e) " + axes[4].get_title(), fontsize=12)
                        if len(axes) > 5 and axes[5].get_title():
                            axes[5].set_title("f) " + axes[5].get_title(), fontsize=12)
                    
                    # Generate filename
                    sat_id = None
                    time_str = '000000'
                    
                    sample_files = files_fmask4_id + files_fmask5_id
                    if sample_files:
                        matching_files = [f for f in sample_files if day in f and mrgs_id_target in f]
                        if matching_files:
                            sample_file = matching_files[0]
                            parts = sample_file.split('.')
                            if len(parts) >= 2:
                                sat_id = parts[1]
                            if len(parts) >= 4:
                                date_part = parts[3]
                                if 'T' in date_part and len(date_part) > 8:
                                    time_str = date_part.split('T')[1] if 'T' in date_part else '000000'
                    
                    if sat_id is None:
                        sat_id = 'S30'
                    
                    mrgs_id_formatted = 'T' + mrgs_id_target if not mrgs_id_target.startswith('T') else mrgs_id_target
                    
                    filename = f'{output_dir}/HLS.{sat_id}.{mrgs_id_formatted}.{day}T{time_str}.v2.0_evaluation.png'
                    
                    # Save figure only if export_figures is True
                    if export_figures:
                        fig.tight_layout()
                        fig.savefig(filename, dpi=300, format='png', bbox_inches='tight')
                        print(f'  Saved figure: {filename}')
                        stats['total_processed'] += 1
                        
                        # Track successful processing
                        if mrgs_id_target not in processed_mrgs_ids:
                            processed_mrgs_ids.append(mrgs_id_target)
                        stats['processed_files'].append(filename)
                    else:
                        print(f'  Displaying interactive plot (not saving to file)')
                        stats['total_processed'] += 1
                        filename = None
                    
                    # Handle plot display based on display_mode
                    if display_mode == 'inline_export':
                        # Mode 1: Inline display + Export figures
                        # Inline backend will display the figure automatically - no need to call plt.show()
                        # Keep the figure open for inline display
                        pass
                    else:  # interactive_no_export
                        # Mode 2: Interactive display + No export
                        try:
                            plt.show(block=False)
                            print(f'  ✓ Displayed interactive plot (not saved)')
                        except Exception as e:
                            print(f'  ✗ Error calling plt.show(): {e}')
                            import traceback
                            traceback.print_exc()
                        # Don't close figure for interactive mode - let user close it
                except Exception as e:
                    print(f'  ✗ Error processing {day}: {e}')
                    stats['total_errors'] += 1
                    stats['errors'].append(f'MRGS {mrgs_id_target}, Day {day}: {str(e)}')
                    import traceback
                    traceback.print_exc()
                    
        except Exception as e:
            print(f'✗ Error processing MRGS ID {mrgs_id_target}: {e}')
            stats['total_errors'] += 1
            stats['errors'].append(f'MRGS {mrgs_id_target}: {str(e)}')
            if mrgs_id_target not in error_mrgs_ids:
                error_mrgs_ids.append(mrgs_id_target)
            import traceback
            traceback.print_exc()
    
    # Print summary statistics
    print('\\n' + '='*60)
    print('BATCH PROCESSING SUMMARY')
    print('='*60)
    print(f'Total MRGS IDs processed: {len(unique_mrgs_ids)}')
    print(f'  ✓ Successfully processed: {len(processed_mrgs_ids)}')
    print(f'  ⊘ Skipped: {len(skipped_mrgs_ids)}')
    print(f'  ✗ Errors: {len(error_mrgs_ids)}')
    print(f'\\nTotal figures:')
    print(f'  ✓ Generated: {stats["total_processed"]}')
    print(f'  ⊘ Skipped (already exist): {stats["total_skipped"]}')
    print(f'  ✗ Errors: {stats["total_errors"]}')
    
    if stats['errors']:
        print(f'\\nError details (first 10):')
        for error in stats['errors'][:10]:
            print(f'  • {error}')
    
    # Return statistics and figure if processing single day
    if day_target is not None and mrgs_id_target is not None:
        stats['fig'] = last_fig
    
    return stats



In [ ]:
def diagnose_missing_granules(
    df_fmask_percentage,
    files_s3_fmask4,
    files_s3_fmask5,
    output_dir='figures'
):
    """
    Diagnose why some granules are not being exported.
    
    Compares expected granules from df_fmask_percentage with:
    1. Available files in files_s3_fmask4 and files_s3_fmask5
    2. Already exported figures
    3. Missing file combinations
    
    Parameters
    ----------
    df_fmask_percentage : pandas.DataFrame
        DataFrame with all expected granules (Date, mrgs_id columns).
    files_s3_fmask4 : list
        List of Fmask 4.7 file paths in S3.
    files_s3_fmask5 : list
        List of Fmask 5.0 file paths in S3.
    output_dir : str, optional
        Directory where figures are saved. Default is 'figures'.
    
    Returns
    -------
    dict
        Dictionary with diagnostic information:
        - 'total_expected': Total granules in df_fmask_percentage
        - 'missing_fmask4_files': Granules missing Fmask4 files
        - 'missing_fmask5_files': Granules missing Fmask5 files
        - 'missing_both_files': Granules missing both files
        - 'already_exported': Granules with existing figures
        - 'needs_processing': Granules that should be processed
    """
    import os
    import pandas as pd
    
    print('='*60)
    print('DIAGNOSTIC: Missing Granules Analysis')
    print('='*60)
    
    # Get all expected granules
    expected_granules = df_fmask_percentage[['Date', 'mrgs_id']].drop_duplicates().copy()
    total_expected = len(expected_granules)
    print(f'\nTotal expected granules: {total_expected}')
    
    # Check for existing exported figures
    exported_files = []
    if os.path.exists(output_dir):
        for file in os.listdir(output_dir):
            if file.endswith('.png') and '_evaluation' in file:
                exported_files.append(file)
    
    print(f'Found {len(exported_files)} exported figures in {output_dir}')
    
    # Extract Date and MRGS ID from exported filenames
    # Format: HLS.S30.T06WWS.2024159T212531.v2.0_evaluation.png
    exported_granules = set()
    for file in exported_files:
        try:
            parts = file.split('.')
            if len(parts) >= 4:
                mrgs_id = parts[2]  # T06WWS
                date_time = parts[3]  # 2024159T212531 or 2024159
                date = date_time.split('T')[0]  # Extract just date part
                exported_granules.add((date, mrgs_id))
        except Exception as e:
            pass
    
    print(f'Found {len(exported_granules)} unique exported granules')
    
    # Check which expected granules have files available
    missing_fmask4 = []
    missing_fmask5 = []
    missing_both = []
    has_files = []
    
    for idx, row in expected_granules.iterrows():
        date = str(row['Date'])
        mrgs_id = str(row['mrgs_id'])
        
        # Handle MRGS ID with/without T prefix
        mrgs_id_with_t = 'T' + mrgs_id if not mrgs_id.startswith('T') else mrgs_id
        mrgs_id_without_t = mrgs_id.replace('T', '') if mrgs_id.startswith('T') else mrgs_id
        
        # Check for Fmask4 files
        fmask4_matches = [f for f in files_s3_fmask4 
                         if date in f and (mrgs_id_with_t in f or mrgs_id_without_t in f)]
        
        # Check for Fmask5 files
        fmask5_matches = [f for f in files_s3_fmask5 
                         if date in f and (mrgs_id_with_t in f or mrgs_id_without_t in f)]
        
        has_fmask4 = len(fmask4_matches) > 0
        has_fmask5 = len(fmask5_matches) > 0
        
        if not has_fmask4 and not has_fmask5:
            missing_both.append((date, mrgs_id))
        elif not has_fmask4:
            missing_fmask4.append((date, mrgs_id))
        elif not has_fmask5:
            missing_fmask5.append((date, mrgs_id))
        else:
            has_files.append((date, mrgs_id))
    
    # Find granules that need processing (have files but not exported)
    needs_processing = []
    already_exported_list = []
    
    for date, mrgs_id in has_files:
        # Check both with and without T prefix
        mrgs_id_with_t = 'T' + mrgs_id if not mrgs_id.startswith('T') else mrgs_id
        mrgs_id_without_t = mrgs_id.replace('T', '') if mrgs_id.startswith('T') else mrgs_id
        
        is_exported = ((date, mrgs_id_with_t) in exported_granules or 
                      (date, mrgs_id_without_t) in exported_granules or
                      (date, mrgs_id) in exported_granules)
        
        if is_exported:
            already_exported_list.append((date, mrgs_id))
        else:
            needs_processing.append((date, mrgs_id))
    
    # Print summary
    print(f'\n{"="*60}')
    print('SUMMARY')
    print(f'{"="*60}')
    print(f'Total expected granules: {total_expected}')
    print(f'Granules with both Fmask4 and Fmask5 files: {len(has_files)}')
    print(f'  - Already exported: {len(already_exported_list)}')
    print(f'  - Needs processing: {len(needs_processing)}')
    print(f'\nGranules missing files:')
    print(f'  - Missing both Fmask4 and Fmask5: {len(missing_both)}')
    print(f'  - Missing only Fmask4: {len(missing_fmask4)}')
    print(f'  - Missing only Fmask5: {len(missing_fmask5)}')
    
    # Show sample missing granules
    if missing_both:
        print(f'\nSample granules missing both files (first 10):')
        for date, mrgs_id in missing_both[:10]:
            print(f'  {date} - {mrgs_id}')
    
    if needs_processing:
        print(f'\nSample granules that need processing (first 10):')
        for date, mrgs_id in needs_processing[:10]:
            print(f'  {date} - {mrgs_id}')
    
    return {
        'total_expected': total_expected,
        'missing_fmask4_files': missing_fmask4,
        'missing_fmask5_files': missing_fmask5,
        'missing_both_files': missing_both,
        'already_exported': already_exported_list,
        'needs_processing': needs_processing,
        'has_files': has_files
    }


In [ ]:
def filter_valid_time_steps(data_array, min_valid_ratio=0.9):
    """
    Filter time steps to keep only those with sufficient valid (non-NaN) pixels.
    Also treats -9999 as invalid (NaN) values.
    Parameters
    ----------
    data_array : xarray.DataArray
        Time-series DataArray with dimensions (time, y, x).
    min_valid_ratio : float, optional
        Minimum ratio of valid (non-NaN) pixels required for a time step to be kept.
        Default is 0.9 (90%).
    Returns
    -------
    xarray.DataArray or None
        Filtered DataArray, or None if input is None or no time steps meet the criteria.
    """
    if data_array is None:
        return None
    if 'time' not in data_array.dims:
        return data_array
    import numpy as np
    # Replace -9999 (no data value) with NaN if present
    # This ensures -9999 is treated as invalid data
    data_array = data_array.where(data_array != -9999, np.nan)
    initial_time_steps = len(data_array.time)
    # Calculate the number of valid (non-NaN) pixels for each time step
    # This now includes both NaN and -9999 as invalid
    valid_pixels_per_time_step = (~np.isnan(data_array)).sum(dim=['x', 'y'])
    # Calculate the total number of pixels in each spatial slice
    total_pixels_per_time_step = data_array.isel(time=0).size
    # Calculate the ratio of valid pixels
    valid_ratio = valid_pixels_per_time_step / total_pixels_per_time_step
    # Filter time steps where the valid ratio is greater than or equal to the minimum
    filtered_data_array = data_array.isel(time=(valid_ratio >= min_valid_ratio))
    final_time_steps = len(filtered_data_array.time)
    print(f"Filtered time steps: {initial_time_steps} -> {final_time_steps} "
          f"(kept {final_time_steps/initial_time_steps*100:.1f}% with >= {min_valid_ratio*100:.0f}% valid pixels)")
    if final_time_steps == 0:
        return None
    return filtered_data_array


In [ ]:
def plot_mrgs_spatial_and_temporal_distribution(files_s3, df_fmask_percentage, 
                                                 save_path=None, figsize=(16, 8), 
                                                 show_plot=True, dpi=300):
    """
    Create a spatial map showing the distribution of MGRS tiles and their temporal coverage.
    
    This function visualizes:
    1. Geographic distribution of MGRS tiles on a world map
    2. Number of observations per tile (as color intensity)
    3. Temporal distribution histogram of observations over time
    
    Parameters
    ----------
    files_s3 : list
        List of S3 file paths containing MGRS tile information
    df_fmask_percentage : pandas.DataFrame
        DataFrame with columns 'mrgs_id', 'Date', and optionally 'satellite_id'
    save_path : str, optional
        Path to save the figure. If None, figure is not saved.
    figsize : tuple, optional
        Figure size as (width, height). Default is (16, 8).
    show_plot : bool, optional
        Whether to display the plot. Default is True.
    dpi : int, optional
        Resolution for saved figure. Default is 300.
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        The generated figure
    axes : tuple
        Tuple of (ax_map, ax_temporal) axes objects
    
    Example
    -------
    >>> fig, axes = plot_mrgs_spatial_and_temporal_distribution(
    ...     files_s3_fmask4, df_fmask_percentage,
    ...     save_path='figures/tile_distribution.png'
    ... )
    """
    import mgrs
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    from datetime import datetime
    
    # Extract unique MGRS IDs from dataframe
    unique_mrgs_ids = df_fmask_percentage['mrgs_id'].unique()
    
    # Count observations per MGRS ID
    obs_counts = df_fmask_percentage.groupby('mrgs_id').size().to_dict()
    
    # Initialize MGRS converter
    m = mgrs.MGRS()
    
    # Convert MGRS IDs to lat/lon coordinates
    coords_data = []
    for mrgs_id in unique_mrgs_ids:
        mrgs_str = str(mrgs_id).strip()
        # Remove 'T' prefix if present
        mgrs_str = mrgs_str[1:] if mrgs_str.startswith('T') else mrgs_str
        
        try:
            # Get center point of MGRS tile
            lat, lon = m.toLatLon(f"{mgrs_str} 50000 50000")
            coords_data.append({
                'mrgs_id': mrgs_id,
                'lat': lat,
                'lon': lon,
                'count': obs_counts.get(mrgs_id, 0)
            })
        except Exception as e:
            print(f"Warning: Could not convert MGRS ID {mrgs_id}: {e}")
            continue
    
    df_coords = pd.DataFrame(coords_data)
    
    # Create figure with two subplots
    fig, (ax_map, ax_temporal) = plt.subplots(1, 2, figsize=figsize)
    
    # --- Left panel: Spatial distribution map ---
    # Plot world map boundaries
    ax_map.set_xlim(-180, 180)
    ax_map.set_ylim(-90, 90)
    ax_map.set_aspect('equal')
    
    # Add coastlines (simple grid)
    ax_map.axhline(y=0, color='gray', linewidth=0.5, alpha=0.3)
    ax_map.axvline(x=0, color='gray', linewidth=0.5, alpha=0.3)
    
    # Plot MGRS tiles as scatter points
    scatter = ax_map.scatter(
        df_coords['lon'], 
        df_coords['lat'],
        c=df_coords['count'],
        s=100,
        cmap='YlOrRd',
        alpha=0.7,
        edgecolors='black',
        linewidth=0.5,
        zorder=5
    )
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax_map, label='Number of Observations')
    
    # Formatting
    ax_map.set_xlabel('Longitude', fontsize=11)
    ax_map.set_ylabel('Latitude', fontsize=11)
    ax_map.set_title(f'Spatial Distribution of MGRS Tiles\n({len(unique_mrgs_ids)} tiles)', 
                     fontsize=12, fontweight='bold')
    ax_map.grid(True, alpha=0.3, linestyle='--')
    ax_map.set_facecolor('#f0f0f0')
    
    # --- Right panel: Temporal distribution ---
    # Convert dates to datetime (handle potential numpy string types)
    df_temp = df_fmask_percentage.copy()
    df_temp['datetime'] = pd.to_datetime(df_temp['Date'].apply(str), format='%Y%j', errors='coerce')
    
    # Remove any NaT values that couldn't be parsed
    df_temp = df_temp.dropna(subset=['datetime'])
    
    # Plot histogram of observations over time
    ax_temporal.hist(df_temp['datetime'], bins=50, 
                     color='steelblue', alpha=0.7, edgecolor='black')
    
    # Formatting
    ax_temporal.set_xlabel('Date', fontsize=11)
    ax_temporal.set_ylabel('Number of Observations', fontsize=11)
    ax_temporal.set_title(f'Temporal Distribution\n({len(df_fmask_percentage)} total observations)', 
                          fontsize=12, fontweight='bold')
    ax_temporal.grid(True, alpha=0.3, axis='y')
    
    # Rotate x-axis labels for better readability
    ax_temporal.tick_params(axis='x', rotation=45)
    
    # Add statistics text box
    stats_text = f"Date range: {df_temp['datetime'].min().strftime('%Y-%m-%d')} to {df_temp['datetime'].max().strftime('%Y-%m-%d')}\n"
    stats_text += f"Total tiles: {len(unique_mrgs_ids)}\n"
    stats_text += f"Total observations: {len(df_fmask_percentage)}\n"
    stats_text += f"Avg obs/tile: {len(df_fmask_percentage)/len(unique_mrgs_ids):.1f}"
    
    ax_temporal.text(0.02, 0.98, stats_text,
                     transform=ax_temporal.transAxes,
                     verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                     fontsize=9)
    
    plt.tight_layout()
    
    # Save figure if path provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    # Show plot if requested
    if show_plot:
        plt.show()
    
    return fig, (ax_map, ax_temporal)


In [ ]:
def plot_figure2_spatial_4panel(
    df_s30,
    df_l30,
    output_dir=None,
    figsize=(14, 7.5),
    save_path=None,
    dpi=300,
    show_plot=True,
):
    """
    Generate Figure 2: 2×2 spatial map of mean Fmask differences.

    Panels:
        (a) HLSS30 Cloud Difference      (b) HLSL30 Cloud Difference
        (c) HLSS30 Cloud Shadow Difference (d) HLSL30 Cloud Shadow Difference

    Each panel contains:
    - Köppen-Geiger zone polygons as background (Arid / Cold / Temperate)
    - Natural Earth country boundaries
    - Scatter points coloured by mean per-tile difference (RdBu_r, symmetric)
    - Half-violin inset (lower-left) showing per-climate-zone distribution with
      median line, sample size, and Wilcoxon significance symbol

    Parameters
    ----------
    df_s30 : pd.DataFrame
        Sentinel-2 (HLSS30) comparison DataFrame with columns:
        'mrgs_id', 'Cloud_diff', 'CloudShadow_diff', 'climate_zone'.
        Climate zone must already be classified (use classify_mrgs_ids_by_climate_zone).
    df_l30 : pd.DataFrame
        Landsat (HLSL30) comparison DataFrame with the same columns.
    output_dir : str, optional
        Directory to save the figure. Defaults to current working directory.
    figsize : tuple, optional
        Figure size (width, height) in inches. Default is (14, 7.5).
    save_path : str, optional
        Full output path. If None, derived from output_dir as
        'Figure2_Spatial_Mean_Differences_4panel.pdf'.
    dpi : int, optional
        Resolution for saved figure. Default is 300.
    show_plot : bool, optional
        If True, call plt.show(). Default is True.

    Returns
    -------
    fig : matplotlib.figure.Figure
    axes : np.ndarray  (2×2 array of Axes)
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    from matplotlib.colors import TwoSlopeNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    import geopandas as gpd
    from pathlib import Path
    from scipy import stats as scipy_stats

    try:
        import mgrs as mgrs_lib
    except ImportError:
        raise ImportError("Install mgrs: %pip install mgrs")

    if output_dir is None:
        output_dir = '.'
    if save_path is None:
        save_path = str(Path(output_dir) / 'Figure2_Spatial_Mean_Differences_4panel.pdf')

    DEFINED_ZONES  = ['Arid', 'Cold', 'Temperate']
    CLIMATE_COLORS = {'Arid': '#d62728', 'Cold': '#2ca02c', 'Temperate': '#1f77b4'}
    ZONE_FILL      = {'Temperate': '0.80', 'Arid': '0.60', 'Cold': '0.40'}

    # ── Aggregate per tile ────────────────────────────────────────────────────
    def _agg(df):
        return (df.groupby('mrgs_id')
                  .agg(Cloud_diff=('Cloud_diff', 'mean'),
                       CloudShadow_diff=('CloudShadow_diff', 'mean'),
                       climate_zone=('climate_zone', 'first'))
                  .reset_index())

    agg_s30 = _agg(df_s30)
    agg_l30 = _agg(df_l30)

    # ── MGRS → lat/lon ────────────────────────────────────────────────────────
    m_mgrs = mgrs_lib.MGRS()

    def _to_latlon(df_agg):
        lats, lons, cloud, shadow, ids, zones = [], [], [], [], [], []
        for _, row in df_agg.iterrows():
            mid = str(row['mrgs_id']).strip().lstrip('T')
            try:
                lat, lon = m_mgrs.toLatLon(f"{mid} 50000 50000")
                lats.append(lat);  lons.append(lon)
                cloud.append(row['Cloud_diff'])
                shadow.append(row['CloudShadow_diff'])
                ids.append(row['mrgs_id'])
                zones.append(row['climate_zone'])
            except Exception:
                pass
        return (np.array(lats), np.array(lons), np.array(cloud),
                np.array(shadow), np.array(ids), np.array(zones))

    lats_s, lons_s, cloud_s, shadow_s, ids_s, zones_s = _to_latlon(agg_s30)
    lats_l, lons_l, cloud_l, shadow_l, ids_l, zones_l = _to_latlon(agg_l30)

    # ── Shared colour limits (per row) ────────────────────────────────────────
    cloud_vmax  = max(np.nanmax(np.abs(cloud_s)),  np.nanmax(np.abs(cloud_l)))
    shadow_vmax = max(np.nanmax(np.abs(shadow_s)), np.nanmax(np.abs(shadow_l)))
    norm_cloud  = TwoSlopeNorm(vmin=-cloud_vmax,  vcenter=0, vmax=cloud_vmax)
    norm_shadow = TwoSlopeNorm(vmin=-shadow_vmax, vcenter=0, vmax=shadow_vmax)
    cmap = cm.get_cmap('RdBu_r')

    # ── Load basemap assets ───────────────────────────────────────────────────
    ne_candidates = [Path('ne_110m_admin_0_countries.zip'),
                     Path('hls_validation/ne_110m_admin_0_countries.zip'),
                     Path('../ne_110m_admin_0_countries.zip')]
    ne_path = next((p for p in ne_candidates if p.exists()), None)
    if ne_path is None:
        raise FileNotFoundError("ne_110m_admin_0_countries.zip not found")
    world = gpd.read_file(str(ne_path))

    koppen_candidates = [
        Path('koppen_geiger_major_zones/koppen_geiger_major_zones.shp'),
        Path('hls_validation/koppen_geiger_major_zones/koppen_geiger_major_zones.shp'),
        Path('koppen_geiger_major_zones.shp'),
    ]
    koppen_path = next((p for p in koppen_candidates if p.exists()), None)
    koppen_zones_gdf = None
    if koppen_path:
        koppen_zones_gdf = gpd.read_file(str(koppen_path))
        if koppen_zones_gdf.crs is None:
            koppen_zones_gdf = koppen_zones_gdf.set_crs('EPSG:4326')
        elif str(koppen_zones_gdf.crs) != 'EPSG:4326':
            koppen_zones_gdf = koppen_zones_gdf.to_crs('EPSG:4326')

    # ── Half-violin inset ─────────────────────────────────────────────────────
    def _add_violin_inset(parent_ax, values, zones_arr, violin_xlim=(-20, 20)):
        ax_v = inset_axes(parent_ax, width='30%', height='26%',
                          loc='lower left', borderpad=1.2)

        data_by_zone, labels = [], []
        for z in DEFINED_ZONES:
            mask = zones_arr == z
            if mask.sum() > 0:
                data_by_zone.append(values[mask])
                labels.append(z)

        if not data_by_zone:
            return

        positions = list(range(len(data_by_zone)))
        parts = ax_v.violinplot(data_by_zone, positions=positions,
                                widths=0.7, showmeans=False, showmedians=False,
                                showextrema=False, vert=False)

        for pc, zone, pos in zip(parts['bodies'], labels, positions):
            verts = pc.get_paths()[0].vertices
            verts[:, 1] = np.clip(verts[:, 1], -np.inf, pos)
            pc.get_paths()[0].vertices = verts
            pc.set_facecolor(CLIMATE_COLORS[zone])
            pc.set_alpha(0.75)
            pc.set_edgecolor('none')
            pc.set_linewidth(0)

        ax_v.set_xlim(*violin_xlim)
        x_min, x_max = ax_v.get_xlim()

        for pos, (zone, data) in enumerate(zip(labels, data_by_zone)):
            median_val = float(np.median(data))
            n_size     = len(data)
            try:
                _, p_val = scipy_stats.wilcoxon(data, alternative='two-sided')
            except Exception:
                p_val = 1.0

            ax_v.plot([median_val, median_val], [pos - 0.35, pos],
                      color='black', linewidth=2.0, zorder=10)

            sig = ('***' if p_val < 0.001 else '**' if p_val < 0.01
                   else '*' if p_val < 0.05 else 'ns')
            x_pos = x_min + 1.01 * (x_max - x_min)
            ax_v.text(x_pos, pos,
                      f'{zone} ({median_val:.2f}, N={n_size}, {sig})',
                      va='center', ha='left', fontsize=7, fontweight='bold',
                      color='black')

        ax_v.axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
        ax_v.set_yticks(positions)
        ax_v.set_yticklabels([''] * len(positions))
        ax_v.tick_params(axis='x', labelsize=7)
        ax_v.tick_params(axis='y', length=0)
        ax_v.set_facecolor('none');  ax_v.patch.set_alpha(0)
        ax_v.grid(False)
        for spine in ['top', 'right', 'left']:
            ax_v.spines[spine].set_visible(False)
        ax_v.spines['bottom'].set_linewidth(1.0)

    # ── Draw one map panel ────────────────────────────────────────────────────
    def _draw_map(ax, lats, lons, values, zones_arr, norm,
                  title, panel_label, n_tiles, violin_xlim=(-20, 20)):
        ax.set_facecolor('white')

        if koppen_zones_gdf is not None:
            for zone_name in ['Temperate', 'Arid', 'Cold']:
                if 'zone_name' in koppen_zones_gdf.columns:
                    sub = koppen_zones_gdf[koppen_zones_gdf['zone_name'] == zone_name]
                elif 'zone_code' in koppen_zones_gdf.columns:
                    code = {'Arid': 1, 'Cold': 2, 'Temperate': 3}[zone_name]
                    sub = koppen_zones_gdf[koppen_zones_gdf['zone_code'] == code]
                else:
                    sub = koppen_zones_gdf
                if len(sub):
                    sub.plot(ax=ax, color=ZONE_FILL.get(zone_name, '0.70'),
                             alpha=0.25, edgecolor='none', zorder=0)

        world.boundary.plot(ax=ax, linewidth=0.4, color='0.4', alpha=0.7, zorder=1)

        diff_abs = np.abs(values)
        diff_max = np.nanmax(diff_abs) if np.nanmax(diff_abs) > 0 else 1.0
        sizes    = 20 + (diff_abs / diff_max) * 160
        sc = ax.scatter(lons, lats, c=values, cmap=cmap, norm=norm,
                        s=sizes, alpha=0.85, linewidths=0.3, edgecolors='black', zorder=2)

        ax.set_xlim(-180, 180);  ax.set_ylim(-90, 90)
        ax.set_title(f'{title}  (N = {n_tiles} tiles)', fontsize=9.5,
                     fontweight='bold', pad=3)
        ax.tick_params(labelsize=7)
        ax.grid(False)
        ax.text(0.02, 0.97, panel_label, transform=ax.transAxes,
                fontsize=11, fontweight='bold', va='top',
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))

        _add_violin_inset(ax, values, zones_arr, violin_xlim=violin_xlim)
        return sc

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=figsize,  # default (14, 7.0) — no title
                              gridspec_kw={'hspace': 0.12, 'wspace': 0.06})

    sc_a = _draw_map(axes[0,0], lats_s, lons_s, cloud_s,  zones_s, norm_cloud,
                     '(a) HLSS30: Cloud Difference', '(a)', len(ids_s))
    sc_b = _draw_map(axes[0,1], lats_l, lons_l, cloud_l,  zones_l, norm_cloud,
                     '(b) HLSL30: Cloud Difference', '(b)', len(ids_l))
    sc_c = _draw_map(axes[1,0], lats_s, lons_s, shadow_s, zones_s, norm_shadow,
                     '(c) HLSS30: Cloud Shadow Difference', '(c)', len(ids_s),
                     violin_xlim=(-5, 5))
    sc_d = _draw_map(axes[1,1], lats_l, lons_l, shadow_l, zones_l, norm_shadow,
                     '(d) HLSL30: Cloud Shadow Difference', '(d)', len(ids_l),
                     violin_xlim=(-5, 5))

    for row_sc, row_axes, label in [
            (sc_a, [axes[0,0], axes[0,1]], 'Cloud Diff (Fmask 5.0.1 − Fmask 4.7) %'),
            (sc_c, [axes[1,0], axes[1,1]], 'Cloud Shadow Diff (Fmask 5.0.1 − Fmask 4.7) %')]:
        cbar = fig.colorbar(row_sc, ax=row_axes, orientation='vertical',
                            fraction=0.022, pad=0.01, shrink=0.90)
        cbar.set_label(label, fontsize=8)
        cbar.ax.tick_params(labelsize=7)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, format='pdf', bbox_inches='tight',
                    pad_inches=0.08, dpi=dpi)
        print(f'\n✓ Figure 2 saved to: {save_path}')

    if show_plot:
        plt.show()

    return fig, axes
